#### Application Codebase Knowledge Graph (KG) Tools

##### Environment setup

###### Package imports

In [45]:
from __future__ import annotations

import os
import sys
from dotenv import load_dotenv
import requests
import pandas as pd
from pathlib import Path
from neo4j import GraphDatabase
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget
import gradio as gr
import inspect
import hashlib
import os
from pathlib import Path
from typing import Any
from tree_sitter_language_pack import get_parser

# Load environment variables
env_file=Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE")

###### Test connections

In [2]:
def test_neo4j_connection(env_file=Path.cwd().parent / ".env"):
    """
    Test connectivity to a Neo4j database using settings
    stored in a .env file.

    Expected variables:

        NEO4J_URI
        NEO4J_USERNAME
        NEO4J_PASSWORD
        NEO4J_DATABASE
    """

    try:
        with GraphDatabase.driver(uri, auth=(username, password), database=database) as driver:

            # Test the connection
            driver.verify_connectivity()

            # Open a session
            with driver.session(database=database) as session:
                result = session.run(
                    """
                    RETURN
                        1 AS connected,
                        datetime() AS server_time
                    """
                )

                record = result.single()

                print("✅ Successfully connected to Neo4j")
                print(f"Database    : {database}")
                print(f"URI         : {uri}")
                print(f"Server Time : {record['server_time']}")

                return True

    except Exception as ex:
        print("❌ Connection failed")
        print(type(ex).__name__)
        print(ex)

        return False

test_neo4j_connection()

✅ Successfully connected to Neo4j
Database    : appkg
URI         : bolt://localhost:7687
Server Time : 2026-11-13T03:06:36.634000000+00:00


True

###### Create Neo4j driver object and python-cypher method

In [3]:
# Create driver object
# Keeping the Neo4j Driver open for the lifetime of your Jupyter session is the normal pattern and does not, by itself, constitute a memory leak.
driver = GraphDatabase.driver(uri, auth=(username, password), database=database)

driver.verify_connectivity()

# Define a generic function to run a Cypher query
def run_cypher_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return result.data()

###### Code: Check pre-existing Neo4j graph schema

In [4]:
# Generate graph from Cypher query
g = Neo4jGraphWidget(driver)
# View schema
g.show_cypher("CALL db.schema.visualization()")

query_node_frequency = """
MATCH (n)
RETURN labels(n) AS label, count(*) AS count
ORDER BY count DESC;
"""
result_query_node_frequency = run_cypher_query(query_node_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_nodes = pd.DataFrame(result_query_node_frequency)
# Flatten the 'label' column since it's a list
df_nodes['label'] = df_nodes['label'].apply(
    lambda x: x[0] if len(x) > 0 else None)
print("Nodes by Count")
print(df_nodes)  # Outputs the list of nodes (Persons)

query_relationship_frequency = """
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, COUNT(r) AS count
ORDER BY count DESC;
"""
result_query_relationship_frequency = run_cypher_query(
    query_relationship_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_rels = pd.DataFrame(result_query_relationship_frequency)
# Flatten the 'label' column since it's a list
# df_rels['label'] = df_rels['label'].apply(lambda x: x[0] if len(x) > 0 else None)
print("Relationships by Count")
print(df_rels)  # Outputs the list of nodes (Persons)

GraphWidget(layout=Layout(height='500px', width='100%'))

Nodes by Count
       label  count
0   FileName   1634
1  Directory    271
2   FileType     47
Relationships by Count
  relationshipType  count
0         CONTAINS   3489
1       IS_TYPE_OF   1634


###### No. of FileNames by FileType

In [5]:
query_count_of_filetypes = """
MATCH (f:FileName)-[:IS_TYPE_OF]->(t:FileType)
RETURN t.type AS FileType, COUNT(f) AS FileCount
ORDER BY FileCount DESC;
"""

result_query_count_of_filetypes = run_cypher_query(query_count_of_filetypes)
df_result_query_count_of_filetypes = pd.DataFrame(result_query_count_of_filetypes)
print(f"Count of FileTypes:\n{df_result_query_count_of_filetypes}\n")
total_files = df_result_query_count_of_filetypes["FileCount"].sum()
print(f"\nTotal number of files: {total_files}\n")

Count of FileTypes:
                    FileType  FileCount
0                 Javascript        555
1                        SVG        319
2                        CSS        160
3                     Header        103
4                      Other         97
5                        C++         77
6                      XHTML         51
7                        PNG         37
8                       HTML         23
9                       JSON         23
10                       PDF         18
11                      Text         16
12  FreeMarker_Java_template         15
13                Typescript         14
14               DTD_XML_def         12
15                     Shell         12
16                       IDL         10
17                       RSS          7
18                      Epub          7
19                       NSH          7
20                       GIF          7
21                       XUL          6
22                 Java_prop          5
23                  

##### Mapping of FileType 'Other' by Extension

In [6]:
query_count_of_filetype_ext = """
MATCH (f:FileName)-[:IS_TYPE_OF]->(t:FileType)
WHERE t.type = 'Other'
WITH f,
     CASE
       WHEN f.name CONTAINS '.' THEN toLower(last(split(f.name, '.')))
       ELSE 'No Extension'
     END AS Extension
RETURN Extension, COUNT(f) AS FileCount
ORDER BY FileCount DESC;
"""

result_query_count_of_filetype_ext = run_cypher_query(query_count_of_filetype_ext)
df_result_query_count_of_filetype_ext = pd.DataFrame(result_query_count_of_filetype_ext)
print(df_result_query_count_of_filetype_ext)

total_files = df_result_query_count_of_filetype_ext["FileCount"].sum()
print(f"\nTotal number of files: {total_files}")

                     Extension  FileCount
0                 No Extension         63
1                          mts          6
2                           xz          5
3                        patch          3
4                          rtf          2
5                          ris          1
6                         ctv6          1
7                      babelrc          1
8                gitattributes          1
9                    gitignore          1
10                  gitmodules          1
11                     desktop          1
12                       plist          1
13                         car          1
14  0_release_build_and_deploy          1
15                         sfx          1
16                         def          1
17                         dsp          1
18                         dsw          1
19                         tag          1
20                         xsl          1
21                    eslintrc          1
22                         ttf    

###### Test Github API connection

In [7]:
def test_github_connection():
    """Test connectivity to GitHub and validate the GitHub PAT."""

    # Find .env in the directory above the notebook's working directory
    env_file = Path.cwd().parent / ".env"

    print(f"Loading environment from: {env_file}")

    if not env_file.exists():
        print("❌ .env file not found")
        return False

    load_dotenv(env_file, override=True)

    token = os.getenv("GITHUB_TOKEN")

    if not token:
        print("❌ GITHUB_TOKEN is missing from .env")
        return False

    # GitHub API headers
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2026-03-10",
    }

    try:
        # Test GitHub API + PAT authentication
        response = requests.get(
            "https://api.github.com/user",
            headers=headers,
            timeout=10,
        )

        print(f"GitHub API status: {response.status_code}")

        if response.status_code == 200:
            user = response.json()

            print("✅ Successfully connected to GitHub")
            print("✅ PAT is valid")
            print(f"GitHub user : {user.get('login')}")
            print(f"User ID     : {user.get('id')}")
            print(f"Account type: {user.get('type')}")

            return True

        elif response.status_code == 401:
            print("❌ GitHub API connection succeeded")
            print("❌ PAT authentication failed")
            print("The token may be invalid, expired, or revoked.")

        elif response.status_code == 403:
            print("⚠️ GitHub API responded with 403 Forbidden")
            print("The PAT was received, but GitHub denied the request.")
            print(response.text)

        else:
            print("❌ GitHub API request failed")
            print(response.text)

        return False

    except requests.exceptions.Timeout:
        print("❌ Connection to GitHub timed out")
        return False

    except requests.exceptions.ConnectionError as ex:
        print("❌ Could not connect to GitHub")
        print(ex)
        return False

    except requests.exceptions.RequestException as ex:
        print("❌ GitHub request failed")
        print(type(ex).__name__)
        print(ex)
        return False

test_github_connection()

Loading environment from: E:\Projects\knowledge_graph_tools\.env
GitHub API status: 200
✅ Successfully connected to GitHub
✅ PAT is valid
GitHub user : georgejaymcmc
User ID     : 88515517
Account type: User


True

##### Repo details

In [26]:
# 1. Enable auto-reloading so changes to your .py file update automatically
%load_ext autoreload
%autoreload 2

# 2. Add your project 'src' directory to Python's path
# Adjust the number of parents depending on where your notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parents[0] # Assumes notebook is one folder deep (e.g., in a 'notebooks/' folder)
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 3. Import your service directly from your local module
from codebase_kg.services.github_repo_service import GithubRepoService

# 4. Initialize and use it
github = GithubRepoService(token="YOUR_GITHUB_TOKEN")
print("Module successfully imported from:", sys.modules['codebase_kg.services.github_repo_service'].__file__)

# Get a list of (name, function_object) tuples
functions = inspect.getmembers(GithubRepoService, inspect.isfunction)

# Print just the function names
print("\nList of Github service functions:")
for name, func in functions:
    print(f"{name}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Module successfully imported from: E:\Projects\knowledge_graph_tools\src\codebase_kg\services\github_repo_service.py

List of Github service functions:
__init__
_classify_files
_repo_data
default_branch
description
directory_count
file_count
file_types
get_blob_content
get_directory_file_count
get_file_content
get_repository_files
language
size
stars
stats
visibility


##### Extracting Zotero repo stats

###### Code: Choose repo to analyse
1. Fork repo to baseline codebase - 16/09/2026

In [43]:
# Repository name
owner = "georgejaymcmc"
repo = "zotero"

# Print result
result = github.stats(owner, repo)

# Pretty print result
for key, value in result.items():
    print(f"{key:20}: {value}")


name                : zotero
full_name           : georgejaymcmc/zotero
description         : Zotero is a free, easy-to-use tool to help you collect, organize, annotate, cite, and share your research sources.
language            : None
size_kb             : 242678
visibility          : public
default_branch      : main
stars               : 0
forks               : 0
watchers            : 0
directories         : 363
files               : 3489
url                 : https://github.com/georgejaymcmc/zotero


###### Code: Purge the cached module from the notebook

In [148]:
# 1. Collect all loaded submodules related to your project package
to_delete = [name for name in sys.modules if name.startswith("codebase_kg")]

# 2. Purge them from the sys.modules memory cache
for module_name in to_delete:
    del sys.modules[module_name]
    print(f"🔥 Purged cache for: {module_name}")

# 3. Re-import your updated class freshly from the disk
from codebase_kg.services.github_repo_service import GithubRepoService


🔥 Purged cache for: codebase_kg
🔥 Purged cache for: codebase_kg.services
🔥 Purged cache for: codebase_kg.services.github_repo_service


###### Code: Output repo file types and directory location to csv

In [37]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "georgejaymcmc"
repo = "zotero"


In [150]:
# 1. RUN THIS METHOD TO GENERATE THE CSV FILE
print("Generating CSV file...")
csv_results = github.file_types(owner, repo, output_csv="zotero_files.csv")

# 2. Pretty print the execution summary returned by file_types
print("\n--- Execution Summary ---")
for key, value in csv_results.items():
    print(f"{key:20}: {value}")


Generating CSV file...
📦 Dataframe successfully dumped to: E:\Projects\knowledge_graph_tools\src\codebase_kg\neo4j_imports\zotero_files.csv

--- Execution Summary ---
total_files         : 3489
directory_count     : 271
output_csv          : zotero_files.csv
file_type_counts    : {'SVG': 774, 'FreeMarker_Java_template': 673, 'Javascript': 578, 'DTD_XML_def': 482, 'CSS': 202, 'Java_prop': 146, 'Other': 108, 'Header': 107, 'C++': 77, 'JSON': 63, 'XHTML': 51, 'PNG': 41, 'HTML': 23, 'PDF': 18, 'Text': 16, 'Typescript': 14, 'Shell': 12, 'IDL': 10, 'NSH': 7, 'GIF': 7, 'RSS': 7, 'Epub': 7, 'Markdown': 6, 'RC': 6, 'XUL': 6, 'NSI': 5, 'XML': 4, 'Python Script': 4, 'ICO': 4, 'SQL': 4, '.ini': 3, '.manifest': 3, 'CSL': 3, 'VBScript': 2, 'WOFF': 2, 'SQLite': 2, 'RDF': 2, 'YAML': 1, 'C': 1, 'NLF': 1, 'XPI': 1, 'ATOM': 1, 'OPML': 1, 'LUA': 1, 'OPF': 1, 'JPG': 1, 'ZIP': 1}
unknown_extensions  : ['', '.0_release_build_and_deploy', '.car', '.ctv6', '.def', '.desktop', '.dsp', '.dsw', '.mts', '.patch', 

##### Import what we know into Neo4j: KGs are built progressively
- directories, file names and file types form the base of the App KG

###### Start from scratch: Delete all nodes, relationships, etc from Neo4j

In [142]:
# Delete all nodes and relationships

query_delete_all_nodes_relationships = """
MATCH (n)
DETACH DELETE n
RETURN count(n) AS deleted_nodes
"""

result_query_delete_all_nodes_relationships = run_cypher_query(query_delete_all_nodes_relationships)

result_query_delete_all_nodes_relationships

[{'deleted_nodes': 0}]

###### Upload repo_files.csv into Neo4j to create base graph elements

In [151]:
# Generic function to run Cypher queries
query_load_repo_files = """
// Load File Type, Directory, and File Name nodes
LOAD CSV WITH HEADERS FROM 'file:///zotero_files.csv' AS row
MERGE (f:FileName {name: row.FileName})
MERGE (d:Directory {name: row.Directory})
MERGE (t:FileType {type: row.FileType})
MERGE (d)-[:CONTAINS]->(f)
MERGE (f)-[:IS_TYPE_OF]->(t);
"""
result_query_load_repo_files = run_cypher_query(query_load_repo_files)

###### Check the upload
- Compare file count with above repo query result

In [48]:
# Generate graph from Cypher query
g = Neo4jGraphWidget(driver)
# View schema
g.show_cypher("CALL db.schema.visualization()")

query_node_frequency = """
MATCH (n)
RETURN labels(n) AS label, count(*) AS count
ORDER BY count DESC;
"""
result_query_node_frequency = run_cypher_query(query_node_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_nodes = pd.DataFrame(result_query_node_frequency)
# Flatten the 'label' column since it's a list
df_nodes['label'] = df_nodes['label'].apply(
    lambda x: x[0] if len(x) > 0 else None)
print("Nodes by Count")
print(df_nodes)  # Outputs the list of nodes (Persons)

query_relationship_frequency = """
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, COUNT(r) AS count
ORDER BY count DESC;
"""
result_query_relationship_frequency = run_cypher_query(
    query_relationship_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_rels = pd.DataFrame(result_query_relationship_frequency)
# Flatten the 'label' column since it's a list
# df_rels['label'] = df_rels['label'].apply(lambda x: x[0] if len(x) > 0 else None)
print("Relationships by Count")
print(df_rels)  # Outputs the list of nodes (Persons)


GraphWidget(layout=Layout(height='500px', width='100%'))

Nodes by Count
       label  count
0   FileName   1634
1  Directory    271
2   FileType     47
Relationships by Count
  relationshipType  count
0         CONTAINS   3489
1       IS_TYPE_OF   1634


##### Gradio chatbot file selector

###### Function to list labels and relationship types

In [28]:
def get_graph_schema():
    """Return node labels and relationship types with counts as a DataFrame."""

    try:

        # ---------------------------------
        # Node labels and properties
        # ---------------------------------

        labels = run_cypher_query("""
            MATCH (n)
            UNWIND labels(n) AS label
            UNWIND keys(n) AS property
            RETURN
                label,
                count(DISTINCT n) AS count,
                collect(DISTINCT property) AS properties
            ORDER BY label
        """)

        # ---------------------------------
        # Relationship types
        # ---------------------------------

        relationships = run_cypher_query("""
            MATCH ()-[r]->()
            RETURN
                type(r) AS relationship,
                count(*) AS count
            ORDER BY relationship
        """)

        # ---------------------------------
        # Build combined result
        # ---------------------------------

        rows = []

        # Nodes
        for row in labels:
            rows.append({
                "Type": "Node",
                "Name": row["label"],
                "Count": row["count"],
                "Properties": ", ".join(
                    sorted(row["properties"])
                )
            })

        # Relationships
        for row in relationships:
            rows.append({
                "Type": "Relationship",
                "Name": row["relationship"],
                "Count": row["count"],
                "Properties": ""
            })

        # ---------------------------------
        # Return DataFrame
        # ---------------------------------

        return pd.DataFrame(
            rows,
            columns=[
                "Type",
                "Name",
                "Count",
                "Properties"
            ]
        )

    except Exception as e:

        return pd.DataFrame({
            "Error": [
                f"{type(e).__name__}: {e}"
            ]
        })

###### Convert cypher query results to pandas df

In [29]:
def execute_cypher(query):
    """Execute Cypher and return the results as a DataFrame."""

    if not query or not query.strip():
        return pd.DataFrame({
            "Result": ["Please enter a Cypher query."]
        })

    try:
        records = run_cypher_query(query)

        if not records:
            return pd.DataFrame({
                "Result": ["Query returned no rows."]
            })

        return pd.DataFrame(records)

    except Exception as e:
        return pd.DataFrame({
            "Error": [
                f"{type(e).__name__}: {e}"
            ]
        })

###### Gradio interface

###### Test queries

In [30]:
query_test="""
MATCH (d:Directory)-[:CONTAINS]->(f:FileName)-[:IS_TYPE_OF]->(ft:FileType)
WHERE ft.type = "Javascript"

WITH f.name AS file_name, count(DISTINCT d) AS directory_count
WHERE directory_count > 1

MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
WHERE f.name = file_name

RETURN
    file_name,
    d.name AS directory
ORDER BY file_name, directory
"""

result_query_test = run_cypher_query(query_test)

result_query_test

[{'file_name': 'commandLineHandler.js', 'directory': 'app/assets'},
 {'file_name': 'commandLineHandler.js',
  'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'contextPane.js', 'directory': 'chrome/content/zotero'},
 {'file_name': 'contextPane.js',
  'directory': 'chrome/content/zotero/elements'},
 {'file_name': 'http.js',
  'directory': 'chrome/content/zotero/actors/translation'},
 {'file_name': 'http.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'id.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'id.js', 'directory': 'resource/tinymce/langs'},
 {'file_name': 'locale.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'locale.js', 'directory': 'resource/tinymce'},
 {'file_name': 'plugin.min.js',
  'directory': 'resource/tinymce/plugins/autolink'},
 {'file_name': 'plugin.min.js', 'directory': 'resource/tinymce/plugins/code'},
 {'file_name': 'plugin.min.js',
  'directory': 'resource/tinymce/plugins/contextmenu'},
 {'file_name'

In [31]:
# Gradio UI
with gr.Blocks(title="Neo4j Cypher Explorer") as demo:
    gr.Markdown("# Neo4j Cypher Explorer")
    gr.Markdown(
        "Explore the current graph model and execute Cypher queries."
    )

    # ---------------------------------------------------------
    # Graph schema
    # ---------------------------------------------------------

    gr.Markdown("## Graph Schema")

    schema_button = gr.Button(
        "Refresh Graph Schema",
        variant="secondary"
    )

    schema_output = gr.Dataframe(
        label="Graph Schema",
        interactive=False,
        wrap=True
    )

    # ---------------------------------------------------------
    # Cypher query
    # ---------------------------------------------------------

    gr.Markdown("## Cypher Query")

    cypher_input = gr.Textbox(
        label="Enter Cypher",
        lines=8,
        value="""
        MATCH (d:Directory)-[:CONTAINS]->(f:FileName)-[:IS_TYPE_OF]->(ft:FileType)
WHERE ft.type = "Javascript"

WITH f.name AS file_name, count(DISTINCT d) AS directory_count
WHERE directory_count > 1

MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
WHERE f.name = file_name

RETURN
    file_name,
    d.name AS directory
ORDER BY file_name, directory
    """
    )

    execute_button = gr.Button(
        "Run Query",
        variant="primary"
    )

    # ---------------------------------------------------------
    # Results
    # ---------------------------------------------------------

    gr.Markdown("## Results")

    result_output = gr.Dataframe(
        label="Query Result",
        interactive=False,
        wrap=True
    )

    # ---------------------------------------------------------
    # Events
    # ---------------------------------------------------------

    schema_button.click(
        fn=get_graph_schema,
        outputs=schema_output
    )

    execute_button.click(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )

    cypher_input.submit(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


##### Display repo file contents
```mermaid
sequenceDiagram
    participant A as Python CLI
    participant B as NEO4J
    participant C as GITHUB

    A->>B: 1. Query Neo4j for list of files
    B->>A: 2. List files in a Gradio SelectBox
    A->>C: 3. Select file to get syntax
    C->>A: 4. Return file contents for display in Gradio
```

###### 1. Query Neo4j for files

In [32]:
def get_files_from_neo4j(file_type=None):
    """Return files from Neo4j, optionally filtered by file type."""

    query = """
    MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
          -[:IS_TYPE_OF]->(ft:FileType)

    WHERE $file_type IS NULL
       OR ft.type = $file_type

    RETURN
        d.name AS directory,
        f.name AS file_name,
        ft.type AS file_type

    ORDER BY directory, file_name
    """

    records = run_cypher_query(
        query,
        {"file_type": file_type}
    )

    return pd.DataFrame(records)

###### 2. Create Gradio file choices

In [33]:
def build_github_path(directory, file_name):
    """Convert Neo4j directory/file information into a GitHub path."""

    directory = str(directory).strip()

    if directory.upper() == "ROOT":
        return file_name

    return f"{directory}/{file_name}"


def get_file_choices(file_type=None):
    """Build Gradio file choices from Neo4j."""

    df = get_files_from_neo4j(file_type)

    if df.empty:
        return []

    choices = []

    for _, row in df.iterrows():

        file_path = build_github_path(
            row["directory"],
            row["file_name"]
        )

        choices.append((file_path, file_path))

    return choices

In [34]:
def get_file_types():
    """Return file types available in the Neo4j codebase."""

    query = """
    MATCH (ft:FileType)
    RETURN DISTINCT ft.type AS file_type
    ORDER BY file_type
    """

    records = run_cypher_query(query)

    return [
        row["file_type"]
        for row in records
        if row["file_type"]
    ]

In [35]:
def update_file_dropdown(file_type):
    """Update the file dropdown based on the selected file type."""

    if not file_type:
        return gr.Dropdown(
            choices=[],
            value=None
        )

    choices = get_file_choices(file_type)

    return gr.Dropdown(
        choices=choices,
        value=None
    )

###### 3. Map Neo4j FileType to `gr.Code` languages

In [36]:
from pathlib import Path


def get_language(file_name):
    """Return the Gradio language for a filename."""

    extension = Path(file_name).suffix.lower()

    language_map = {
        ".js": "javascript",
        ".jsx": "javascript",
        ".ts": "typescript",
        ".tsx": "typescript",

        ".py": "python",

        ".json": "json",

        ".html": "html",
        ".htm": "html",

        ".css": "css",

        ".sql": "sql",

        ".java": "java",

        ".c": "c",
        ".h": "c",
        ".cpp": "cpp",
        ".hpp": "cpp",

        ".cs": "csharp",

        ".md": "markdown",

        ".xml": "xml",

        ".yaml": "yaml",
        ".yml": "yaml",

        ".sh": "shell",

        ".txt": "text",
    }

    return language_map.get(extension, "text")

###### 4. Retrieve code from Github

In [37]:

def get_selected_file(file_path):
    """Retrieve the selected file from GitHub."""
    if not file_path:
        return gr.Code(value="", language=None)

    df = get_files_from_neo4j()

    matches = df[
        df.apply(
            lambda row: build_github_path(
                row["directory"],
                row["file_name"]
            ) == file_path,
            axis=1
        )
    ]

    if matches.empty:
        return gr.Code(value="", language=None)

    file_name = matches.iloc[0]["file_name"]
    language = get_language(file_name)

    # Gradio uses None for plain text instead of "text"
    if language == "text":
        language = None

    code = github.get_file_content(
        owner,
        repo,
        file_path
    )

    return gr.Code(value=str(code), language=language)

###### 5. Gradio UI

In [38]:
with gr.Blocks(title="Code Explorer") as code_demo:

    gr.Markdown("# Code Explorer")
    gr.Markdown(
        "Select a file type, then select a file from the Neo4j codebase graph."
    )

    file_type_dropdown = gr.Dropdown(
        label="Select File Type",
        choices=get_file_types(),
        value=None,
        interactive=True
    )

    file_dropdown = gr.Dropdown(
        label="Select File",
        choices=[],
        value=None,
        interactive=True
    )

    code_output = gr.Code(
        label="Source Code",
        language="javascript",
        interactive=False,
        lines=40,
        max_lines=60
    )

    file_type_dropdown.change(
        fn=update_file_dropdown,
        inputs=file_type_dropdown,
        outputs=file_dropdown
    )

    file_dropdown.change(
        fn=get_selected_file,
        inputs=file_dropdown,
        outputs=code_output
    )

code_demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


###### Testing/debugging key Gradio functions

`def get_file_content(self, owner, repo, file_path)`

In [ ]:
github.get_file_content('georgejaymcmc', 'zotero', 'chrome/content/zotero-platform/unix/integration.css' )

In [185]:
get_selected_file('chrome/content/zotero-platform/unix/integration.css')

('window.citation-dialog {\n\t-moz-appearance: none;\n\tbackground: -moz-linear-gradient(-90deg, rgb(243,123,119) 0, rgb(180,47,38) 50%, rgb(156,36,27) 50%) !important;\n}\n\nwindow.citation-dialog.note-dialog {\n\tbackground: -moz-linear-gradient(-90deg, rgb(249, 231, 179) 0, rgb(228, 193, 94) 50%, rgb(221, 184, 81) 50%) !important;\n}\n\n.citation-dialog.entry {\n\tpadding: 10px;\n}',
 'css')

In [42]:
print(f"Inspect Github service function code for {github._repo_data}: \n{inspect.getsource(github._repo_data)}")
print(f"Github service object type for {github._repo_data}:\n{type(github._repo_data(owner, repo))}")

Inspect Github service function code for <bound method GithubRepoService._repo_data of <codebase_kg.services.github_repo_service.GithubRepoService object at 0x000001F3B1722750>>: 
    def _repo_data(self, owner, repo):
        """Get repository metadata, using the cache when available."""

        cache_key = f"{owner}/{repo}"

        if cache_key not in self._repo_cache:
            url = f"https://api.github.com/repos/{owner}/{repo}"

            response = requests.get(
                url,
                headers=self.headers,
                timeout=10
            )

            if response.status_code != 200:
                raise RuntimeError(
                    f"GitHub API error {response.status_code}: "
                    f"{response.text}"
                )

            self._repo_cache[cache_key] = response.json()

        return self._repo_cache[cache_key]

Github service object type for <bound method GithubRepoService._repo_data of <codebase_kg.services.github_repo_ser

##### Application code syntax analysis overview
```mermaid
flowchart LR
subgraph App KG
    B(Application code)
    B-->|Extract|L(&#x2460 AST)
    B-->|Extract|M(&#x2461 Lexical search)
    O(Application KG Model)
    L-->O
    M-->O
    O-->B
end
```

Two types of analysis are required
1. Abstract Syntax Trees (AST)
   - using language-specific parsers to determine classes, functions, symbols, within each file
   - control flow graphs
   - call graphs but only for files from the same language family
2. Lexical search
   - to see which files and/or functions are called between languages
      - e.g. `javascript` files in `html`

##### Javascript ASTs - Using the `tree-sitter` language parse library

**Workflow**
1. Environment and configuration
   - Install dependencies, configure paths and initialise the parser.
2. Repository discovery
   - Identify JavaScript files and build files_df.
3. AST parsing
   - Parse each file and retain its syntax tree and parsing diagnostics.
4. AST utilities
   - Create reusable functions for traversing nodes, extracting source text and tracking locations.
5. Symbol extraction
   - Extract declarations into symbols_df.
6. Scope extraction
   - Build lexical scope information into scopes_df.
7. Import and export extraction
   - Build imports_df and exports_df.
8. Reference and assignment extraction
   - Build references_df and assignments_df.
9. Function call extraction
   - Build calls_df, preserving raw call expressions and caller context.
10. Class and inheritance extraction
    - Build inheritance_df and extend symbol metadata for classes and members.
11. Symbol resolution
    - Resolve references, imports, member access and calls into additional DataFrames.
12. Validation and reporting
    - Review extraction coverage, unresolved references and parsing errors.
13. Neo4j graph construction
    - Persist the code model and relationships with provenance.

###### 1. Environment and configuration

In [46]:
# Repository details
owner = "georgejaymcmc"
repo = "zotero"

# Repository identifier
REPOSITORY_ID = f"{owner}/{repo}"

# Repository revision (optional)
REPOSITORY_REVISION = None

# Start with these extensions. Add TypeScript later if required.
SOURCE_EXTENSIONS = {".js", ".jsx", ".mjs", ".cjs"}

EXCLUDED_DIRS = {
    ".git", "node_modules", "dist", "build", "coverage",
    ".venv", "__pycache__", "vendor"
}

REPOSITORY_REVISION = None  # Set to a Git commit hash when available.

parser = get_parser("javascript")


def stable_id(*parts: Any, prefix: str = "") -> str:
    """Create a deterministic identifier from stable input values."""
    raw = "|".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]
    return f"{prefix}{digest}"


def node_text(node, source_code: bytes) -> str:
    """Return the exact source text covered by a Tree-sitter node."""
    return source_code[node.start_byte:node.end_byte].decode("utf-8", errors="replace")


def node_location(node) -> dict:
    return {
        "start_line": node.start_point.row + 1,
        "start_column": node.start_point.column + 1,
        "end_line": node.end_point.row + 1,
        "end_column": node.end_point.column + 1,
        "start_byte": node.start_byte,
        "end_byte": node.end_byte,
    }


def walk_tree(node):
    """Yield a node and all descendants in source-tree order."""
    yield node
    for child in node.children:
        yield from walk_tree(child)


###### 2. Repository discovery

In [48]:

def discover_repository() -> pd.DataFrame:
    """
    Discover JavaScript files from the existing Neo4j repository graph.
    Neo4j is treated as the source of truth.
    """

    query = """
    MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
          -[:IS_TYPE_OF]->(ft:FileType)

    WHERE ft.type = $file_type

    RETURN
        d.name AS directory,
        f.name AS file_name,
        ft.type AS neo4j_file_type

    ORDER BY directory, file_name
    """

    records = run_cypher_query(
        query,
        {"file_type": "Javascript"}
    )

    neo4j_files_df = pd.DataFrame(records)

    if neo4j_files_df.empty:
        raise ValueError(
            "No Javascript files were returned from Neo4j."
        )

    # Reconstruct the relative file path.
    neo4j_files_df["file_path"] = (
        neo4j_files_df["directory"].apply(
            lambda x: "" if x in ("", ".") else str(x).strip("/")
        )
        + "/"
        + neo4j_files_df["file_name"].astype(str)
    ).str.lstrip("/")

    # Derive the extension from the filename.
    neo4j_files_df["extension"] = (
        neo4j_files_df["file_name"]
        .apply(lambda x: Path(x).suffix.lower())
    )

    # Preserve the existing downstream DataFrame structure.
    neo4j_files_df["file_id"] = neo4j_files_df["file_path"].apply(
        lambda path: stable_id(
            REPOSITORY_ID,
            path,
            prefix="file_"
        )
    )

    neo4j_files_df["repository_id"] = REPOSITORY_ID
    neo4j_files_df["language"] = "javascript"

    neo4j_files_df["file_type"] = (
        neo4j_files_df["file_path"].apply(
            lambda path: (
                "test"
                if re.search(
                    r"(^|[./\_-])(test|spec)([./\_-]|$)",
                    path,
                    re.I
                )
                else "source"
            )
        )
    )

    # These properties are not returned by the current Neo4j query.
    neo4j_files_df["file_size"] = None
    neo4j_files_df["file_sha"] = None
    neo4j_files_df["repository_revision"] = REPOSITORY_REVISION

    # Resource files are retained, but flagged for AST consideration.
    neo4j_files_df["ast_candidate"] = (
        neo4j_files_df["directory"].apply(
            lambda directory: (
                Path(directory).parts[0] != "resource"
                if directory not in ("", ".")
                else True
            )
        )
    )

    columns = [
        "file_id",
        "repository_id",
        "file_path",
        "file_name",
        "directory",
        "extension",
        "language",
        "file_type",
        "file_size",
        "file_sha",
        "repository_revision",
        "ast_candidate"
    ]

    return (
        neo4j_files_df[columns]
        .drop_duplicates(subset=["file_path"])
        .sort_values("file_path")
        .reset_index(drop=True)
    )


# Discover JavaScript files from Neo4j
files_df = discover_repository()

display(files_df.head(10))

print(f"Discovered {len(files_df):,} JavaScript files")
print(f"AST candidates: {files_df['ast_candidate'].sum():,}")
print(f"Resource files retained: {(~files_df['ast_candidate']).sum():,}")


,file_id,repository_id,file_path,file_name,directory,extension,language,file_type,file_size,file_sha,repository_revision,ast_candidate
0,file_b3236ac1011d7acd,georgejaymcmc/zotero,ROOT/eslint.config.mjs,eslint.config.mjs,ROOT,.mjs,javascript,source,None,None,None,True
1,file_611738591214a13f,georgejaymcmc/zotero,app/assets/commandLineHandler.js,commandLineHandler.js,app/assets,.js,javascript,source,None,None,None,True
2,file_da78ee625cb55359,georgejaymcmc/zotero,app/assets/prefs.js,prefs.js,app/assets,.js,javascript,source,None,None,None,True
3,file_bc42db590d995f48,georgejaymcmc/zotero,chrome/content/scaffold/content.js,content.js,chrome/content/scaffold,.js,javascript,source,None,None,None,True
4,file_02dccdc8a7fbf869,georgejaymcmc/zotero,chrome/content/scaffold/load.js,load.js,chrome/content/scaffold,.js,javascript,source,None,None,None,True
5,file_767abb7cb0102665,georgejaymcmc/zotero,chrome/content/scaffold/scaffold.js,scaffold.js,chrome/content/scaffold,.js,javascript,source,None,None,None,True
6,file_18307e89f1ddb102,georgejaymcmc/zotero,chrome/content/scaffold/scaffoldItemPreview.js,scaffoldItemPreview.js,chrome/content/scaffold,.js,javascript,source,None,None,None,True
7,file_a6b19f7562edaee3,georgejaymcmc/zotero,chrome/content/scaffold/scaffoldItemPreviews.js,scaffoldItemPreviews.js,chrome/content/scaffold,.js,javascript,source,None,None,None,True
8,file_2f58a1107b53283e,georgejaymcmc/zotero,chrome/content/scaffold/select.js,select.js,chrome/content/scaffold,.js,javascript,source,None,None,None,True
9,file_89397ac970c2c582,georgejaymcmc/zotero,chrome/content/scaffold/templates/newWeb.js,newWeb.js,chrome/content/scaffold/templates,.js,javascript,source,None,None,None,True


Discovered 578 JavaScript files
AST candidates: 490
Resource files retained: 88


In [49]:
# Inspect the resource files
resource_files_df = files_df[
    files_df["ast_candidate"] == False
].copy()

print("Resource files:", len(resource_files_df))

display(
    resource_files_df[
        ["file_path", "extension", "file_size"]
    ].sort_values("file_path")
)

Resource files: 88


,file_path,extension,file_size
357,resource/PluralForm.mjs,.mjs,None
358,resource/allfaz.mjs,.mjs,None
359,resource/classnames.js,.js,None
360,resource/concurrentCaller.mjs,.mjs,None
361,resource/config.mjs,.mjs,None
...,...,...,...
440,resource/vs/language/json/jsonWorker.js,.js,None
441,resource/vs/language/typescript/tsMode.js,.js,None
442,resource/vs/language/typescript/tsWorker.js,.js,None
443,resource/vs/loader.js,.js,None


In [50]:
# Summarise resource files by directory
resource_files_df["resource_subdirectory"] = (
    resource_files_df["directory"]
    .apply(lambda x: x.split("/")[1] if "/" in x else "(root)")
)

display(
    resource_files_df.groupby("resource_subdirectory")
    .agg(
        file_count=("file_path", "count"),
        extensions=("extension", lambda x: ", ".join(sorted(x.unique())))
    )
    .sort_values("file_count", ascending=False)
)

,file_count,extensions
resource_subdirectory,,
tinymce,53,.js
(root),21,".js, .mjs"
vs,12,.js
feeds,2,.mjs


###### 3. Parse `javascript` files

In [51]:
source_by_file = {}
retrieval_errors = []

for row in files_df.itertuples(index=False):

    try:
        source_by_file[row.file_path] = github.get_file_content(
            owner,
            repo,
            row.file_path
        )

    except Exception as e:
        retrieval_errors.append({
            "file_path": row.file_path,
            "error": str(e)
        })

print(f"Retrieved: {len(source_by_file):,} files")
print(f"Errors: {len(retrieval_errors):,}")

Retrieved: 550 files
Errors: 28


In [65]:
# ---------------------------------------------------------
# Step 3 — Validate Tree-sitter parsing
# ---------------------------------------------------------

ast_trees = {}
parse_errors = []
missing_source = []

ast_files_df = files_df.copy()

for row in ast_files_df.itertuples(index=False):

    file_path = row.file_path

    # Skip files that could not be retrieved
    if file_path not in source_by_file:
        missing_source.append(file_path)
        continue

    try:
        source_bytes = source_by_file[file_path].encode("utf-8")

        ast_trees[file_path] = parser.parse(source_bytes)

    except Exception as e:
        parse_errors.append({
            "file_path": file_path,
            "error": str(e)
        })


files_with_syntax_errors = [
    file_path
    for file_path, tree in ast_trees.items()
    if tree.root_node.has_error
]


print(f"AST candidates: {len(ast_files_df):,}")
print(f"Successfully parsed: {len(ast_trees):,}")
print(f"Missing source: {len(missing_source):,}")
print(f"Parser exceptions: {len(parse_errors):,}")
print(f"Files with syntax errors: {len(files_with_syntax_errors):,}")

AST candidates: 578
Successfully parsed: 550
Missing source: 28
Parser exceptions: 0
Files with syntax errors: 1


In [67]:
# Examine files with parsing errors if there are any
syntax_error_records = []

for file_path, tree in ast_trees.items():

    root = tree.root_node

    if root.has_error:
        error_nodes = [
            node
            for node in walk_tree(root)
            if node.type == "ERROR" or node.is_missing
        ]

        syntax_error_records.append({
            "file_path": file_path,
            "error_node_count": len(error_nodes),
            "error_types": sorted({
                node.type for node in error_nodes
            }),
            "error_locations": [
                {
                    "type": node.type,
                    **node_location(node),
                    "text": node_text(
                        node,
                        source_by_file[file_path].encode("utf-8")
                    )[:100]
                }
                for node in error_nodes[:5]
            ]
        })


# Only create and display the DataFrame if errors were found
if syntax_error_records:

    syntax_errors_df = pd.DataFrame(syntax_error_records)

    display(
        syntax_errors_df[
            ["file_path", "error_node_count", "error_types"]
        ]
    )

else:
    print("No parsing errors detected")


,file_path,error_node_count,error_types
0,test/tests/rtfScanTest.js,14,[ERROR]


###### 4. AST Inspection Utilities

In [74]:
# Print ast helper function
def print_ast(node, source_code: bytes, depth=0, max_depth=4):

    if depth > max_depth:
        return

    indent = "  " * depth
    text = node_text(node, source_code).replace("\n", " ")

    if len(text) > 120:
        text = text[:117] + "..."

    location = node_location(node)

    print(
        f"{indent}{node.type} "
        f"[{location['start_line']}:{location['start_column']}] "
        f"{text}"
    )

    for child in node.named_children:
        print_ast(
            child,
            source_code,
            depth + 1,
            max_depth
        )

In [75]:
# Inspect one file by relative path.
def inspect_file(relative_path, max_depth=4):

    match = files_df.loc[
        files_df.file_path == relative_path
    ]

    if match.empty:
        raise KeyError(
            f"File not found in files_df: {relative_path}"
        )

    if relative_path not in source_by_file:
        raise KeyError(
            f"Source code not available: {relative_path}"
        )

    if relative_path not in ast_trees:
        raise KeyError(
            f"AST not available for file: {relative_path}"
        )

    row = match.iloc[0]
    tree = ast_trees[relative_path]

    # Convert retrieved text to bytes for Tree-sitter utilities.
    source_bytes = source_by_file[relative_path].encode("utf-8")

    print(f"FILE: {relative_path}")
    print(f"FILE TYPE: {row.file_type}")
    print(f"AST CANDIDATE: {row.ast_candidate}")
    print(f"SYNTAX ERRORS: {tree.root_node.has_error}")
    print("=" * 80)

    print_ast(
        tree.root_node,
        source_bytes,
        max_depth=max_depth
    )

In [76]:
inspect_file("chrome/content/zotero/createParentDialog.js", max_depth=4)

FILE: chrome/content/zotero/createParentDialog.js
FILE TYPE: source
AST CANDIDATE: True
SYNTAX ERRORS: False
program [1:1] /*     ***** BEGIN LICENSE BLOCK *****      Copyright © 2009 Center for History and New Media                      Ge...
  comment [1:1] /*     ***** BEGIN LICENSE BLOCK *****      Copyright © 2009 Center for History and New Media                      Ge...
  comment [26:1] /* global Zotero_Lookup: false */
  lexical_declaration [28:1] const ZoteroCreateParentDialog = { // eslint-disable-line no-unused-vars 	init() { 		this.io = window.arguments[0];  ...
    variable_declarator [28:7] ZoteroCreateParentDialog = { // eslint-disable-line no-unused-vars 	init() { 		this.io = window.arguments[0];  		this...
      identifier [28:7] ZoteroCreateParentDialog
      object [28:34] { // eslint-disable-line no-unused-vars 	init() { 		this.io = window.arguments[0];  		this.inputEl = document.getElem...
        comment [28:36] // eslint-disable-line no-unused-vars
        metho

In [77]:
target_file = "chrome/content/zotero/createParentDialog.js"

print("In source_by_file:", target_file in source_by_file)
print("In ast_trees:", target_file in ast_trees)

In source_by_file: True
In ast_trees: True


In [78]:
for file_path in source_by_file:
    if "createParentDialog.js" in file_path:
        print("Source:", file_path)

for file_path in ast_trees:
    if "createParentDialog.js" in file_path:
        print("AST:", file_path)

Source: chrome/content/zotero/createParentDialog.js
AST: chrome/content/zotero/createParentDialog.js


In [79]:
# Validating Tree-sitter parsing

ast_trees = {}
parse_errors = []

ast_files_df = files_df.copy()

missing_source = []

for row in ast_files_df.itertuples(index=False):

    file_path = row.file_path

    if file_path not in source_by_file:
        missing_source.append(file_path)
        continue

    try:
        source_bytes = source_by_file[file_path].encode("utf-8")

        ast_trees[file_path] = parser.parse(source_bytes)

    except Exception as e:
        parse_errors.append({
            "file_path": file_path,
            "error": str(e)
        })

files_with_syntax_errors = [
    file_path
    for file_path, tree in ast_trees.items()
    if tree.root_node.has_error
]

print(f"Files in inventory: {len(files_df):,}")
print(f"AST candidates: {len(ast_files_df):,}")
print(f"Successfully parsed: {len(ast_trees):,}")
print(f"Parser exceptions: {len(parse_errors):,}")
print(f"Missing source: {len(missing_source):,}")
print(f"Files with syntax errors: {len(files_with_syntax_errors):,}")

Files in inventory: 578
AST candidates: 578
Successfully parsed: 550
Parser exceptions: 0
Missing source: 28
Files with syntax errors: 1


In [80]:
# Inspect source retrieval failures

retrieval_errors_df = pd.DataFrame(retrieval_errors)

print("Retrieval failures:", len(retrieval_errors_df))

display(
    retrieval_errors_df[
        ["file_path", "error"]
    ].sort_values("file_path")
)

Retrieval failures: 28


,file_path,error
0,ROOT/eslint.config.mjs,Unable to retrieve 'ROOT/eslint.config.mjs' fr...
1,resource/classnames.js,Unable to retrieve 'resource/classnames.js' fr...
2,resource/jspath.js,Unable to retrieve 'resource/jspath.js' from G...
3,resource/monacopilot-core.mjs,Unable to retrieve 'resource/monacopilot-core....
4,resource/monacopilot.mjs,Unable to retrieve 'resource/monacopilot.mjs' ...
5,resource/pako.js,Unable to retrieve 'resource/pako.js' from Git...
6,resource/prop-types.js,Unable to retrieve 'resource/prop-types.js' fr...
7,resource/react-autosuggest.js,Unable to retrieve 'resource/react-autosuggest...
8,resource/react-dom-server.js,Unable to retrieve 'resource/react-dom-server....
9,resource/react-dom.js,Unable to retrieve 'resource/react-dom.js' fro...


** Parsing error to be looked into **

That's useful. The error:

Unexpected GitHub encoding: None

suggests the problem is in your github.get_file_content() service, specifically how it handles the GitHub API response encoding.

GitHub's Contents API normally returns Base64-encoded content for files. However, the encoding field can be absent or None in certain responses, particularly for larger files or different response formats.
What I suggest
Let's inspect the implementation of get_file_content() before changing it.

###### 5. Symbol Extraction

In [83]:
# Helper function
def child_by_type(node, node_type):
    """
    Return the first direct named child matching node_type.
    """
    if node is None:
        return None

    for child in node.named_children:
        if child.type == node_type:
            return child

    return None

In [84]:
FUNCTION_TYPES = {
    "function_declaration", "generator_function_declaration",
    "function_expression", "generator_function", "arrow_function",
    "method_definition"
}
CLASS_TYPES = {"class_declaration", "class"}
VARIABLE_DECL_TYPES = {"lexical_declaration", "variable_declaration"}
SCOPE_NODE_TYPES = {
    "program", "function_declaration", "generator_function_declaration",
    "function_expression", "generator_function", "arrow_function",
    "method_definition", "statement_block", "class_declaration", "class",
    "catch_clause", "for_statement", "for_in_statement"
}


def first_identifier(node):
    if node is None:
        return None
    for n in walk_tree(node):
        if n.type in {"identifier", "property_identifier", "shorthand_property_identifier"}:
            return n
    return None


def declaration_name_node(node):
    # Function/class declarations: name is usually a direct identifier child.
    for child in node.named_children:
        if child.type == "identifier":
            return child
    return None


def extract_symbols(file_id: str, tree, source_code: bytes) -> pd.DataFrame:
    records = []
    root = tree.root_node

    for node in walk_tree(root):
        symbol_type = None
        name_node = None
        declaration_type = node.type

        if node.type in {"function_declaration", "generator_function_declaration"}:
            symbol_type = "function"
            name_node = declaration_name_node(node)
        elif node.type in {"class_declaration", "class"}:
            symbol_type = "class"
            name_node = declaration_name_node(node)
        elif node.type == "method_definition":
            symbol_type = "method"
            name_node = child_by_type(node, "property_identifier") or first_identifier(node)
        elif node.type == "variable_declarator":
            name_node = child_by_type(node, "identifier")
            if name_node is None:
                # Destructuring is retained as a declaration record, but individual
                # bound names are expanded in a later enhancement.
                name_node = child_by_type(node, "array_pattern") or child_by_type(node, "object_pattern")
            if name_node is not None:
                parent = node.parent
                kind = parent.type if parent else "variable_declarator"
                symbol_type = "variable"
                declaration_type = kind
        elif node.type == "function_expression":
            symbol_type = "function_expression"
            name_node = declaration_name_node(node)
            if name_node is None:
                name_node = node  # anonymous function; synthetic name below
        elif node.type == "arrow_function":
            symbol_type = "arrow_function"
            name_node = node  # synthetic name below

        if symbol_type is None or name_node is None:
            continue

        name = node_text(name_node, source_code).strip()
        if name_node == node:
            name = f"<anonymous_{node.start_point.row + 1}_{node.start_point.column + 1}>"

        loc = node_location(node)
        records.append({
            "symbol_id": stable_id(file_id, symbol_type, name, loc["start_byte"], prefix="sym_"),
            "file_id": file_id,
            "name": name,
            "qualified_name": None,  # populated after parent/scope analysis
            "symbol_type": symbol_type,
            "declaration_type": declaration_type,
            "parent_symbol_id": None,
            "scope_id": None,
            "is_exported": None,  # populated from exports_df
            "start_line": loc["start_line"],
            "end_line": loc["end_line"],
            "start_column": loc["start_column"],
            "end_column": loc["end_column"],
            "start_byte": loc["start_byte"],
            "end_byte": loc["end_byte"],
            "source_text": node_text(node, source_code),
        })

    return pd.DataFrame(records)

# Extract symbols from all successfully parsed files

symbol_frames = []

for fid, tree in ast_trees.items():

    source_bytes = source_by_file[fid].encode("utf-8")

    symbol_df = extract_symbols(
        fid,
        tree,
        source_bytes
    )

    if not symbol_df.empty:
        symbol_frames.append(symbol_df)

symbols_df = (
    pd.concat(symbol_frames, ignore_index=True)
    if symbol_frames
    else pd.DataFrame()
)

display(symbols_df.head(10))

print(f"Files processed: {len(ast_trees):,}")
print(f"Symbols extracted: {len(symbols_df):,}")
print(f"Unique files represented: {symbols_df['file_id'].nunique():,}")

,symbol_id,file_id,name,qualified_name,symbol_type,declaration_type,parent_symbol_id,scope_id,is_exported,start_line,end_line,start_column,end_column,start_byte,end_byte,source_text
0,sym_eeadad4dc0560a5c,app/assets/commandLineHandler.js,"{ CommandLineOptions, TestOptions }",None,variable,lexical_declaration,None,None,None,1,1,5,127,4,126,"{ CommandLineOptions, TestOptions } = ChromeUt..."
1,sym_3b0d08eff5808038,app/assets/commandLineHandler.js,processTestOptions,None,variable,variable_declaration,None,None,None,32,32,5,31,1403,1429,processTestOptions = false
2,sym_d3cd37ca2d6ec503,app/assets/commandLineHandler.js,<anonymous_69_3>,None,function_expression,function_expression,None,None,None,69,103,3,3,2856,3935,async function () {\n\t\ttry {\n\t\t\tlet port...
3,sym_21cf878906be0255,app/assets/commandLineHandler.js,portOrPath,None,variable,lexical_declaration,None,None,None,71,71,8,93,2891,2976,portOrPath = Services.prefs.getBranch('').getI...
4,sym_0db84f83c1429247,app/assets/commandLineHandler.js,{ DevToolsLoader },None,variable,lexical_declaration,None,None,None,73,75,10,5,2991,3099,{ DevToolsLoader } = ChromeUtils.importESModul...
5,sym_281dac345cf207b2,app/assets/commandLineHandler.js,loader,None,variable,lexical_declaration,None,None,None,76,78,10,6,3110,3173,loader = new DevToolsLoader({\n\t\t\t\tfreshCo...
6,sym_962f420570f783cf,app/assets/commandLineHandler.js,{ DevToolsServer },None,variable,lexical_declaration,None,None,None,79,79,10,80,3184,3254,"{ DevToolsServer } = loader.require(""devtools/..."
7,sym_b30c6e0a7b612583,app/assets/commandLineHandler.js,{ SocketListener },None,variable,lexical_declaration,None,None,None,80,80,10,80,3265,3335,"{ SocketListener } = loader.require(""devtools/..."
8,sym_340803e89f92fd2d,app/assets/commandLineHandler.js,socketOptions,None,variable,lexical_declaration,None,None,None,90,90,10,40,3571,3601,socketOptions = { portOrPath }
9,sym_c5727e816fee1abb,app/assets/commandLineHandler.js,listener,None,variable,lexical_declaration,None,None,None,91,91,10,70,3612,3672,"listener = new SocketListener(DevToolsServer, ..."


Files processed: 550
Symbols extracted: 63,228
Unique files represented: 499


In [85]:
# Outputs of symbols_df
print(symbols_df.head(10))
symbols_df.columns.tolist()

              symbol_id                           file_id  \
0  sym_eeadad4dc0560a5c  app/assets/commandLineHandler.js   
1  sym_3b0d08eff5808038  app/assets/commandLineHandler.js   
2  sym_d3cd37ca2d6ec503  app/assets/commandLineHandler.js   
3  sym_21cf878906be0255  app/assets/commandLineHandler.js   
4  sym_0db84f83c1429247  app/assets/commandLineHandler.js   
5  sym_281dac345cf207b2  app/assets/commandLineHandler.js   
6  sym_962f420570f783cf  app/assets/commandLineHandler.js   
7  sym_b30c6e0a7b612583  app/assets/commandLineHandler.js   
8  sym_340803e89f92fd2d  app/assets/commandLineHandler.js   
9  sym_c5727e816fee1abb  app/assets/commandLineHandler.js   

                                  name qualified_name          symbol_type  \
0  { CommandLineOptions, TestOptions }           None             variable   
1                   processTestOptions           None             variable   
2                     <anonymous_69_3>           None  function_expression   
3               

['symbol_id',
 'file_id',
 'name',
 'qualified_name',
 'symbol_type',
 'declaration_type',
 'parent_symbol_id',
 'scope_id',
 'is_exported',
 'start_line',
 'end_line',
 'start_column',
 'end_column',
 'start_byte',
 'end_byte',
 'source_text']

In [87]:
# Check distribution of symbol extraction
display(
    symbols_df["symbol_type"]
    .value_counts()
    .to_frame("count")
)

,count
symbol_type,
variable,40539
function_expression,13940
arrow_function,4527
method,2692
function,1377
class,153


In [88]:
# Check symbols of representative file
target_file = "chrome/content/zotero/createParentDialog.js"

display(
    symbols_df.loc[
        symbols_df["file_id"] == target_file,
        [
            "name",
            "symbol_type",
            "declaration_type",
            "start_line",
            "end_line",
            "source_text"
        ]
    ].head(30)
)

,name,symbol_type,declaration_type,start_line,end_line,source_text
3242,ZoteroCreateParentDialog,variable,lexical_declaration,28,95,ZoteroCreateParentDialog = { // eslint-disable...
3243,init,method,method_definition,29,48,init() {\n\t\tthis.io = window.arguments[0];\n...
3244,performLookup,method,method_definition,50,62,async performLookup() {\n\t\tlet newItems = aw...
3245,newItems,variable,lexical_declaration,51,55,newItems = await Zotero_Lookup.addItemsFromIde...
3246,handleInput,method,method_definition,64,67,handleInput(event) {\n\t\tconst input = event....
3247,input,variable,lexical_declaration,65,65,input = event.target.value.trim()
3248,handleStatusChange,method,method_definition,69,79,handleStatusChange(isLookingUp)\t{\n\t\tthis.i...
3249,handleAcceptClick,method,method_definition,81,89,handleAcceptClick(ev) {\n\t\tev.preventDefault...
3250,handleManualEntry,method,method_definition,91,94,handleManualEntry() {\n\t\tthis.io.dataOut = {...


###### 6. Scope extraction and symbol-to-scope assignment

In [89]:
def extract_scopes(file_id: str, tree) -> tuple[pd.DataFrame, dict[int, str]]:
    """Create lexical-scope candidates and map AST node start bytes to scope IDs.

    This is an initial structural scope model. JavaScript's exact var/let/const
    binding rules are applied during resolution, not inferred solely from this tree.
    """
    root = tree.root_node
    records = []
    node_scope = {}

    def visit(node, current_scope_id=None, parent_scope_id=None):
        creates_scope = node.type in SCOPE_NODE_TYPES
        scope_id = current_scope_id

        if creates_scope:
            loc = node_location(node)
            scope_id = stable_id(file_id, node.type, loc["start_byte"], prefix="scope_")
            records.append({
                "scope_id": scope_id,
                "file_id": file_id,
                "parent_scope_id": parent_scope_id,
                "scope_type": (
                    "module" if node.type == "program" else
                    "function" if node.type in FUNCTION_TYPES else
                    "class" if node.type in CLASS_TYPES else
                    "block" if node.type == "statement_block" else
                    "catch" if node.type == "catch_clause" else
                    node.type
                ),
                "owner_symbol_id": None,
                "start_line": loc["start_line"],
                "end_line": loc["end_line"],
                "start_byte": loc["start_byte"],
                "end_byte": loc["end_byte"],
            })
            parent_for_children = scope_id
        else:
            parent_for_children = parent_scope_id

        node_scope[node.start_byte] = scope_id
        for child in node.named_children:
            visit(child, scope_id, parent_for_children)

    visit(root)
    return pd.DataFrame(records), node_scope


scope_frames = []
scope_maps = {}
for fid, tree in ast_trees.items():
    sdf, node_map = extract_scopes(fid, tree)
    scope_frames.append(sdf)
    scope_maps[fid] = node_map

scopes_df = pd.concat(scope_frames, ignore_index=True) if scope_frames else pd.DataFrame()


# Assign each symbol to the narrowest containing scope by source range.
def assign_symbol_scopes(symbols: pd.DataFrame, scopes: pd.DataFrame) -> pd.DataFrame:
    if symbols.empty or scopes.empty:
        return symbols
    result = symbols.copy()
    for idx, sym in result.iterrows():
        candidates = scopes[
            (scopes.file_id == sym.file_id) &
            (scopes.start_byte <= sym.start_byte) &
            (scopes.end_byte >= sym.end_byte)
            ].copy()
        if not candidates.empty:
            candidates["span"] = candidates.end_byte - candidates.start_byte
            chosen = candidates.sort_values("span").iloc[0]
            result.at[idx, "scope_id"] = chosen.scope_id
    return result


symbols_df = assign_symbol_scopes(symbols_df, scopes_df)
display(scopes_df.head(10))
display(symbols_df[["name", "symbol_type", "scope_id", "start_line"]].head(10))


,scope_id,file_id,parent_scope_id,scope_type,owner_symbol_id,start_line,end_line,start_byte,end_byte
0,scope_54aafa45f9b4cdc9,app/assets/commandLineHandler.js,NaN,module,None,1,105,0,3942
1,scope_fe6f655821a3ec08,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,5,7,319,354
2,scope_b07c257be1a2b99e,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,10,12,434,476
3,scope_8c90f214a6765d32,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,14,16,570,612
4,scope_09a96ddd302599c9,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,20,22,932,1014
5,scope_711571ea4929e8ff,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,28,30,1324,1397
6,scope_26951c041e1e30a3,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,33,36,1476,1540
7,scope_df92ae78ffd2c11c,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,37,39,1595,1640
8,scope_cb719a9fd91af10f,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,40,42,1698,1746
9,scope_a4f65ccc8a56fd46,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,44,66,1772,2807


,name,symbol_type,scope_id,start_line
0,"{ CommandLineOptions, TestOptions }",variable,scope_54aafa45f9b4cdc9,1
1,processTestOptions,variable,scope_54aafa45f9b4cdc9,32
2,<anonymous_69_3>,function_expression,scope_4114a771dc9a0d0c,69
3,portOrPath,variable,scope_96b025d4493a471e,71
4,{ DevToolsLoader },variable,scope_96b025d4493a471e,73
5,loader,variable,scope_96b025d4493a471e,76
6,{ DevToolsServer },variable,scope_96b025d4493a471e,79
7,{ SocketListener },variable,scope_96b025d4493a471e,80
8,socketOptions,variable,scope_96b025d4493a471e,90
9,listener,variable,scope_96b025d4493a471e,91


In [90]:
# Check function with createParentDialog.js
target_file = "chrome/content/zotero/createParentDialog.js"

display(
    symbols_df.loc[
        symbols_df["file_id"] == target_file,
        [
            "name",
            "symbol_type",
            "scope_id",
            "parent_symbol_id",
            "start_line",
            "end_line"
        ]
    ].sort_values("start_line")
)

,name,symbol_type,scope_id,parent_symbol_id,start_line,end_line
3242,ZoteroCreateParentDialog,variable,scope_e807a690d1b2f289,None,28,95
3243,init,method,scope_d9236a24d70e0404,None,29,48
3244,performLookup,method,scope_bc87c45c851b427a,None,50,62
3245,newItems,variable,scope_4b2fd8ed08e5c76d,None,51,55
3246,handleInput,method,scope_00289fee34a2c3e9,None,64,67
3247,input,variable,scope_26631897e984f0a8,None,65,65
3248,handleStatusChange,method,scope_0aefa65b6e72d731,None,69,79
3249,handleAcceptClick,method,scope_0ee1653b5718256d,None,81,89
3250,handleManualEntry,method,scope_407ab593dcbbca6a,None,91,94


In [91]:
display(scopes_df.head(10))
print(scopes_df.columns.tolist())

,scope_id,file_id,parent_scope_id,scope_type,owner_symbol_id,start_line,end_line,start_byte,end_byte
0,scope_54aafa45f9b4cdc9,app/assets/commandLineHandler.js,NaN,module,None,1,105,0,3942
1,scope_fe6f655821a3ec08,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,5,7,319,354
2,scope_b07c257be1a2b99e,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,10,12,434,476
3,scope_8c90f214a6765d32,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,14,16,570,612
4,scope_09a96ddd302599c9,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,20,22,932,1014
5,scope_711571ea4929e8ff,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,28,30,1324,1397
6,scope_26951c041e1e30a3,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,33,36,1476,1540
7,scope_df92ae78ffd2c11c,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,37,39,1595,1640
8,scope_cb719a9fd91af10f,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,40,42,1698,1746
9,scope_a4f65ccc8a56fd46,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,44,66,1772,2807


['scope_id', 'file_id', 'parent_scope_id', 'scope_type', 'owner_symbol_id', 'start_line', 'end_line', 'start_byte', 'end_byte']


In [92]:
target_file = "chrome/content/zotero/createParentDialog.js"

display(
    scopes_df.loc[
        scopes_df["file_id"] == target_file
    ].sort_values("start_byte")
)

,scope_id,file_id,parent_scope_id,scope_type,owner_symbol_id,start_line,end_line,start_byte,end_byte
4600,scope_e807a690d1b2f289,chrome/content/zotero/createParentDialog.js,NaN,module,None,1,96,0,2813
4601,scope_d9236a24d70e0404,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,29,48,1041,1844
4602,scope_e0e2d37d7ec268a0,chrome/content/zotero/createParentDialog.js,scope_d9236a24d70e0404,block,None,29,48,1048,1844
4603,scope_bc87c45c851b427a,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,50,62,1848,2166
4604,scope_4b2fd8ed08e5c76d,chrome/content/zotero/createParentDialog.js,scope_bc87c45c851b427a,block,None,50,62,1870,2166
4605,scope_295234ee1d9b9f67,chrome/content/zotero/createParentDialog.js,scope_4b2fd8ed08e5c76d,block,None,58,61,2093,2163
4606,scope_00289fee34a2c3e9,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,64,67,2170,2280
4607,scope_26631897e984f0a8,chrome/content/zotero/createParentDialog.js,scope_00289fee34a2c3e9,block,None,64,67,2189,2280
4608,scope_0aefa65b6e72d731,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,69,79,2284,2588
4609,scope_c9997f477a053827,chrome/content/zotero/createParentDialog.js,scope_0aefa65b6e72d731,block,None,69,79,2316,2588


###### 7. Import and export extraction

In [93]:
import re
import pandas as pd


# ---------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------

def named_children(node):
    return list(node.named_children) if node else []


def find_ancestor(node, node_type):
    current = node.parent if node else None

    while current:
        if current.type == node_type:
            return current
        current = current.parent

    return None


def identifier_nodes(node):
    if node is None:
        return []

    return [
        n for n in walk_tree(node)
        if n.type == "identifier"
    ]


def simple_identifier(node, source_code):
    if node is None:
        return None

    if node.type == "identifier":
        return node_text(node, source_code)

    return None


# ---------------------------------------------------------
# IMPORTS
# ---------------------------------------------------------

def extract_imports(file_id: str, tree, source_code: bytes):

    import_records = []
    binding_records = []

    for node in walk_tree(tree.root_node):

        # -------------------------------------------------
        # ES module imports
        # -------------------------------------------------

        if node.type == "import_statement":

            import_id = stable_id(
                file_id, node.start_byte, prefix="imp_"
            )

            text = node_text(node, source_code)

            source_node = next(
                (c for c in node.named_children if c.type == "string"),
                None
            )

            module_specifier = (
                node_text(source_node, source_code).strip("'\"")
                if source_node else None
            )

            clause = next(
                (c for c in node.named_children
                 if c.type == "import_clause"),
                None
            )

            if clause is None:
                import_kind = "side_effect"
            else:
                import_kind = "es_module"

            import_records.append({
                "import_id": import_id,
                "file_id": file_id,
                "module_specifier": module_specifier,
                "import_text": text,
                "import_kind": import_kind,
                "import_system": "es_module",
                **node_location(node),
            })

            if clause is None:
                continue

            # Default import:
            # import Zotero from "zotero";
            for child in clause.named_children:

                if child.type == "identifier":
                    local_name = node_text(child, source_code)

                    binding_records.append({
                        "import_binding_id": stable_id(
                            import_id, local_name, prefix="ib_"
                        ),
                        "import_id": import_id,
                        "file_id": file_id,
                        "local_name": local_name,
                        "imported_name": "default",
                        "binding_type": "default",
                        "binding_text": node_text(child, source_code),
                    })

                # Namespace import:
                # import * as Services from "Services";
                elif child.type == "namespace_import":

                    ids = identifier_nodes(child)

                    local_name = (
                        node_text(ids[-1], source_code)
                        if ids else None
                    )

                    binding_records.append({
                        "import_binding_id": stable_id(
                            import_id, local_name, prefix="ib_"
                        ),
                        "import_id": import_id,
                        "file_id": file_id,
                        "local_name": local_name,
                        "imported_name": "*",
                        "binding_type": "namespace",
                        "binding_text": node_text(child, source_code),
                    })

                # Named imports:
                # import { foo, bar as baz } from "module";
                elif child.type == "named_imports":

                    for specifier in child.named_children:

                        if specifier.type != "import_specifier":
                            continue

                        ids = identifier_nodes(specifier)

                        imported_name = (
                            node_text(ids[0], source_code)
                            if ids else None
                        )

                        local_name = (
                            node_text(ids[-1], source_code)
                            if ids else None
                        )

                        binding_records.append({
                            "import_binding_id": stable_id(
                                import_id, local_name, prefix="ib_"
                            ),
                            "import_id": import_id,
                            "file_id": file_id,
                            "local_name": local_name,
                            "imported_name": imported_name,
                            "binding_type": "named",
                            "binding_text": node_text(
                                specifier, source_code
                            ),
                        })

        # -------------------------------------------------
        # CommonJS require() imports
        # -------------------------------------------------

        elif node.type == "call_expression":

            fn = child_by_type(node, "identifier")
            args = child_by_type(node, "arguments")

            if not fn or node_text(fn, source_code) != "require":
                continue

            if not args:
                continue

            string_arg = next(
                (c for c in args.named_children if c.type == "string"),
                None
            )

            module_specifier = (
                node_text(string_arg, source_code).strip("'\"")
                if string_arg else None
            )

            import_id = stable_id(
                file_id, node.start_byte, prefix="imp_"
            )

            import_text = node_text(node, source_code)

            import_records.append({
                "import_id": import_id,
                "file_id": file_id,
                "module_specifier": module_specifier,
                "import_text": import_text,
                "import_kind": "commonjs_require",
                "import_system": "commonjs",
                **node_location(node),
            })

            # Find a containing variable declaration, if present.
            declarator = find_ancestor(node, "variable_declarator")

            if declarator:

                declarator_children = declarator.named_children

                if declarator_children:
                    target = declarator_children[0]

                    # const module = require("module");
                    if target.type == "identifier":

                        local_name = node_text(target, source_code)

                        binding_records.append({
                            "import_binding_id": stable_id(
                                import_id, local_name, prefix="ib_"
                            ),
                            "import_id": import_id,
                            "file_id": file_id,
                            "local_name": local_name,
                            "imported_name": None,
                            "binding_type": "module",
                            "binding_text": node_text(
                                target, source_code
                            ),
                        })

                    # const { foo, bar: baz } = require("module");
                    elif target.type in {
                        "object_pattern", "array_pattern"
                    }:

                        for identifier in identifier_nodes(target):

                            local_name = node_text(
                                identifier, source_code
                            )

                            binding_records.append({
                                "import_binding_id": stable_id(
                                    import_id,
                                    identifier.start_byte,
                                    prefix="ib_"
                                ),
                                "import_id": import_id,
                                "file_id": file_id,
                                "local_name": local_name,
                                "imported_name": None,
                                "binding_type": "named",
                                "binding_text": node_text(
                                    identifier, source_code
                                ),
                            })

            else:
                # require("module") with no obvious variable binding.
                binding_records.append({
                    "import_binding_id": stable_id(
                        import_id, "unbound", prefix="ib_"
                    ),
                    "import_id": import_id,
                    "file_id": file_id,
                    "local_name": None,
                    "imported_name": None,
                    "binding_type": "unbound",
                    "binding_text": import_text,
                })

    return (
        pd.DataFrame(import_records),
        pd.DataFrame(binding_records)
    )


# ---------------------------------------------------------
# EXPORTS
# ---------------------------------------------------------

def extract_exports(file_id: str, tree, source_code: bytes):

    export_records = []
    binding_records = []

    for node in walk_tree(tree.root_node):

        # -------------------------------------------------
        # ES module exports
        # -------------------------------------------------

        if node.type == "export_statement":

            export_id = stable_id(
                file_id, node.start_byte, prefix="exp_"
            )

            export_text = node_text(node, source_code)

            export_records.append({
                "export_id": export_id,
                "file_id": file_id,
                "export_text": export_text,
                "export_kind": "es_module",
                "export_system": "es_module",
                **node_location(node),
            })

            export_clause = next(
                (c for c in node.named_children
                 if c.type == "export_clause"),
                None
            )

            if export_clause:

                for specifier in export_clause.named_children:

                    if specifier.type != "export_specifier":
                        continue

                    ids = identifier_nodes(specifier)

                    local_name = (
                        node_text(ids[0], source_code)
                        if ids else None
                    )

                    exported_name = (
                        node_text(ids[-1], source_code)
                        if ids else None
                    )

                    binding_records.append({
                        "export_binding_id": stable_id(
                            export_id,
                            specifier.start_byte,
                            prefix="eb_"
                        ),
                        "export_id": export_id,
                        "file_id": file_id,
                        "local_name": local_name,
                        "exported_name": exported_name,
                        "binding_type": "export_clause",
                        "binding_text": node_text(
                            specifier, source_code
                        ),
                    })

                continue

            # Exported declaration:
            # export function foo() {}
            # export class Foo {}
            # export const foo = ...
            declaration = next(
                (
                    c for c in node.named_children
                    if c.type in {
                        "function_declaration",
                        "generator_function_declaration",
                        "class_declaration",
                        "lexical_declaration",
                        "variable_declaration"
                    }
                ),
                None
            )

            if declaration:

                candidates = []

                if declaration.type in {
                    "function_declaration",
                    "generator_function_declaration",
                    "class_declaration"
                }:
                    ids = identifier_nodes(declaration)

                    if ids:
                        candidates.append(ids[0])

                else:
                    for declarator in walk_tree(declaration):
                        if declarator.type == "variable_declarator":
                            target = (
                                declarator.named_children[0]
                                if declarator.named_children else None
                            )

                            if target and target.type == "identifier":
                                candidates.append(target)

                for candidate in candidates:

                    local_name = node_text(candidate, source_code)

                    binding_records.append({
                        "export_binding_id": stable_id(
                            export_id,
                            candidate.start_byte,
                            prefix="eb_"
                        ),
                        "export_id": export_id,
                        "file_id": file_id,
                        "local_name": local_name,
                        "exported_name": local_name,
                        "binding_type": "declaration",
                        "binding_text": node_text(
                            declaration, source_code
                        ),
                    })

            else:
                # Default expression or another export form not
                # represented by the patterns above.
                binding_records.append({
                    "export_binding_id": stable_id(
                        export_id, "expression", prefix="eb_"
                    ),
                    "export_id": export_id,
                    "file_id": file_id,
                    "local_name": None,
                    "exported_name": (
                        "default" if "default" in export_text else None
                    ),
                    "binding_type": "expression",
                    "binding_text": export_text,
                })

        # -------------------------------------------------
        # CommonJS exports
        # -------------------------------------------------

        elif node.type == "assignment_expression":

            children = node.named_children

            if len(children) < 2:
                continue

            left = children[0]
            right = children[-1]

            left_text = re.sub(
                r"\s+", "", node_text(left, source_code)
            )

            export_kind = None
            exported_name = None

            if left_text == "module.exports":
                export_kind = "module_assignment"
                exported_name = "default"

            elif left_text.startswith("module.exports."):
                export_kind = "module_property_assignment"
                exported_name = left_text.split(
                    "module.exports.", 1
                )[1]

            elif left_text.startswith("exports."):
                export_kind = "exports_property_assignment"
                exported_name = left_text.split(
                    "exports.", 1
                )[1]

            if export_kind is None:
                continue

            export_id = stable_id(
                file_id, node.start_byte, prefix="exp_"
            )

            export_text = node_text(node, source_code)
            local_name = simple_identifier(right, source_code)

            export_records.append({
                "export_id": export_id,
                "file_id": file_id,
                "export_text": export_text,
                "export_kind": export_kind,
                "export_system": "commonjs",
                **node_location(node),
            })

            binding_records.append({
                "export_binding_id": stable_id(
                    export_id, "binding", prefix="eb_"
                ),
                "export_id": export_id,
                "file_id": file_id,
                "local_name": local_name,
                "exported_name": exported_name,
                "binding_type": (
                    "identifier" if local_name else "expression"
                ),
                "binding_text": node_text(right, source_code),
            })

    return (
        pd.DataFrame(export_records),
        pd.DataFrame(binding_records)
    )


# ---------------------------------------------------------
# Run extraction across all successfully parsed files
# ---------------------------------------------------------

import_frames = []
import_binding_frames = []
export_frames = []
export_binding_frames = []

for fid, tree in ast_trees.items():

    source_bytes = source_by_file[fid].encode("utf-8")

    imports, import_bindings = extract_imports(
        fid, tree, source_bytes
    )

    exports, export_bindings = extract_exports(
        fid, tree, source_bytes
    )

    if not imports.empty:
        import_frames.append(imports)

    if not import_bindings.empty:
        import_binding_frames.append(import_bindings)

    if not exports.empty:
        export_frames.append(exports)

    if not export_bindings.empty:
        export_binding_frames.append(export_bindings)


imports_df = (
    pd.concat(import_frames, ignore_index=True)
    if import_frames else pd.DataFrame()
)

import_bindings_df = (
    pd.concat(import_binding_frames, ignore_index=True)
    if import_binding_frames else pd.DataFrame()
)

exports_df = (
    pd.concat(export_frames, ignore_index=True)
    if export_frames else pd.DataFrame()
)

export_bindings_df = (
    pd.concat(export_binding_frames, ignore_index=True)
    if export_binding_frames else pd.DataFrame()
)


print(f"Import statements: {len(imports_df):,}")
print(f"Import bindings:   {len(import_bindings_df):,}")
print(f"Export statements: {len(exports_df):,}")
print(f"Export bindings:   {len(export_bindings_df):,}")

display(imports_df.head(10))
display(import_bindings_df.head(10))
display(exports_df.head(10))
display(export_bindings_df.head(10))

Import statements: 319
Import bindings:   289
Export statements: 225
Export bindings:   269


,import_id,file_id,module_specifier,import_text,import_kind,import_system,start_line,start_column,end_line,end_column,start_byte,end_byte
0,imp_d158a1fb0518df49,chrome/content/scaffold/zoteroWebTranslationEn...,chrome://zotero/content/xpcom/translate/testTr...,import { AbstractWebTranslationEnvironment } f...,es_module,es_module,1,1,1,130,0,129
1,imp_e493445f19fe03e3,chrome/content/zotero/HiddenBrowser.mjs,chrome://zotero/content/BlockingObserver.mjs,"import { BlockingObserver } from ""chrome://zot...",es_module,es_module,27,1,27,81,939,1019
2,imp_cb4be66627adec70,chrome/content/zotero/HiddenBrowser.mjs,chrome://zotero/content/actors/ActorManager.mjs,"import ""chrome://zotero/content/actors/ActorMa...",side_effect,es_module,28,1,28,58,1020,1077
3,imp_7069ee4ea240432e,chrome/content/zotero/ZoteroProtocolHandler.mjs,resource://gre/modules/NetUtil.sys.mjs,"import { NetUtil } from ""resource://gre/module...",es_module,es_module,35,1,35,66,1361,1426
4,imp_d1ad3d311f92135c,chrome/content/zotero/ZoteroProtocolHandler.mjs,chrome://zotero/content/zotero.mjs,"import { Zotero } from ""chrome://zotero/conten...",es_module,es_module,45,1,45,61,1676,1736
5,imp_cd2b7380c8bcdbd0,chrome/content/zotero/ace/aceWrapper.js,ace/mode/javascript,"require(""ace/mode/javascript"")",commonjs_require,commonjs,37,19,37,49,1245,1275
6,imp_0931f2c723de92ab,chrome/content/zotero/ace/aceWrapper.js,ace/mode/text,"require(""ace/mode/text"")",commonjs_require,commonjs,38,13,38,37,1294,1318
7,imp_d0c66847273a737a,chrome/content/zotero/ace/aceWrapper.js,ace/edit_session,"require(""ace/edit_session"")",commonjs_require,commonjs,39,16,39,43,1340,1367
8,imp_687d376ff5f69a67,chrome/content/zotero/actors/ActorManager.mjs,resource://gre/modules/ActorManagerParent.sys.mjs,"import ""resource://gre/modules/ActorManagerPar...",side_effect,es_module,2,1,2,60,27,86
9,imp_726a4a4a2ffc2e07,chrome/content/zotero/actors/DocumentIsReadyCh...,chrome://zotero/content/actors/actorUtils.mjs,"import { documentIsReady } from ""chrome://zote...",es_module,es_module,1,1,1,81,0,80


,import_binding_id,import_id,file_id,local_name,imported_name,binding_type,binding_text
0,ib_b003f0d03c4953cf,imp_d158a1fb0518df49,chrome/content/scaffold/zoteroWebTranslationEn...,AbstractWebTranslationEnvironment,AbstractWebTranslationEnvironment,named,AbstractWebTranslationEnvironment
1,ib_8ecc706c97eb3119,imp_e493445f19fe03e3,chrome/content/zotero/HiddenBrowser.mjs,BlockingObserver,BlockingObserver,named,BlockingObserver
2,ib_5d368b4588c05bd5,imp_7069ee4ea240432e,chrome/content/zotero/ZoteroProtocolHandler.mjs,NetUtil,NetUtil,named,NetUtil
3,ib_d1c684c600085b0f,imp_d1ad3d311f92135c,chrome/content/zotero/ZoteroProtocolHandler.mjs,Zotero,Zotero,named,Zotero
4,ib_163faae4e036ef0d,imp_cd2b7380c8bcdbd0,chrome/content/zotero/ace/aceWrapper.js,None,None,unbound,"require(""ace/mode/javascript"")"
5,ib_0655920dd0ec4ee6,imp_0931f2c723de92ab,chrome/content/zotero/ace/aceWrapper.js,None,None,unbound,"require(""ace/mode/text"")"
6,ib_d3dea4f2dec3ee92,imp_d0c66847273a737a,chrome/content/zotero/ace/aceWrapper.js,None,None,unbound,"require(""ace/edit_session"")"
7,ib_54806d3898f016d4,imp_726a4a4a2ffc2e07,chrome/content/zotero/actors/DocumentIsReadyCh...,documentIsReady,documentIsReady,named,documentIsReady
8,ib_98862c126c4ff3f3,imp_559e3be0974421cf,chrome/content/zotero/actors/MendeleyAuthChild...,documentIsReady,documentIsReady,named,documentIsReady
9,ib_025928b548e79e6b,imp_66bed0813dfff3a2,chrome/content/zotero/actors/PageDataChild.mjs,documentIsReady,documentIsReady,named,documentIsReady


,export_id,file_id,export_text,export_kind,export_system,start_line,start_column,end_line,end_column,start_byte,end_byte
0,exp_109af1fe96eb42b7,chrome/content/scaffold/zoteroWebTranslationEn...,export class ZoteroWebTranslationEnvironment e...,es_module,es_module,3,1,77,2,131,2267
1,exp_c7b424b7751b6abe,chrome/content/zotero/BlockingObserver.mjs,export class BlockingObserver {\n\tshouldBlock...,es_module,es_module,30,1,88,2,1038,2457
2,exp_be267970a819465d,chrome/content/zotero/EPUB.mjs,export class EPUB {\n\t_zipReader;\n\n\t_conte...,es_module,es_module,39,1,190,2,1244,5679
3,exp_d23c8e3084e62bd8,chrome/content/zotero/HiddenBrowser.mjs,export class HiddenBrowser {\n\t/**\n\t * @par...,es_module,es_module,47,1,343,2,1583,11122
4,exp_dc4e9b1185dc78f1,chrome/content/zotero/RemoteTranslate.mjs,export class RemoteTranslate {\n\t_browser = n...,es_module,es_module,33,1,289,2,1229,8640
5,exp_7977c7f49d721d31,chrome/content/zotero/ZoteroProtocolHandler.mjs,export function ZoteroProtocolHandler() {\n\tt...,es_module,es_module,47,1,895,2,1738,25288
6,exp_3089388998dcc90c,chrome/content/zotero/actors/DocumentIsReadyCh...,export class DocumentIsReadyChild extends JSWi...,es_module,es_module,3,1,12,2,82,353
7,exp_22f1725401402091,chrome/content/zotero/actors/ExternalLinkHandl...,export class ExternalLinkHandlerChild extends ...,es_module,es_module,1,1,67,2,0,1698
8,exp_563c13b4ffc99d7c,chrome/content/zotero/actors/ExternalLinkHandl...,export class ExternalLinkHandlerParent extends...,es_module,es_module,5,1,14,2,99,302
9,exp_2eb99ac8bd41ea1c,chrome/content/zotero/actors/FeedAbstractChild...,export class FeedAbstractChild extends JSWindo...,es_module,es_module,1,1,62,2,0,1625


,export_binding_id,export_id,file_id,local_name,exported_name,binding_type,binding_text
0,eb_9a30e35842a1f24e,exp_109af1fe96eb42b7,chrome/content/scaffold/zoteroWebTranslationEn...,ZoteroWebTranslationEnvironment,ZoteroWebTranslationEnvironment,declaration,class ZoteroWebTranslationEnvironment extends ...
1,eb_bed3f8b4450f4b76,exp_c7b424b7751b6abe,chrome/content/zotero/BlockingObserver.mjs,BlockingObserver,BlockingObserver,declaration,class BlockingObserver {\n\tshouldBlock;\n\t\n...
2,eb_67963ec6e6105121,exp_be267970a819465d,chrome/content/zotero/EPUB.mjs,EPUB,EPUB,declaration,class EPUB {\n\t_zipReader;\n\n\t_contentOPF =...
3,eb_27b4fba1a3b40a3c,exp_d23c8e3084e62bd8,chrome/content/zotero/HiddenBrowser.mjs,HiddenBrowser,HiddenBrowser,declaration,class HiddenBrowser {\n\t/**\n\t * @param {Obj...
4,eb_b6f2e583e7315e9f,exp_dc4e9b1185dc78f1,chrome/content/zotero/RemoteTranslate.mjs,RemoteTranslate,RemoteTranslate,declaration,class RemoteTranslate {\n\t_browser = null;\n\...
5,eb_c13af8c44e49b9b1,exp_7977c7f49d721d31,chrome/content/zotero/ZoteroProtocolHandler.mjs,ZoteroProtocolHandler,ZoteroProtocolHandler,declaration,function ZoteroProtocolHandler() {\n\tthis.wra...
6,eb_a68eb6a951b5914d,exp_3089388998dcc90c,chrome/content/zotero/actors/DocumentIsReadyCh...,DocumentIsReadyChild,DocumentIsReadyChild,declaration,class DocumentIsReadyChild extends JSWindowAct...
7,eb_2beb37b50b3c05ad,exp_22f1725401402091,chrome/content/zotero/actors/ExternalLinkHandl...,ExternalLinkHandlerChild,ExternalLinkHandlerChild,declaration,class ExternalLinkHandlerChild extends JSWindo...
8,eb_affc2ba47b8322e7,exp_563c13b4ffc99d7c,chrome/content/zotero/actors/ExternalLinkHandl...,ExternalLinkHandlerParent,ExternalLinkHandlerParent,declaration,class ExternalLinkHandlerParent extends JSWind...
9,eb_e578013edd27f709,exp_2eb99ac8bd41ea1c,chrome/content/zotero/actors/FeedAbstractChild...,FeedAbstractChild,FeedAbstractChild,declaration,class FeedAbstractChild extends JSWindowActorC...


In [94]:
# Export and Import binding QA checks
qa_results = []


def qa_check(name, condition, observed, expected, reason):
    qa_results.append({
        "test": name,
        "passed": bool(condition),
        "observed": observed,
        "expected": expected,
        "reason": reason,
    })


# ---------------------------------------------------------
# 1. Statement and binding counts
# ---------------------------------------------------------

qa_check(
    "Import statements extracted",
    len(imports_df) > 0,
    len(imports_df),
    "Greater than zero",
    "Zotero contains both ES module and CommonJS imports."
)

qa_check(
    "Import bindings extracted",
    len(import_bindings_df) > 0,
    len(import_bindings_df),
    "Greater than zero",
    "Bindings are needed to identify individual imported names."
)

qa_check(
    "Export statements extracted",
    len(exports_df) > 0,
    len(exports_df),
    "Greater than zero",
    "Both ES module and CommonJS export patterns are present."
)

qa_check(
    "Export bindings extracted",
    len(export_bindings_df) > 0,
    len(export_bindings_df),
    "Greater than zero",
    "Bindings identify the names exposed by each module."
)


# ---------------------------------------------------------
# 2. Unique identifiers
# ---------------------------------------------------------

for df, column, label in [
    (imports_df, "import_id", "Import IDs"),
    (import_bindings_df, "import_binding_id", "Import binding IDs"),
    (exports_df, "export_id", "Export IDs"),
    (export_bindings_df, "export_binding_id", "Export binding IDs"),
]:

    if column in df.columns:
        duplicates = int(df[column].duplicated().sum())

        qa_check(
            f"Unique {label}",
            duplicates == 0,
            duplicates,
            0,
            "Each extracted graph entity needs a stable unique identifier."
        )


# ---------------------------------------------------------
# 3. Binding-to-statement integrity
# ---------------------------------------------------------

if not import_bindings_df.empty and not imports_df.empty:

    orphan_import_bindings = (
        ~import_bindings_df["import_id"].isin(imports_df["import_id"])
    ).sum()

    qa_check(
        "Import bindings have parent imports",
        orphan_import_bindings == 0,
        int(orphan_import_bindings),
        0,
        "Every ImportBinding should belong to an Import."
    )


if not export_bindings_df.empty and not exports_df.empty:

    orphan_export_bindings = (
        ~export_bindings_df["export_id"].isin(exports_df["export_id"])
    ).sum()

    qa_check(
        "Export bindings have parent exports",
        orphan_export_bindings == 0,
        int(orphan_export_bindings),
        0,
        "Every ExportBinding should belong to an Export."
    )


# ---------------------------------------------------------
# 4. File references
# ---------------------------------------------------------

for df, label in [
    (imports_df, "Imports"),
    (import_bindings_df, "Import bindings"),
    (exports_df, "Exports"),
    (export_bindings_df, "Export bindings"),
]:

    if not df.empty and "file_id" in df.columns:

        unknown_files = (
            ~df["file_id"].isin(ast_trees.keys())
        ).sum()

        qa_check(
            f"{label} reference known files",
            unknown_files == 0,
            int(unknown_files),
            0,
            "Each extracted entity should retain its source file."
        )


# ---------------------------------------------------------
# 5. Source location sanity
# ---------------------------------------------------------

for df, label in [
    (imports_df, "Imports"),
    (exports_df, "Exports"),
]:

    if not df.empty:

        invalid_locations = (
            df["start_line"].isna()
            | df["end_line"].isna()
            | (df["start_line"] < 1)
            | (df["end_line"] < df["start_line"])
        ).sum()

        qa_check(
            f"{label} have valid source locations",
            invalid_locations == 0,
            int(invalid_locations),
            0,
            "Source locations provide traceability back to the code."
        )


# ---------------------------------------------------------
# 6. Import and export classification
# ---------------------------------------------------------

if not imports_df.empty:

    observed_systems = set(imports_df["import_system"].dropna())

    qa_check(
        "Both import systems represented",
        {"es_module", "commonjs"}.issubset(observed_systems),
        sorted(observed_systems),
        ["commonjs", "es_module"],
        "Zotero uses both ES module and CommonJS patterns."
    )


if not exports_df.empty:

    observed_systems = set(exports_df["export_system"].dropna())

    qa_check(
        "Both export systems represented",
        {"es_module", "commonjs"}.issubset(observed_systems),
        sorted(observed_systems),
        ["commonjs", "es_module"],
        "Zotero uses both ES module and CommonJS exports."
    )


# ---------------------------------------------------------
# 7. Binding classification
# ---------------------------------------------------------

if not import_bindings_df.empty:

    qa_check(
        "Import binding types populated",
        import_bindings_df["binding_type"].notna().all(),
        import_bindings_df["binding_type"].isna().sum(),
        0,
        "Binding type helps distinguish module, named and namespace imports."
    )


if not export_bindings_df.empty:

    qa_check(
        "Export binding types populated",
        export_bindings_df["binding_type"].notna().all(),
        export_bindings_df["binding_type"].isna().sum(),
        0,
        "Binding type distinguishes declarations, clauses and expressions."
    )


# ---------------------------------------------------------
# 8. Display QA results
# ---------------------------------------------------------

qa_df = pd.DataFrame(qa_results)

display(qa_df)

print(
    f"QA checks passed: {qa_df['passed'].sum()} "
    f"/ {len(qa_df)}"
)

,test,passed,observed,expected,reason
0,Import statements extracted,True,319,Greater than zero,Zotero contains both ES module and CommonJS im...
1,Import bindings extracted,True,289,Greater than zero,Bindings are needed to identify individual imp...
2,Export statements extracted,True,225,Greater than zero,Both ES module and CommonJS export patterns ar...
3,Export bindings extracted,True,269,Greater than zero,Bindings identify the names exposed by each mo...
4,Unique Import IDs,True,0,0,Each extracted graph entity needs a stable uni...
5,Unique Import binding IDs,True,0,0,Each extracted graph entity needs a stable uni...
6,Unique Export IDs,True,0,0,Each extracted graph entity needs a stable uni...
7,Unique Export binding IDs,True,0,0,Each extracted graph entity needs a stable uni...
8,Import bindings have parent imports,True,0,0,Every ImportBinding should belong to an Import.
9,Export bindings have parent exports,True,0,0,Every ExportBinding should belong to an Export.


QA checks passed: 20 / 20


In [95]:
# Classification summaries
print("IMPORT SYSTEMS")
display(
    imports_df.groupby(
        ["import_system", "import_kind"], dropna=False
    ).size().reset_index(name="count")
)

print("IMPORT BINDING TYPES")
display(
    import_bindings_df.groupby(
        "binding_type", dropna=False
    ).size().reset_index(name="count")
)

print("EXPORT SYSTEMS AND TYPES")
display(
    exports_df.groupby(
        ["export_system", "export_kind"], dropna=False
    ).size().reset_index(name="count")
)

print("EXPORT BINDING TYPES")
display(
    export_bindings_df.groupby(
        "binding_type", dropna=False
    ).size().reset_index(name="count")
)

IMPORT SYSTEMS


,import_system,import_kind,count
0,commonjs,commonjs_require,210
1,es_module,es_module,107
2,es_module,side_effect,2


IMPORT BINDING TYPES


,binding_type,count
0,default,54
1,module,145
2,named,84
3,unbound,6


EXPORT SYSTEMS AND TYPES


,export_system,export_kind,count
0,commonjs,exports_property_assignment,62
1,commonjs,module_assignment,46
2,commonjs,module_property_assignment,22
3,es_module,es_module,95


EXPORT BINDING TYPES


,binding_type,count
0,declaration,118
1,export_clause,13
2,expression,55
3,identifier,83


What I expect to see, and why
| QA area |Expected observation	| Why it matters |
|---------|---------------------|----------------|
| ES imports |	A substantial number | Zotero uses ES modules. |
| CommonJS imports	| A substantial number	| Existing and legacy modules use require(). |
| Default imports	| Present	| Common ES import pattern. |
| Named imports	| Present	| Needed to identify individual module dependencies. |
| Namespace imports	| Present	| Important for module-level dependency analysis. |
| Side-effect imports	| A small number	| These may initialise or register behaviour without importing a named symbol. |
| CommonJS module bindings	| Present	| Captures variables assigned to require(). |
| CommonJS member bindings	| A smaller number	| Captures imports such as require("x").foo. |
| Unbound CommonJS imports	| Possibly present	| Captures require() calls without a clear variable declaration. |
| ES declaration exports	| Present	| Captures exported functions, classes and variables. |
| ES export clauses	| Present	| Captures named and aliased exports. |
| CommonJS whole-module exports	| Present	| Captures module.exports = .... |
| CommonJS property exports	| Present	| Captures module.exports.foo and exports.foo. |
| Binding IDs	| All unique	| Prevents graph entity collisions. |
| Binding parent references	| No orphans	| Ensures bindings can be connected to their statements. |
| Source locations	| Valid line ranges	| Allows us to trace graph evidence back to source code. |

Important observations from our earlier analysis
A few findings should remain explicit in our interpretation:
1. The earlier 12 ambiguous export-to-symbol matches are not extraction failures. We found cases where a local name matched multiple declarations. Preserve those candidates rather than selecting one arbitrarily.
2. Seven named export bindings previously had no matching local symbol. This can arise from unsupported syntax, incomplete extraction or unusual export patterns. We should retain the binding and investigate later if it becomes relevant.
3. module.exports = value is not necessarily equivalent to an ES default export. We previously used default as a convenient exported-name convention. The export_system and export_kind properties preserve the distinction.
4. CommonJS exports are not fully captured by these three assignment patterns. For example, Object.assign(exports, ...), mutations of an exported object and reassignment through aliases are outside this first pass.
5. Import and export extraction does not establish that a dependency resolves. A module specifier is evidence of an import, not proof that we have found its destination file or the symbol it references.
6. Do not interpret every missing binding as an error. Some imports are side-effect-only, some exports are expressions, and some relationships will remain unresolved.
The earlier investigation also found 130 CommonJS export assignments: 46 whole-module assignments, 22 module.exports.property assignments and 62 exports.property assignments. Those are useful comparison points for the classification summary.

** What I would record for later **
I would retain these as outstanding investigation items, not defects:
- Namespace import coverage.
- CommonJS destructuring aliases.
- CommonJS member import classification.
- Re-export-from-module patterns.
- Multiple declarations or destructured exports.
- Exports created through object mutation or Object.assign().
The important thing is that we have captured the majority of the module-boundary evidence without attempting to resolve every semantic detail.

###### 8. Identifier references and assignments

For Section 8, I would keep your two extractors, but make a few small changes to improve the evidence we capture. The goal is still good-enough extraction, not full JavaScript name resolution.

Our earlier investigation established that:
- References should retain identifiers even when we cannot resolve them to declarations.
- Assignment targets can be identifiers or member expressions, not just variable names.
- Variable declarations are useful assignment-like evidence, but should remain distinguishable from subsequent assignments.
- Destructuring needs to be retained as a source expression, even if we don't yet resolve every individual binding.
- A call such as handleFlagWithParam(...) contains a reference to the callee; a member call such as obj.method() needs to preserve the member expression as well as its constituent identifiers.
- scope_id and resolved_symbol_id can remain None at this stage. Resolution is a later graph-analysis activity.
There are two issues in the current code worth addressing:
1. call_callee is only assigned when the identifier's immediate parent is call_expression. In a member call such as obj.method(), the immediate parent is a member_expression.
2. The current assignment classification only identifies a direct identifier as an assignment target. For obj.value = x, the target is a member_expression, so the identifier-level references don't fully represent the assignment target.
I would preserve the original expressions in assignments_df and improve the reference roles modestly.

In [96]:
def is_within_node(node, parent_node):
    """Return True when node is contained within parent_node."""
    return (
        node.start_byte >= parent_node.start_byte
        and node.end_byte <= parent_node.end_byte
    )


def get_assignment_target(node):
    """Find an assignment ancestor and determine whether node is in its LHS."""

    current = node.parent

    while current:
        if current.type in {
            "assignment_expression",
            "augmented_assignment_expression"
        }:
            children = current.named_children

            if children:
                target = children[0]

                if is_within_node(node, target):
                    return current, target

        current = current.parent

    return None, None


def get_call_callee(node):
    """Determine whether an identifier is part of a call's callee."""

    current = node

    while current.parent:

        parent = current.parent

        if parent.type == "call_expression":
            children = parent.named_children

            if children and children[0] == current:
                return parent

        if parent.type not in {
            "member_expression",
            "subscript_expression",
            "call_expression"
        }:
            break

        current = parent

    return None


def extract_references(file_id: str, tree, source_code: bytes) -> pd.DataFrame:

    records = []

    for node in walk_tree(tree.root_node):

        if node.type not in {
            "identifier",
            "shorthand_property_identifier"
        }:
            continue

        parent = node.parent

        # Exclude obvious declaration positions.
        if parent and parent.type in {
            "variable_declarator",
            "function_declaration",
            "generator_function_declaration",
            "class_declaration",
            "method_definition",
            "import_specifier",
            "formal_parameters"
        }:
            continue

        role = "reference"

        # Assignment target detection, including member expressions.
        assignment_node, assignment_target = get_assignment_target(node)

        if assignment_node is not None:
            role = "assignment_target"

        # Direct or member-expression call callee.
        elif get_call_callee(node) is not None:

            if parent and parent.type == "member_expression":
                children = parent.named_children

                if children and children[-1] == node:
                    role = "call_callee_member"
                else:
                    role = "call_callee"
            else:
                role = "call_callee"

        # Member property that is not a call callee.
        elif (
            parent
            and parent.type == "member_expression"
            and parent.named_children
            and parent.named_children[-1] == node
        ):
            role = "member_property"

        # Keep shorthand object properties distinguishable.
        elif node.type == "shorthand_property_identifier":
            role = "shorthand_property"

        records.append({
            "reference_id": stable_id(
                file_id, node.start_byte, prefix="ref_"
            ),
            "file_id": file_id,
            "name": node_text(node, source_code),
            "reference_role": role,
            "scope_id": None,
            "resolved_symbol_id": None,
            **node_location(node),
        })

    return pd.DataFrame(records)


def extract_assignments(file_id: str, tree, source_code: bytes) -> pd.DataFrame:

    records = []

    assignment_types = {
        "assignment_expression",
        "augmented_assignment_expression",
        "variable_declarator"
    }

    for node in walk_tree(tree.root_node):

        if node.type not in assignment_types:
            continue

        children = node.named_children

        target = children[0] if children else None
        value = children[-1] if len(children) > 1 else None

        # Preserve the complete expressions rather than attempting
        # to resolve identifiers or destructuring patterns.
        target_expression = (
            node_text(target, source_code) if target else None
        )

        value_expression = (
            node_text(value, source_code)
            if value and value != target
            else None
        )

        records.append({
            "assignment_id": stable_id(
                file_id, node.start_byte, prefix="asg_"
            ),
            "file_id": file_id,
            "assignment_kind": node.type,
            "target_expression": target_expression,
            "value_expression": value_expression,
            **node_location(node),
        })

    return pd.DataFrame(records)


# ---------------------------------------------------------
# Run extraction across all successfully parsed files
# ---------------------------------------------------------

reference_frames = []
assignment_frames = []

for fid, tree in ast_trees.items():

    source_bytes = source_by_file[fid].encode("utf-8")

    reference_df = extract_references(
        fid, tree, source_bytes
    )

    assignment_df = extract_assignments(
        fid, tree, source_bytes
    )

    if not reference_df.empty:
        reference_frames.append(reference_df)

    if not assignment_df.empty:
        assignment_frames.append(assignment_df)


references_df = (
    pd.concat(reference_frames, ignore_index=True)
    if reference_frames else pd.DataFrame()
)

assignments_df = (
    pd.concat(assignment_frames, ignore_index=True)
    if assignment_frames else pd.DataFrame()
)


print(f"References extracted:  {len(references_df):,}")
print(f"Assignments extracted: {len(assignments_df):,}")

print(
    f"Files with references: "
    f"{references_df['file_id'].nunique():,}"
    if not references_df.empty else "Files with references: 0"
)

print(
    f"Files with assignments: "
    f"{assignments_df['file_id'].nunique():,}"
    if not assignments_df.empty else "Files with assignments: 0"
)

display(references_df.head(10))
display(assignments_df.head(10))

References extracted:  254,402
Assignments extracted: 72,972
Files with references: 540
Files with assignments: 496


,reference_id,file_id,name,reference_role,scope_id,resolved_symbol_id,start_line,start_column,end_line,end_column,start_byte,end_byte
0,ref_6a0ce5fabc0d788e,app/assets/commandLineHandler.js,ChromeUtils,call_callee,None,None,1,43,1,54,42,53
1,ref_f274b1f8b0893350,app/assets/commandLineHandler.js,cmdLine,reference,None,None,5,5,5,12,262,269
2,ref_b1d050d716f004e8,app/assets/commandLineHandler.js,Ci,reference,None,None,5,22,5,24,279,281
3,ref_9d9636f34aed2aa1,app/assets/commandLineHandler.js,cmdLine,assignment_target,None,None,6,2,6,9,322,329
4,ref_80f5d9a64f1c4b0b,app/assets/commandLineHandler.js,cmdLine,call_callee,None,None,10,5,10,12,392,399
5,ref_878bff57d642f3f6,app/assets/commandLineHandler.js,CommandLineOptions,assignment_target,None,None,11,2,11,20,437,455
6,ref_96a87ec634a4b578,app/assets/commandLineHandler.js,cmdLine,call_callee,None,None,14,10,14,17,524,531
7,ref_348ffe0ecd591af6,app/assets/commandLineHandler.js,CommandLineOptions,assignment_target,None,None,15,2,15,20,573,591
8,ref_a41c796f6fe1bbbb,app/assets/commandLineHandler.js,CommandLineOptions,reference,None,None,20,5,20,23,898,916
9,ref_22adff4180b67460,app/assets/commandLineHandler.js,Services,call_callee,None,None,21,2,21,10,935,943


,assignment_id,file_id,assignment_kind,target_expression,value_expression,start_line,start_column,end_line,end_column,start_byte,end_byte
0,asg_688a35739fce99cc,app/assets/commandLineHandler.js,variable_declarator,"{ CommandLineOptions, TestOptions }","ChromeUtils.importESModule(""chrome://zotero/co...",1,5,1,127,4,126
1,asg_9d9636f34aed2aa1,app/assets/commandLineHandler.js,assignment_expression,cmdLine.preventDefault,true,6,2,6,31,322,351
2,asg_878bff57d642f3f6,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDebugLog,2,11,2,11,38,437,473
3,asg_348ffe0ecd591af6,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDebugLog,1,15,2,15,38,573,609
4,asg_559ddbdd7eb6dbd6,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDataDir,"cmdLine.handleFlagWithParam(""datadir"", false)",24,1,24,80,1016,1095
5,asg_71ad3a1356db8a14,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.file,"cmdLine.handleFlagWithParam(""file"", false)",26,1,26,69,1158,1226
6,asg_46de9770a57e8f6a,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.url,"cmdLine.handleFlagWithParam(""url"", false)",27,1,27,67,1228,1294
7,asg_18b1bc890b11b3ec,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.url,cmdLine.resolveURI(CommandLineOptions.url),29,2,29,69,1327,1394
8,asg_b47b92a4792dec83,app/assets/commandLineHandler.js,variable_declarator,processTestOptions,false,32,5,32,31,1403,1429
9,asg_e4e507887ce443fe,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.test,true,34,2,34,32,1479,1509


In [97]:
# QA Checks
qa_results = []


def qa_check(name, condition, observed, expected, reason):
    qa_results.append({
        "test": name,
        "passed": bool(condition),
        "observed": observed,
        "expected": expected,
        "reason": reason,
    })


# ---------------------------------------------------------
# 1. Basic extraction checks
# ---------------------------------------------------------

qa_check(
    "References extracted",
    len(references_df) > 0,
    len(references_df),
    "> 0",
    "Identifier references are evidence of code dependencies and usage."
)

qa_check(
    "Assignments extracted",
    len(assignments_df) > 0,
    len(assignments_df),
    "> 0",
    "Assignments provide evidence of state changes and data flow."
)


# ---------------------------------------------------------
# 2. Unique identifiers
# ---------------------------------------------------------

for df, column, label in [
    (references_df, "reference_id", "Reference IDs"),
    (assignments_df, "assignment_id", "Assignment IDs"),
]:

    duplicates = (
        int(df[column].duplicated().sum())
        if not df.empty else 0
    )

    qa_check(
        f"Unique {label}",
        duplicates == 0,
        duplicates,
        0,
        "Each graph entity needs a unique identifier."
    )


# ---------------------------------------------------------
# 3. File references
# ---------------------------------------------------------

for df, label in [
    (references_df, "References"),
    (assignments_df, "Assignments"),
]:

    unknown_files = (
        int((~df["file_id"].isin(ast_trees.keys())).sum())
        if not df.empty else 0
    )

    qa_check(
        f"{label} reference known files",
        unknown_files == 0,
        unknown_files,
        0,
        "Each extracted entity must retain its originating file."
    )


# ---------------------------------------------------------
# 4. Source location checks
# ---------------------------------------------------------

for df, label in [
    (references_df, "References"),
    (assignments_df, "Assignments"),
]:

    if df.empty:
        invalid_locations = 0
    else:
        invalid_locations = int((
            df["start_line"].isna()
            | df["end_line"].isna()
            | (df["start_line"] < 1)
            | (df["end_line"] < df["start_line"])
        ).sum())

    qa_check(
        f"{label} have valid source locations",
        invalid_locations == 0,
        invalid_locations,
        0,
        "Source locations allow us to trace evidence back to the code."
    )


# ---------------------------------------------------------
# 5. Reference classification
# ---------------------------------------------------------

if not references_df.empty:

    missing_roles = int(
        references_df["reference_role"].isna().sum()
    )

    qa_check(
        "Reference roles populated",
        missing_roles == 0,
        missing_roles,
        0,
        "Reference roles help distinguish usage patterns."
    )

    qa_check(
        "Reference names populated",
        references_df["name"].notna().all(),
        int(references_df["name"].isna().sum()),
        0,
        "Every reference should retain its identifier text."
    )


# ---------------------------------------------------------
# 6. Assignment classification
# ---------------------------------------------------------

if not assignments_df.empty:

    expected_assignment_types = {
        "assignment_expression",
        "augmented_assignment_expression",
        "variable_declarator"
    }

    observed_assignment_types = set(
        assignments_df["assignment_kind"].dropna()
    )

    unexpected_types = (
        observed_assignment_types - expected_assignment_types
    )

    qa_check(
        "Assignment types are recognised",
        len(unexpected_types) == 0,
        sorted(unexpected_types),
        [],
        "Only the intended assignment node types should be extracted."
    )

    missing_targets = int(
        assignments_df["target_expression"].isna().sum()
    )

    qa_check(
        "Assignment targets populated",
        missing_targets == 0,
        missing_targets,
        0,
        "The target expression identifies what is being assigned."
    )


# ---------------------------------------------------------
# 7. Resolution remains deliberately deferred
# ---------------------------------------------------------

qa_check(
    "Reference scope resolution deferred",
    references_df.empty or references_df["scope_id"].isna().all(),
    (
        int(references_df["scope_id"].notna().sum())
        if not references_df.empty else 0
    ),
    0,
    "Scope linkage will be added in a later analysis step."
)

qa_check(
    "Symbol resolution deferred",
    references_df.empty or references_df["resolved_symbol_id"].isna().all(),
    (
        int(references_df["resolved_symbol_id"].notna().sum())
        if not references_df.empty else 0
    ),
    0,
    "Candidate symbol resolution is intentionally deferred."
)


# ---------------------------------------------------------
# 8. Summary
# ---------------------------------------------------------

qa_df = pd.DataFrame(qa_results)

display(qa_df)

print(
    f"QA checks passed: "
    f"{qa_df['passed'].sum()} / {len(qa_df)}"
)

,test,passed,observed,expected,reason
0,References extracted,True,254402,> 0,Identifier references are evidence of code dep...
1,Assignments extracted,True,72972,> 0,Assignments provide evidence of state changes ...
2,Unique Reference IDs,True,0,0,Each graph entity needs a unique identifier.
3,Unique Assignment IDs,True,0,0,Each graph entity needs a unique identifier.
4,References reference known files,True,0,0,Each extracted entity must retain its originat...
5,Assignments reference known files,True,0,0,Each extracted entity must retain its originat...
6,References have valid source locations,True,0,0,Source locations allow us to trace evidence ba...
7,Assignments have valid source locations,True,0,0,Source locations allow us to trace evidence ba...
8,Reference roles populated,True,0,0,Reference roles help distinguish usage patterns.
9,Reference names populated,True,0,0,Every reference should retain its identifier t...


QA checks passed: 14 / 14


In [98]:
# Classification summaries
print("REFERENCE ROLES")

display(
    references_df.groupby(
        "reference_role", dropna=False
    ).size().reset_index(name="count")
)


print("ASSIGNMENT TYPES")

display(
    assignments_df.groupby(
        "assignment_kind", dropna=False
    ).size().reset_index(name="count")
)


print("ASSIGNMENT TARGET EXAMPLES")

display(
    assignments_df[
        [
            "file_id",
            "assignment_kind",
            "target_expression",
            "value_expression",
            "start_line",
        ]
    ].head(30)
)

REFERENCE ROLES


,reference_role,count
0,assignment_target,28043
1,call_callee,88030
2,reference,136642
3,shorthand_property,1687


ASSIGNMENT TYPES


,assignment_kind,count
0,assignment_expression,30883
1,augmented_assignment_expression,1550
2,variable_declarator,40539


ASSIGNMENT TARGET EXAMPLES


,file_id,assignment_kind,target_expression,value_expression,start_line
0,app/assets/commandLineHandler.js,variable_declarator,"{ CommandLineOptions, TestOptions }","ChromeUtils.importESModule(""chrome://zotero/co...",1
1,app/assets/commandLineHandler.js,assignment_expression,cmdLine.preventDefault,true,6
2,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDebugLog,2,11
3,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDebugLog,1,15
4,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDataDir,"cmdLine.handleFlagWithParam(""datadir"", false)",24
5,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.file,"cmdLine.handleFlagWithParam(""file"", false)",26
6,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.url,"cmdLine.handleFlagWithParam(""url"", false)",27
7,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.url,cmdLine.resolveURI(CommandLineOptions.url),29
8,app/assets/commandLineHandler.js,variable_declarator,processTestOptions,false,32
9,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.test,true,34


What I expect and why
| QA area | Expected observation | Why |
|---|---|---|
| Reference count | Thousands of references | JavaScript code contains many identifier usages. |
| Assignment count | Thousands of assignments/declarations | Captures state changes and initialisations. |
| reference | Largest or one of the largest categories | General identifier usage. |
| call_callee | Present | Direct function calls. |
| call_callee_member | Present | Method calls such as obj.method(). |
| member_property | Present | Property access that isn't necessarily a call. |
| assignment_target | Present | Identifiers involved in assignments. |
| shorthand_property | Present if used | Object shorthand such as { value }. |
| variable_declarator | Substantial | Captures initial variable values. |
| assignment_expression | Substantial | Captures subsequent assignments. |
| augmented_assignment_expression | Possibly smaller | Captures operations such as += and ||=. |
Important limitations to retain
- We are not resolving references_df to symbols_df yet.
- We are not assuming that every assignment is a simple variable assignment.
- We are not attempting to infer runtime data flow.
- A destructuring assignment is retained as an expression rather than decomposed into fully resolved variables.
- A member property name is not automatically a reference to a declared symbol.
One caveat with the revised reference extractor: the ancestor-based assignment detection may classify several identifiers within a member-expression target as assignment_target. That's intentional for this first pass; it preserves the target expression's constituent evidence rather than pretending we have fully resolved its semantics.

###### 9. Function Call Extraction

In [99]:
# First pass structural extraction
def extract_calls(file_id: str, tree, source_code: bytes) -> pd.DataFrame:
    records = []

    for node in walk_tree(tree.root_node):
        if node.type not in {"call_expression", "new_expression"}:
            continue

        children = node.named_children

        if not children:
            continue

        # The first named child represents the function or constructor expression.
        callee_node = children[0]

        # Locate the arguments node without assuming positional child indexes.
        arguments_node = next(
            (child for child in children if child.type == "arguments"),
            None
        )

        argument_nodes = (
            arguments_node.named_children
            if arguments_node is not None
            else []
        )

        records.append({
            "call_id": stable_id(
                file_id,
                node.start_byte,
                node.end_byte,
                prefix="call_"
            ),
            "file_id": file_id,
            "call_kind": (
                "constructor_call"
                if node.type == "new_expression"
                else "function_call"
            ),
            "call_node_type": node.type,
            "callee_expression": node_text(callee_node, source_code),
            "call_text": node_text(node, source_code),
            "arguments_text": (
                node_text(arguments_node, source_code)
                if arguments_node is not None
                else None
            ),
            "argument_count": len(argument_nodes),
            **node_location(node)
        })

    return pd.DataFrame(records)

In [100]:
# Extraction across all files
calls_frames = []

for file_path, tree in ast_trees.items():
    source_bytes = source_by_file[file_path].encode("utf-8")

    file_calls_df = extract_calls(
        file_id=file_path,
        tree=tree,
        source_code=source_bytes
    )

    if not file_calls_df.empty:
        calls_frames.append(file_calls_df)

calls_df = (
    pd.concat(calls_frames, ignore_index=True)
    if calls_frames
    else pd.DataFrame()
)

print(f"Function and constructor calls extracted: {len(calls_df):,}")
print(f"Files with calls: {calls_df['file_id'].nunique():,}")

Function and constructor calls extracted: 108,839
Files with calls: 539


In [101]:
# QA Checks
qa_checks = {
    "Call IDs are unique":
        calls_df["call_id"].is_unique,

    "Call IDs are populated":
        calls_df["call_id"].notna().all(),

    "File IDs are populated":
        calls_df["file_id"].notna().all(),

    "Call expressions are populated":
        calls_df["call_text"].notna().all()
        and calls_df["call_text"].ne("").all(),

    "Callee expressions are populated":
        calls_df["callee_expression"].notna().all()
        and calls_df["callee_expression"].ne("").all(),

    "Call node types are valid":
        calls_df["call_node_type"].isin(
            ["call_expression", "new_expression"]
        ).all(),

    "Call kinds are valid":
        calls_df["call_kind"].isin(
            ["function_call", "constructor_call"]
        ).all(),

    "Call kind matches AST node type":
        (
            (
                calls_df["call_node_type"] == "call_expression"
            )
            == (
                calls_df["call_kind"] == "function_call"
            )
        ).all(),

    "Start lines are positive":
        calls_df["start_line"].gt(0).all(),

    "End lines are not before start lines":
        calls_df["end_line"].ge(calls_df["start_line"]).all(),

    "Argument counts are non-negative":
        calls_df["argument_count"].ge(0).all(),

    "Argument counts are integers":
        pd.api.types.is_integer_dtype(calls_df["argument_count"]),
}

for check, passed in qa_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print(f"\nQA checks passed: {sum(qa_checks.values())} / {len(qa_checks)}")

PASS: Call IDs are unique
PASS: Call IDs are populated
PASS: File IDs are populated
PASS: Call expressions are populated
PASS: Callee expressions are populated
PASS: Call node types are valid
PASS: Call kinds are valid
PASS: Call kind matches AST node type
PASS: Start lines are positive
PASS: End lines are not before start lines
PASS: Argument counts are non-negative
PASS: Argument counts are integers

QA checks passed: 12 / 12


In [102]:
# Extraction Summary
print("Call kind distribution:")
display(
    calls_df["call_kind"]
    .value_counts()
    .rename_axis("call_kind")
    .reset_index(name="count")
)

print("\nAST node type distribution:")
display(
    calls_df["call_node_type"]
    .value_counts()
    .rename_axis("call_node_type")
    .reset_index(name="count")
)

print("\nSample call sites:")
display(
    calls_df[
        [
            "file_id",
            "call_kind",
            "callee_expression",
            "call_text",
            "argument_count",
            "start_line"
        ]
    ].head(15)
)

Call kind distribution:


,call_kind,count
0,function_call,104307
1,constructor_call,4532



AST node type distribution:


,call_node_type,count
0,call_expression,104307
1,new_expression,4532



Sample call sites:


,file_id,call_kind,callee_expression,call_text,argument_count,start_line
0,app/assets/commandLineHandler.js,function_call,ChromeUtils.importESModule,"ChromeUtils.importESModule(""chrome://zotero/co...",1,1
1,app/assets/commandLineHandler.js,function_call,cmdLine.handleFlag,"cmdLine.handleFlag(""ZoteroDebug"", false)",2,10
2,app/assets/commandLineHandler.js,function_call,cmdLine.handleFlag,"cmdLine.handleFlag(""ZoteroDebugText"", false)",2,14
3,app/assets/commandLineHandler.js,function_call,"Services.prefs.getBranch(""toolkit.startup."").c...","Services.prefs.getBranch(""toolkit.startup."").c...",1,21
4,app/assets/commandLineHandler.js,function_call,Services.prefs.getBranch,"Services.prefs.getBranch(""toolkit.startup."")",1,21
5,app/assets/commandLineHandler.js,function_call,cmdLine.handleFlagWithParam,"cmdLine.handleFlagWithParam(""datadir"", false)",2,24
6,app/assets/commandLineHandler.js,function_call,cmdLine.handleFlagWithParam,"cmdLine.handleFlagWithParam(""file"", false)",2,26
7,app/assets/commandLineHandler.js,function_call,cmdLine.handleFlagWithParam,"cmdLine.handleFlagWithParam(""url"", false)",2,27
8,app/assets/commandLineHandler.js,function_call,cmdLine.resolveURI,cmdLine.resolveURI(CommandLineOptions.url),1,29
9,app/assets/commandLineHandler.js,function_call,cmdLine.handleFlag,"cmdLine.handleFlag(""ZoteroTest"", false)",2,33


###### 10. Class and inheritance extraction

In [103]:
# Extraction loop
def extract_classes_and_inheritance(
    file_id: str,
    tree,
    source_code: bytes
):
    class_records = []
    inheritance_records = []

    for node in walk_tree(tree.root_node):
        if node.type not in {"class_declaration", "class"}:
            continue

        class_name_node = declaration_name_node(node)
        class_name = (
            node_text(class_name_node, source_code)
            if class_name_node
            else None
        )

        heritage = next(
            (
                child for child in node.named_children
                if child.type == "class_heritage"
            ),
            None
        )

        base_expression = None
        heritage_text = None

        if heritage:
            heritage_text = node_text(heritage, source_code)

            named_children = heritage.named_children
            if named_children:
                base_expression = node_text(
                    named_children[-1],
                    source_code
                )

        location = node_location(node)

        class_id = stable_id(
            file_id,
            node.start_byte,
            node.end_byte,
            prefix="class_"
        )

        class_records.append({
            "class_id": class_id,
            "file_id": file_id,
            "class_name": class_name,
            "class_node_type": node.type,
            "heritage_text": heritage_text,
            "base_expression": base_expression,
            "source_text": node_text(node, source_code),
            **location
        })

        if base_expression:
            inheritance_records.append({
                "inheritance_id": stable_id(
                    file_id,
                    node.start_byte,
                    node.end_byte,
                    prefix="inh_"
                ),
                "class_id": class_id,
                "file_id": file_id,
                "class_name": class_name,
                "base_expression": base_expression,
                "heritage_text": heritage_text,
                "resolved_base_symbol_id": None,
                **location
            })

    return (
        pd.DataFrame(class_records),
        pd.DataFrame(inheritance_records)
    )


class_frames = []
inheritance_frames = []

for file_path, tree in ast_trees.items():
    source_bytes = source_by_file[file_path].encode("utf-8")

    file_classes_df, file_inheritance_df = (
        extract_classes_and_inheritance(
            file_id=file_path,
            tree=tree,
            source_code=source_bytes
        )
    )

    if not file_classes_df.empty:
        class_frames.append(file_classes_df)

    if not file_inheritance_df.empty:
        inheritance_frames.append(file_inheritance_df)


classes_df = (
    pd.concat(class_frames, ignore_index=True)
    if class_frames else pd.DataFrame()
)

inheritance_df = (
    pd.concat(inheritance_frames, ignore_index=True)
    if inheritance_frames else pd.DataFrame()
)

print(f"Class declarations extracted: {len(classes_df):,}")
print(f"Classes with inheritance expressions: {len(inheritance_df):,}")
print(f"Files containing classes: {classes_df['file_id'].nunique():,}")
print(f"Files containing inheritance: {inheritance_df['file_id'].nunique():,}")

Class declarations extracted: 384
Classes with inheritance expressions: 151
Files containing classes: 126
Files containing inheritance: 105


In [104]:
# QA Checks
qa_checks = {
    "Class IDs are unique":
        classes_df["class_id"].is_unique,

    "Class IDs are populated":
        classes_df["class_id"].notna().all(),

    "Class file IDs are populated":
        classes_df["file_id"].notna().all(),

    "Class node types are valid":
        classes_df["class_node_type"].isin(
            ["class_declaration", "class"]
        ).all(),

    "Class source text is populated":
        classes_df["source_text"].notna().all()
        and classes_df["source_text"].ne("").all(),

    "Class start lines are positive":
        classes_df["start_line"].gt(0).all(),

    "Class end lines are not before start lines":
        classes_df["end_line"].ge(classes_df["start_line"]).all(),

    "Inheritance IDs are unique":
        inheritance_df["inheritance_id"].is_unique,

    "Inheritance IDs are populated":
        inheritance_df["inheritance_id"].notna().all(),

    "Inheritance class IDs are populated":
        inheritance_df["class_id"].notna().all(),

    "Inheritance file IDs are populated":
        inheritance_df["file_id"].notna().all(),

    "Base expressions are populated":
        inheritance_df["base_expression"].notna().all()
        and inheritance_df["base_expression"].ne("").all(),

    "Every inheritance class exists in classes_df":
        inheritance_df["class_id"].isin(
            classes_df["class_id"]
        ).all(),

    "Every inheritance record has heritage text":
        inheritance_df["heritage_text"].notna().all()
        and inheritance_df["heritage_text"].ne("").all(),

    "Unresolved base symbols remain unassigned":
        inheritance_df["resolved_base_symbol_id"].isna().all(),
}

for check, passed in qa_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print(
    f"\nQA checks passed: "
    f"{sum(qa_checks.values())} / {len(qa_checks)}"
)

PASS: Class IDs are unique
PASS: Class IDs are populated
PASS: Class file IDs are populated
PASS: Class node types are valid
PASS: Class source text is populated
PASS: Class start lines are positive
PASS: Class end lines are not before start lines
PASS: Inheritance IDs are unique
PASS: Inheritance IDs are populated
PASS: Inheritance class IDs are populated
PASS: Inheritance file IDs are populated
PASS: Base expressions are populated
PASS: Every inheritance class exists in classes_df
PASS: Every inheritance record has heritage text
PASS: Unresolved base symbols remain unassigned

QA checks passed: 15 / 15


In [105]:
# Extraction summaries
print("Class node types:")
display(
    classes_df["class_node_type"]
    .value_counts()
    .rename_axis("class_node_type")
    .reset_index(name="count")
)

print("\nNamed and anonymous classes:")
display(
    classes_df.assign(
        class_name_type=classes_df["class_name"].apply(
            lambda x: "named" if pd.notna(x) else "anonymous"
        )
    )["class_name_type"]
    .value_counts()
    .rename_axis("class_name_type")
    .reset_index(name="count")
)

print("\nInheritance base expressions:")
display(
    inheritance_df["base_expression"]
    .value_counts()
    .head(20)
    .rename_axis("base_expression")
    .reset_index(name="count")
)

print("\nSample inheritance records:")
display(
    inheritance_df[
        [
            "file_id",
            "class_name",
            "base_expression",
            "start_line",
            "end_line"
        ]
    ].head(20)
)

Class node types:


,class_node_type,count
0,class,237
1,class_declaration,147



Named and anonymous classes:


,class_name_type,count
0,anonymous,231
1,named,153



Inheritance base expressions:


,base_expression,count
0,XULElementBase,31
1,LocalAPIEndpoint,23
2,ItemPaneSectionElementBase,14
3,JSWindowActorChild,10
4,React.PureComponent,9
5,JSWindowActorParent,7
6,XULElement,6
7,Error,5
8,ItemTreeRow,5
9,PluginAPIBase,4



Sample inheritance records:


,file_id,class_name,base_expression,start_line,end_line
0,chrome/content/scaffold/scaffoldItemPreview.js,ScaffoldItemPreview,XULElementBase,31,216
1,chrome/content/scaffold/scaffoldItemPreviews.js,ScaffoldItemPreviews,XULElementBase,29,143
2,chrome/content/scaffold/zoteroWebTranslationEn...,ZoteroWebTranslationEnvironment,AbstractWebTranslationEnvironment,3,77
3,chrome/content/zotero/actors/DocumentIsReadyCh...,DocumentIsReadyChild,JSWindowActorChild,3,12
4,chrome/content/zotero/actors/ExternalLinkHandl...,ExternalLinkHandlerChild,JSWindowActorChild,1,67
5,chrome/content/zotero/actors/ExternalLinkHandl...,ExternalLinkHandlerParent,JSWindowActorParent,5,14
6,chrome/content/zotero/actors/FeedAbstractChild...,FeedAbstractChild,JSWindowActorChild,1,62
7,chrome/content/zotero/actors/FeedAbstractParen...,FeedAbstractParent,JSWindowActorParent,5,24
8,chrome/content/zotero/actors/MendeleyAuthChild...,MendeleyAuthChild,JSWindowActorChild,5,39
9,chrome/content/zotero/actors/MendeleyAuthParen...,MendeleyAuthParent,JSWindowActorParent,7,20


###### 11. Initial symbol and call extraction summary

In [107]:
import re
import pandas as pd


def build_local_symbol_lookup(symbols: pd.DataFrame) -> dict:
    """Index symbols by (file_id, name), preserving all candidates."""
    lookup = {}

    if symbols.empty:
        return lookup

    valid_symbols = symbols.dropna(subset=["file_id", "name", "symbol_id"])

    for (file_id, name), group in valid_symbols.groupby(
        ["file_id", "name"], sort=False
    ):
        lookup[(file_id, name)] = group["symbol_id"].tolist()

    return lookup


def resolve_local_references(
    references: pd.DataFrame,
    symbols: pd.DataFrame
) -> pd.DataFrame:
    """
    Conservative same-file reference resolution.

    Only unambiguous same-file name matches are resolved.
    All candidate symbol IDs are retained for later analysis.
    """
    result = references.copy()

    result["candidate_symbol_ids"] = pd.Series(
        [[] for _ in range(len(result))], index=result.index, dtype=object
    )
    result["candidate_count"] = 0
    result["resolved_symbol_id"] = None
    result["resolution_status"] = "UNRESOLVED"
    result["resolution_method"] = None

    if result.empty or symbols.empty:
        return result

    lookup = build_local_symbol_lookup(symbols)

    for idx, ref in result.iterrows():
        if pd.isna(ref.get("name")) or pd.isna(ref.get("file_id")):
            continue

        candidates = lookup.get(
            (ref["file_id"], ref["name"]), []
        )

        result.at[idx, "candidate_symbol_ids"] = candidates
        result.at[idx, "candidate_count"] = len(candidates)

        if len(candidates) == 1:
            result.at[idx, "resolved_symbol_id"] = candidates[0]
            result.at[idx, "resolution_status"] = "RESOLVED"
            result.at[idx, "resolution_method"] = "same_file_name"

        elif len(candidates) > 1:
            result.at[idx, "resolution_status"] = "AMBIGUOUS"
            result.at[idx, "resolution_method"] = "same_file_name"

    return result


def resolve_local_calls(
    calls: pd.DataFrame,
    symbols: pd.DataFrame
) -> pd.DataFrame:
    """
    Conservative same-file resolution of simple identifier callees.

    Member calls and complex expressions remain unresolved.
    """
    result = calls.copy()

    result["candidate_symbol_ids"] = pd.Series(
        [[] for _ in range(len(result))], index=result.index, dtype=object
    )
    result["candidate_count"] = 0
    result["resolved_callee_symbol_id"] = None
    result["resolution_status"] = "UNRESOLVED"
    result["resolution_method"] = None

    if result.empty or symbols.empty:
        return result

    lookup = build_local_symbol_lookup(symbols)

    simple_identifier = re.compile(r"[A-Za-z_$][\w$]*")

    for idx, call in result.iterrows():
        callee = call.get("callee_expression")
        file_id = call.get("file_id")

        if not isinstance(callee, str):
            continue

        if not simple_identifier.fullmatch(callee):
            continue

        candidates = lookup.get((file_id, callee), [])

        result.at[idx, "candidate_symbol_ids"] = candidates
        result.at[idx, "candidate_count"] = len(candidates)

        if len(candidates) == 1:
            result.at[idx, "resolved_callee_symbol_id"] = candidates[0]
            result.at[idx, "resolution_status"] = "RESOLVED"
            result.at[idx, "resolution_method"] = "same_file_name"

        elif len(candidates) > 1:
            result.at[idx, "resolution_status"] = "AMBIGUOUS"
            result.at[idx, "resolution_method"] = "same_file_name"

    return result


resolved_references_df = resolve_local_references(
    references_df,
    symbols_df
)

resolved_calls_df = resolve_local_calls(
    calls_df,
    symbols_df
)

print(f"References processed: {len(resolved_references_df):,}")
print(f"Calls processed: {len(resolved_calls_df):,}")


References processed: 254,402
Calls processed: 108,839


In [108]:
# QA Checks
qa_checks = {
    "Reference row count preserved":
        len(resolved_references_df) == len(references_df),

    "Call row count preserved":
        len(resolved_calls_df) == len(calls_df),

    "Reference IDs remain unique":
        resolved_references_df["reference_id"].is_unique,

    "Call IDs remain unique":
        resolved_calls_df["call_id"].is_unique,

    "Reference candidate counts are non-negative":
        resolved_references_df["candidate_count"].ge(0).all(),

    "Call candidate counts are non-negative":
        resolved_calls_df["candidate_count"].ge(0).all(),

    "Resolved references have exactly one candidate":
        resolved_references_df.loc[
            resolved_references_df["resolution_status"] == "RESOLVED",
            "candidate_count"
        ].eq(1).all(),

    "Resolved calls have exactly one candidate":
        resolved_calls_df.loc[
            resolved_calls_df["resolution_status"] == "RESOLVED",
            "candidate_count"
        ].eq(1).all(),

    "Ambiguous references retain multiple candidates":
        resolved_references_df.loc[
            resolved_references_df["resolution_status"] == "AMBIGUOUS",
            "candidate_count"
        ].gt(1).all(),

    "Ambiguous calls retain multiple candidates":
        resolved_calls_df.loc[
            resolved_calls_df["resolution_status"] == "AMBIGUOUS",
            "candidate_count"
        ].gt(1).all(),

    "Unresolved references have no resolved symbol":
        resolved_references_df.loc[
            resolved_references_df["resolution_status"] != "RESOLVED",
            "resolved_symbol_id"
        ].isna().all(),

    "Unresolved calls have no resolved symbol":
        resolved_calls_df.loc[
            resolved_calls_df["resolution_status"] != "RESOLVED",
            "resolved_callee_symbol_id"
        ].isna().all(),
}

for check, passed in qa_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print(
    f"\nQA checks passed: "
    f"{sum(qa_checks.values())} / {len(qa_checks)}"
)


PASS: Reference row count preserved
PASS: Call row count preserved
PASS: Reference IDs remain unique
PASS: Call IDs remain unique
PASS: Reference candidate counts are non-negative
PASS: Call candidate counts are non-negative
PASS: Resolved references have exactly one candidate
PASS: Resolved calls have exactly one candidate
PASS: Ambiguous references retain multiple candidates
PASS: Ambiguous calls retain multiple candidates
PASS: Unresolved references have no resolved symbol
PASS: Unresolved calls have no resolved symbol

QA checks passed: 12 / 12


In [109]:
# Resolution summaries
def resolution_summary(df, name):
    print(f"\n{name}")
    print("-" * len(name))

    display(
        df["resolution_status"]
        .value_counts()
        .rename_axis("resolution_status")
        .reset_index(name="count")
    )

    print("Candidate counts:")
    display(
        df["candidate_count"]
        .value_counts()
        .sort_index()
        .rename_axis("candidate_count")
        .reset_index(name="count")
    )


resolution_summary(
    resolved_references_df,
    "Reference resolution"
)

resolution_summary(
    resolved_calls_df,
    "Call resolution"
)


print("\nResolved reference examples:")
display(
    resolved_references_df[
        resolved_references_df["resolution_status"] == "RESOLVED"
    ][
        [
            "file_id",
            "name",
            "reference_role",
            "resolved_symbol_id",
            "resolution_method"
        ]
    ].head(15)
)


print("\nResolved call examples:")
display(
    resolved_calls_df[
        resolved_calls_df["resolution_status"] == "RESOLVED"
    ][
        [
            "file_id",
            "callee_expression",
            "call_kind",
            "resolved_callee_symbol_id",
            "resolution_method"
        ]
    ].head(15)
)



Reference resolution
--------------------


,resolution_status,count
0,AMBIGUOUS,109082
1,UNRESOLVED,80274
2,RESOLVED,65046


Candidate counts:


,candidate_count,count
0,0,80274
1,1,65046
2,2,17109
3,3,10727
4,4,7816
...,...,...
75,301,1320
76,397,1856
77,503,2372
78,513,4211



Call resolution
---------------


,resolution_status,count
0,UNRESOLVED,99027
1,RESOLVED,7734
2,AMBIGUOUS,2078


... (2 outputs left out)


Resolved reference examples:


,file_id,name,reference_role,resolved_symbol_id,resolution_method
22,app/assets/commandLineHandler.js,processTestOptions,assignment_target,sym_3b0d08eff5808038,same_file_name
27,app/assets/commandLineHandler.js,processTestOptions,reference,sym_3b0d08eff5808038,same_file_name
61,app/assets/commandLineHandler.js,loader,call_callee,sym_281dac345cf207b2,same_file_name
62,app/assets/commandLineHandler.js,loader,call_callee,sym_281dac345cf207b2,same_file_name
68,app/assets/commandLineHandler.js,portOrPath,shorthand_property,sym_21cf878906be0255,same_file_name
71,app/assets/commandLineHandler.js,socketOptions,reference,sym_340803e89f92fd2d,same_file_name
72,app/assets/commandLineHandler.js,listener,call_callee,sym_c5727e816fee1abb,same_file_name
76,app/assets/commandLineHandler.js,portOrPath,reference,sym_21cf878906be0255,same_file_name
158,chrome/content/scaffold/content.js,onLoad,reference,sym_d7f11767ca70d14d,same_file_name
163,chrome/content/scaffold/load.js,listbox,call_callee,sym_1ac07a30fa06bd8e,same_file_name



Resolved call examples:


,file_id,callee_expression,call_kind,resolved_callee_symbol_id,resolution_method
176,chrome/content/scaffold/scaffold.js,_setBrowserLoadingIndicator,function_call,sym_46519c6b3f885a6f,same_file_name
239,chrome/content/scaffold/scaffold.js,_handleAPIKeyChange,function_call,sym_81f462a0d004c10f,same_file_name
256,chrome/content/scaffold/scaffold.js,_logOutput,function_call,sym_f156a36b54d9c420,same_file_name
258,chrome/content/scaffold/scaffold.js,_logOutput,function_call,sym_f156a36b54d9c420,same_file_name
266,chrome/content/scaffold/scaffold.js,_updateTitle,function_call,sym_4454f4dba0c40265,same_file_name
284,chrome/content/scaffold/scaffold.js,_getMetadataObject,function_call,sym_b416bb6e47ad20fa,same_file_name
288,chrome/content/scaffold/scaffold.js,_getImport,function_call,sym_72ca57930cb247e8,same_file_name
289,chrome/content/scaffold/scaffold.js,_getImport,function_call,sym_72ca57930cb247e8,same_file_name
322,chrome/content/scaffold/scaffold.js,_loadTestsFromPane,function_call,sym_38272a2c116a89f7,same_file_name
326,chrome/content/scaffold/scaffold.js,_loadTestsFromPane,function_call,sym_38272a2c116a89f7,same_file_name


###### 12. Validation and Extraction Summary

In [110]:
# Extraction loop
# ---------------------------------------------------------
# Stage 12A — Extraction Summary
# ---------------------------------------------------------

import pandas as pd

# Consolidate retrieval and parsing errors
error_records = []

for item in retrieval_errors:
    error_records.append({
        "error_type": "source_retrieval",
        "file_path": item.get("file_path"),
        "error": item.get("error")
    })

for item in parse_errors:
    error_records.append({
        "error_type": "ast_parsing",
        "file_path": item.get("file_path"),
        "error": item.get("error")
    })

errors_df = pd.DataFrame(
    error_records,
    columns=["error_type", "file_path", "error"]
)


def dataframe_summary(name: str, df: pd.DataFrame) -> dict:
    return {
        "dataframe": name,
        "rows": len(df),
        "columns": len(df.columns),
        "files": df["file_id"].nunique()
            if "file_id" in df.columns else None
    }


dataframes = {
    "files_df": files_df,
    "errors_df": errors_df,
    "symbols_df": symbols_df,
    "scopes_df": scopes_df,
    "imports_df": imports_df,
    "import_bindings_df": import_bindings_df,
    "exports_df": exports_df,
    "export_bindings_df": export_bindings_df,
    "references_df": references_df,
    "assignments_df": assignments_df,
    "calls_df": calls_df,
    "classes_df": classes_df,
    "inheritance_df": inheritance_df,
    "resolved_references_df": resolved_references_df,
    "resolved_calls_df": resolved_calls_df,
}

summary_df = pd.DataFrame([
    dataframe_summary(name, df)
    for name, df in dataframes.items()
])

display(summary_df)


# ---------------------------------------------------------
# Extraction distributions
# ---------------------------------------------------------

def display_distribution(df, column, title):
    if not df.empty and column in df.columns:
        print(f"\n{title}")
        display(
            df.groupby(column, dropna=False)
            .size()
            .rename("count")
            .reset_index()
            .sort_values("count", ascending=False)
        )


display_distribution(
    symbols_df, "symbol_type", "Symbol Types"
)

display_distribution(
    scopes_df, "scope_type", "Scope Types"
)

display_distribution(
    imports_df, "import_system", "Import Systems"
)

display_distribution(
    exports_df, "export_system", "Export Systems"
)

display_distribution(
    resolved_references_df,
    "resolution_status",
    "Reference Resolution Status"
)

display_distribution(
    resolved_calls_df,
    "resolution_status",
    "Call Resolution Status"
)

display_distribution(
    errors_df,
    "error_type",
    "Extraction Errors"
)

,dataframe,rows,columns,files
0,files_df,578,12,578.0
1,errors_df,28,3,NaN
2,symbols_df,63228,16,499.0
3,scopes_df,74469,9,550.0
4,imports_df,319,12,96.0
5,import_bindings_df,289,7,94.0
6,exports_df,225,11,107.0
7,export_bindings_df,269,7,107.0
8,references_df,254402,12,540.0
9,assignments_df,72972,11,496.0



Symbol Types


,symbol_type,count
5,variable,40539
3,function_expression,13940
0,arrow_function,4527
4,method,2692
2,function,1377
1,class,153



Scope Types


,scope_type,count
0,block,46351
5,function,22651
3,for_in_statement,1913
4,for_statement,1734
1,catch,1078
6,module,550
2,class,192



Import Systems


... (5 outputs left out)


Call Resolution Status


,resolution_status,count
2,UNRESOLVED,99027
1,RESOLVED,7734
0,AMBIGUOUS,2078



Extraction Errors


,error_type,count
0,source_retrieval,28


In [111]:
# QA Checks
# ---------------------------------------------------------
# Stage 12B — Consolidated QA
# ---------------------------------------------------------

qa_checks = []

def add_check(name, passed, actual=None, expected=None):
    qa_checks.append({
        "check": name,
        "passed": bool(passed),
        "actual": actual,
        "expected": expected
    })


# 1. Expected extraction totals
expected_counts = {
    "files_df": 578,
    "errors_df": 28,
    "symbols_df": 63228,
    "scopes_df": 74469,
    "imports_df": 319,
    "import_bindings_df": 289,
    "exports_df": 225,
    "export_bindings_df": 269,
    "references_df": 254402,
    "assignments_df": 72972,
    "calls_df": 108839,
    "classes_df": 384,
    "inheritance_df": 151,
}

for name, expected in expected_counts.items():
    actual = len(dataframes[name])
    add_check(
        f"{name} row count",
        actual == expected,
        actual,
        expected
    )


# 2. Resolution preserves source records
add_check(
    "Reference resolution preserves rows",
    len(resolved_references_df) == len(references_df),
    len(resolved_references_df),
    len(references_df)
)

add_check(
    "Call resolution preserves rows",
    len(resolved_calls_df) == len(calls_df),
    len(resolved_calls_df),
    len(calls_df)
)


# 3. Unique identifiers
unique_id_columns = {
    "symbols_df": ("symbols_df", "symbol_id"),
    "scopes_df": ("scopes_df", "scope_id"),
    "imports_df": ("imports_df", "import_id"),
    "exports_df": ("exports_df", "export_id"),
    "references_df": ("references_df", "reference_id"),
    "assignments_df": ("assignments_df", "assignment_id"),
    "calls_df": ("calls_df", "call_id"),
    "classes_df": ("classes_df", "class_id"),
    "inheritance_df": ("inheritance_df", "inheritance_id"),
    "resolved_references_df": (
        "resolved_references_df", "reference_id"
    ),
    "resolved_calls_df": ("resolved_calls_df", "call_id"),
}

for name, column in unique_id_columns.values():
    df_check = dataframes[name]
    actual = df_check[column].nunique()
    expected = len(df_check)

    add_check(
        f"{name} {column} unique",
        actual == expected,
        actual,
        expected
    )


# 4. Resolution status consistency
for name, df_check in [
    ("references", resolved_references_df),
    ("calls", resolved_calls_df)
]:
    counts = df_check["candidate_count"]

    add_check(
        f"{name}: resolved has exactly one candidate",
        (df_check.loc[
            df_check.resolution_status == "RESOLVED",
            "candidate_count"
        ] == 1).all()
    )

    add_check(
        f"{name}: ambiguous has multiple candidates",
        (df_check.loc[
            df_check.resolution_status == "AMBIGUOUS",
            "candidate_count"
        ] > 1).all()
    )

    add_check(
        f"{name}: unresolved has zero candidates",
        (df_check.loc[
            df_check.resolution_status == "UNRESOLVED",
            "candidate_count"
        ] == 0).all()
    )


# 5. Display results
qa_df = pd.DataFrame(qa_checks)

display(qa_df)

print(
    f"QA checks passed: "
    f"{qa_df['passed'].sum()} / {len(qa_df)}"
)

if not qa_df["passed"].all():
    display(qa_df.loc[~qa_df["passed"]])

,check,passed,actual,expected
0,files_df row count,True,578.0,578.0
1,errors_df row count,True,28.0,28.0
2,symbols_df row count,True,63228.0,63228.0
3,scopes_df row count,True,74469.0,74469.0
4,imports_df row count,True,319.0,319.0
5,import_bindings_df row count,True,289.0,289.0
6,exports_df row count,True,225.0,225.0
7,export_bindings_df row count,True,269.0,269.0
8,references_df row count,True,254402.0,254402.0
9,assignments_df row count,True,72972.0,72972.0


QA checks passed: 32 / 32


###### 13. Neo4j persistence

In [112]:
# ---------------------------------------------------------
# Step 13.1 — Inspect existing Neo4j graph
# ---------------------------------------------------------

queries = {
    "labels": """
        CALL db.labels()
        YIELD label
        RETURN label
        ORDER BY label
    """,

    "relationship_types": """
        CALL db.relationshipTypes()
        YIELD relationshipType
        RETURN relationshipType
        ORDER BY relationshipType
    """,

    "constraints": """
        SHOW CONSTRAINTS
        YIELD name, type, entityType, labelsOrTypes,
              properties
        RETURN *
        ORDER BY name
    """,

    "indexes": """
        SHOW INDEXES
        YIELD name, type, entityType, labelsOrTypes,
              properties, state
        RETURN *
        ORDER BY name
    """
}

for name, query in queries.items():
    print(f"\n{'=' * 70}")
    print(name.upper())
    print("=" * 70)

    result = run_cypher_query(query)
    display(pd.DataFrame(result))


LABELS


,label
0,Directory
1,FileName
2,FileType



RELATIONSHIP_TYPES


,relationshipType
0,CONTAINS
1,IS_TYPE_OF



CONSTRAINTS


""



INDEXES


,name,type,entityType,labelsOrTypes,properties,state
0,index_1b9dcc97,LOOKUP,RELATIONSHIP,None,None,ONLINE
1,index_460996c0,LOOKUP,NODE,None,None,ONLINE


In [113]:
# Step 13.1B — Inspect existing graph structure

queries = {
    "directories": """
        MATCH (d:Directory)
        RETURN d
        LIMIT 5
    """,

    "file_names": """
        MATCH (f:FileName)
        RETURN f
        LIMIT 5
    """,

    "file_types": """
        MATCH (ft:FileType)
        RETURN ft
        LIMIT 10
    """,

    "relationship_counts": """
        MATCH ()-[r]->()
        RETURN type(r) AS relationship_type,
               count(r) AS count
        ORDER BY relationship_type
    """
}

for name, query in queries.items():
    print(f"\n{name.upper()}")
    result = run_cypher_query(query)
    display(pd.DataFrame(result))


DIRECTORIES


,d
0,{'name': 'scss/mac/components'}
1,{'name': 'scss/preferences'}
2,{'name': 'scss/themes'}
3,{'name': 'scss/win'}
4,{'name': 'scss/win/abstracts'}



FILE_NAMES


,f
0,{'name': 'book_and_child_note.ris'}
1,{'name': 'book_and_snapshot.rdf'}
2,{'name': 'cell.csl'}
3,{'name': 'gbk.html'}
4,{'name': 'gbk.txt'}



FILE_TYPES


,ft
0,{'type': 'SQLite'}
1,{'type': 'CSL'}
2,{'type': 'RDF'}
3,{'type': 'PDF'}
4,{'type': 'XPI'}
5,{'type': 'ATOM'}
6,{'type': 'RSS'}
7,{'type': 'OPML'}
8,{'type': 'Epub'}
9,{'type': 'LUA'}



RELATIONSHIP_COUNTS


,relationship_type,count
0,CONTAINS,3489
1,IS_TYPE_OF,1634


In [114]:
# Step 13.1C - Inspect Relationship structure

queries = {
    "contains_samples": """
        MATCH (a)-[r:CONTAINS]->(b)
        RETURN labels(a) AS source_labels,
               properties(a) AS source,
               properties(r) AS relationship,
               labels(b) AS target_labels,
               properties(b) AS target
        LIMIT 10
    """,

    "contains_endpoint_counts": """
        MATCH (a)-[r:CONTAINS]->(b)
        RETURN labels(a) AS source_labels,
               labels(b) AS target_labels,
               count(r) AS relationship_count
        ORDER BY relationship_count DESC
    """,

    "file_type_samples": """
        MATCH (f:FileName)-[r:IS_TYPE_OF]->(ft:FileType)
        RETURN properties(f) AS file,
               properties(r) AS relationship,
               properties(ft) AS file_type
        LIMIT 10
    """
}

for name, query in queries.items():
    print(f"\n{name.upper()}")
    result = run_cypher_query(query)
    display(pd.DataFrame(result))


CONTAINS_SAMPLES


,source_labels,source,relationship,target_labels,target
0,[Directory],{'name': 'scss/mac/components'},{},[FileName],{'name': '_input.scss'}
1,[Directory],{'name': 'scss/mac/components'},{},[FileName],{'name': '_menulist.scss'}
2,[Directory],{'name': 'scss/mac/components'},{},[FileName],{'name': '_menupopup.scss'}
3,[Directory],{'name': 'scss/preferences'},{},[FileName],{'name': '_account.scss'}
4,[Directory],{'name': 'scss/preferences'},{},[FileName],{'name': '_advanced.scss'}
5,[Directory],{'name': 'scss/preferences'},{},[FileName],{'name': '_cite.scss'}
6,[Directory],{'name': 'scss/preferences'},{},[FileName],{'name': '_export.scss'}
7,[Directory],{'name': 'scss/preferences'},{},[FileName],{'name': '_general.scss'}
8,[Directory],{'name': 'scss/preferences'},{},[FileName],{'name': '_sync_reset.scss'}
9,[Directory],{'name': 'scss/themes'},{},[FileName],{'name': '_dark.scss'}



CONTAINS_ENDPOINT_COUNTS


,source_labels,target_labels,relationship_count
0,[Directory],[FileName],3489



FILE_TYPE_SAMPLES


,file,relationship,file_type
0,{'name': 'test.sqlite'},{},{'type': 'SQLite'}
1,{'name': '57173eea-07fc-3382-a3f7-548dff05def8...,{},{'type': 'SQLite'}
2,{'name': 'cell.csl'},{},{'type': 'CSL'}
3,{'name': 'handbook-of-legal-citations-zh.csl'},{},{'type': 'CSL'}
4,{'name': 'apa.csl'},{},{'type': 'CSL'}
5,{'name': 'book_and_snapshot.rdf'},{},{'type': 'RDF'}
6,{'name': 'update.rdf'},{},{'type': 'RDF'}
7,{'name': 'duplicatesMerge_JSTOR_1.pdf'},{},{'type': 'PDF'}
8,{'name': 'duplicatesMerge_JSTOR_2.pdf'},{},{'type': 'PDF'}
9,{'name': 'duplicatesMerge_annotated_1.pdf'},{},{'type': 'PDF'}


In [115]:
# Step 13.1C — Check existing file identity

queries = {
    "duplicate_filenames": """
        MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
        WITH f, count(d) AS directory_count,
             collect(d.name) AS directories
        WHERE directory_count > 1
        RETURN f.name AS file_name,
               directory_count,
               directories
        ORDER BY directory_count DESC
        LIMIT 20
    """,

    "multiple_file_types": """
        MATCH (f:FileName)-[:IS_TYPE_OF]->(ft:FileType)
        WITH f, count(ft) AS type_count,
             collect(ft.type) AS file_types
        WHERE type_count > 1
        RETURN f.name AS file_name,
               type_count,
               file_types
        ORDER BY type_count DESC
        LIMIT 20
    """,

    "file_type_counts": """
        MATCH (:FileName)-[:IS_TYPE_OF]->(ft:FileType)
        RETURN ft.type AS file_type,
               count(*) AS relationship_count
        ORDER BY relationship_count DESC
    """
}

for name, query in queries.items():
    print(f"\n{name.upper()}")
    result = run_cypher_query(query)
    display(pd.DataFrame(result))


DUPLICATE_FILENAMES


,file_name,directory_count,directories
0,browserSets.ftl,48,"[chrome/locale/af-ZA/zotero/mozilla, chrome/lo..."
1,arrowscrollbox.ftl,48,"[chrome/locale/af-ZA/zotero/mozilla, chrome/lo..."
2,zotero.dtd,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."
3,standalone.dtd,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."
4,zotero.ftl,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."
5,zotero.properties,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."
6,scaffold.ftl,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."
7,note-editor.ftl,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."
8,schema.ftl,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."
9,searchbox.dtd,48,"[chrome/locale/af-ZA/zotero, chrome/locale/ar/..."



MULTIPLE_FILE_TYPES


""



FILE_TYPE_COUNTS


,file_type,relationship_count
0,Javascript,555
1,SVG,319
2,CSS,160
3,Header,103
4,Other,97
5,C++,77
6,XHTML,51
7,PNG,37
8,HTML,23
9,JSON,23


In [116]:
# Step 13.2 — Validate JavaScript file mapping

query = """
MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
      -[:IS_TYPE_OF]->(ft:FileType)
WHERE ft.type = 'Javascript'
RETURN d.name AS directory,
       f.name AS file_name,
       ft.type AS file_type
ORDER BY directory, file_name
"""

neo4j_js_files_df = pd.DataFrame(run_cypher_query(query))

neo4j_js_files_df["file_path"] = (
    neo4j_js_files_df["directory"].fillna("").apply(
        lambda x: x.rstrip("/")
    )
    + "/"
    + neo4j_js_files_df["file_name"]
).str.lstrip("/")

print("Neo4j JavaScript files:", len(neo4j_js_files_df))
print("Unique paths:", neo4j_js_files_df["file_path"].nunique())
print("Extraction DataFrame files:", len(files_df))
print("Extraction unique paths:", files_df["file_path"].nunique())

missing_from_neo4j = sorted(
    set(files_df["file_path"]) - set(neo4j_js_files_df["file_path"])
)

extra_in_neo4j = sorted(
    set(neo4j_js_files_df["file_path"]) - set(files_df["file_path"])
)

print("Missing from Neo4j:", len(missing_from_neo4j))
print("Extra in Neo4j:", len(extra_in_neo4j))

Neo4j JavaScript files: 578
Unique paths: 578
Extraction DataFrame files: 578
Extraction unique paths: 578
Missing from Neo4j: 0
Extra in Neo4j: 0


In [117]:
# Step 13.3 — Build canonical JavaScript file mapping

js_file_mapping_df = (
    neo4j_js_files_df[
        ["directory", "file_name", "file_path"]
    ]
    .merge(
        files_df[
            ["file_id", "file_path", "file_size",
             "file_sha", "repository_revision"]
        ],
        on="file_path",
        how="left",
        validate="one_to_one"
    )
)

# Mapping QA
print("Mapped files:", len(js_file_mapping_df))
print("Unique file IDs:", js_file_mapping_df["file_id"].nunique())
print("Missing file IDs:", js_file_mapping_df["file_id"].isna().sum())
print("Duplicate file IDs:", js_file_mapping_df["file_id"].duplicated().sum())

display(js_file_mapping_df.head(10))

Mapped files: 578
Unique file IDs: 578
Missing file IDs: 0
Duplicate file IDs: 0


,directory,file_name,file_path,file_id,file_size,file_sha,repository_revision
0,ROOT,eslint.config.mjs,ROOT/eslint.config.mjs,file_b3236ac1011d7acd,None,None,None
1,app/assets,commandLineHandler.js,app/assets/commandLineHandler.js,file_611738591214a13f,None,None,None
2,app/assets,prefs.js,app/assets/prefs.js,file_da78ee625cb55359,None,None,None
3,chrome/content/scaffold,content.js,chrome/content/scaffold/content.js,file_bc42db590d995f48,None,None,None
4,chrome/content/scaffold,load.js,chrome/content/scaffold/load.js,file_02dccdc8a7fbf869,None,None,None
5,chrome/content/scaffold,scaffold.js,chrome/content/scaffold/scaffold.js,file_767abb7cb0102665,None,None,None
6,chrome/content/scaffold,scaffoldItemPreview.js,chrome/content/scaffold/scaffoldItemPreview.js,file_18307e89f1ddb102,None,None,None
7,chrome/content/scaffold,scaffoldItemPreviews.js,chrome/content/scaffold/scaffoldItemPreviews.js,file_a6b19f7562edaee3,None,None,None
8,chrome/content/scaffold,select.js,chrome/content/scaffold/select.js,file_2f58a1107b53283e,None,None,None
9,chrome/content/scaffold,translators.js,chrome/content/scaffold/translators.js,file_5ae4090ae7003e9e,None,None,None


In [118]:
# Step 13.4 — Add canonical File identity to existing nodes

rows = js_file_mapping_df[
    ["directory", "file_name", "file_id", "file_path"]
].to_dict(orient="records")

query = """
UNWIND $rows AS row

MATCH (d:Directory {name: row.directory})
      -[:CONTAINS]->
      (f:FileName {name: row.file_name})

SET f:File,
    f.file_id = row.file_id,
    f.file_path = row.file_path

RETURN count(f) AS updated_files,
       count(DISTINCT f) AS distinct_files
"""

result = run_cypher_query(query, parameters={"rows": rows})
display(pd.DataFrame(result))

,updated_files,distinct_files
0,578,555


In [119]:
# Step 13.4A - Diagnose duplicate matches# Step 13.4A — Identify non-unique graph matches

query = """
UNWIND $rows AS row

MATCH (d:Directory {name: row.directory})
      -[r:CONTAINS]->
      (f:FileName {name: row.file_name})

WITH row,
     count(*) AS match_count,
     count(DISTINCT f) AS distinct_nodes,
     count(DISTINCT r) AS distinct_relationships,
     collect(DISTINCT elementId(f)) AS node_ids

WHERE match_count > 1

RETURN row.file_path AS file_path,
       match_count,
       distinct_nodes,
       distinct_relationships,
       node_ids

ORDER BY distinct_nodes DESC, match_count DESC
"""

result = run_cypher_query(
    query,
    parameters={
        "rows": js_file_mapping_df[
            ["directory", "file_name", "file_path"]
        ].to_dict(orient="records")
    }
)

duplicate_matches_df = pd.DataFrame(result)

print("Files with non-unique matches:", len(duplicate_matches_df))
display(duplicate_matches_df)

Files with non-unique matches: 0


""


In [120]:
# Step 13.4B — Count canonical File nodes
query = """
MATCH (f:FileName)
WHERE f.file_path IS NOT NULL
RETURN
    count(f) AS file_nodes_with_path,
    count(DISTINCT f.file_path) AS unique_file_paths,
    count(DISTINCT f.file_id) AS unique_file_ids
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,file_nodes_with_path,unique_file_paths,unique_file_ids
0,555,555,555


In [121]:
# Step 13.4C — Find unmatched JavaScript files
query = """
UNWIND $rows AS row

OPTIONAL MATCH (d:Directory {name: row.directory})
               -[:CONTAINS]->
               (f:FileName {name: row.file_name})

WITH row, count(f) AS matches

WHERE matches = 0

RETURN row.file_path AS file_path,
       row.directory AS directory,
       row.file_name AS file_name

ORDER BY file_path
"""

unmatched_files_df = pd.DataFrame(
    run_cypher_query(
        query,
        parameters={
            "rows": js_file_mapping_df[
                ["directory", "file_name", "file_path"]
            ].to_dict(orient="records")
        }
    )
)

print("Unmatched files:", len(unmatched_files_df))
display(unmatched_files_df)

Unmatched files: 0


""


In [122]:
# Step 13.4D — Check whether different paths share nodes
query = """
UNWIND $rows AS row

MATCH (d:Directory {name: row.directory})
      -[:CONTAINS]->
      (f:FileName {name: row.file_name})

RETURN row.file_path AS file_path,
       elementId(f) AS neo4j_node_id,
       d.name AS directory,
       f.name AS file_name
ORDER BY file_path
"""

node_mapping_df = pd.DataFrame(
    run_cypher_query(
        query,
        parameters={
            "rows": js_file_mapping_df[
                ["directory", "file_name", "file_path"]
            ].to_dict(orient="records")
        }
    )
)

shared_nodes_df = (
    node_mapping_df
    .groupby("neo4j_node_id")
    .agg(
        path_count=("file_path", "nunique"),
        file_paths=("file_path", list)
    )
    .query("path_count > 1")
    .reset_index()
)

print("Matched paths:", len(node_mapping_df))
print("Distinct Neo4j nodes:", node_mapping_df["neo4j_node_id"].nunique())
print("Nodes shared by multiple paths:", len(shared_nodes_df))

display(shared_nodes_df)

Matched paths: 578
Distinct Neo4j nodes: 555
Nodes shared by multiple paths: 14


,neo4j_node_id,path_count,file_paths
0,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:1001,2,"[chrome/content/zotero/xpcom/reader.js, js-bui..."
1,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:1048,2,"[chrome/content/zotero/xpcom/zotero.js, defaul..."
2,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:1693,10,[resource/tinymce/plugins/autolink/plugin.min....
3,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:258,2,"[app/assets/commandLineHandler.js, chrome/cont..."
4,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:261,2,"[app/assets/prefs.js, chrome/content/zotero/xp..."
5,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:619,2,"[chrome/content/scaffold/translators.js, chrom..."
6,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:682,2,[chrome/content/zotero/actors/translation/http...
7,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:683,2,[chrome/content/zotero/actors/translation/tran...
8,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:710,3,"[chrome/content/zotero/components/utils.js, js..."
9,4:fc80ce30-6f9e-4bd4-b6d6-d9de90479ead:715,2,"[chrome/content/zotero/contextPane.js, chrome/..."


- We've found the root cause. The existing FileName nodes represent reusable filenames, not individual files. For example, one node named plugin.min.js can be connected to several directories.

** Revised graph structure **

```
(:Directory)-[:CONTAINS_FILE]->(:File)
                              |
                              +--[:HAS_NAME]->(:FileName)
                              |
                              +--[:IS_TYPE_OF]->(:FileType)
```

In [123]:
# Step 13.4E — Restore FileName nodes

query = """
MATCH (f:FileName)
WHERE f.file_id IS NOT NULL OR f.file_path IS NOT NULL
REMOVE f:File, f.file_id, f.file_path
RETURN count(f) AS restored_file_name_nodes
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,restored_file_name_nodes
0,555


In [124]:
# Step 13.5 — Create canonical File nodes
query = """
CREATE CONSTRAINT file_id_unique IF NOT EXISTS
FOR (f:File)
REQUIRE f.file_id IS UNIQUE
"""

run_cypher_query(query)

[]

In [125]:
rows = js_file_mapping_df[
    ["directory", "file_name", "file_id", "file_path"]
].to_dict(orient="records")

query = """
UNWIND $rows AS row

MATCH (d:Directory {name: row.directory})
      -[:CONTAINS]->
      (fn:FileName {name: row.file_name})

MERGE (f:File {file_id: row.file_id})
SET f.file_path = row.file_path,
    f.file_name = row.file_name,
    f.directory = row.directory

MERGE (d)-[:CONTAINS_FILE]->(f)
MERGE (f)-[:HAS_NAME]->(fn)

RETURN count(DISTINCT f) AS created_or_matched_files
"""

result = run_cypher_query(query, parameters={"rows": rows})
display(pd.DataFrame(result))

,created_or_matched_files
0,578


In [126]:
# Step 13.5A — Validate canonical File nodes

queries = {
    "file_counts": """
        MATCH (f:File)
        RETURN count(f) AS total_files,
               count(DISTINCT f.file_id) AS unique_ids,
               count(DISTINCT f.file_path) AS unique_paths
    """,

    "directory_relationships": """
        MATCH (:Directory)-[:CONTAINS_FILE]->(f:File)
        RETURN count(f) AS relationships,
               count(DISTINCT f) AS distinct_files
    """,

    "filename_relationships": """
        MATCH (f:File)-[:HAS_NAME]->(:FileName)
        RETURN count(f) AS relationships,
               count(DISTINCT f) AS distinct_files
    """,

    "missing_properties": """
        MATCH (f:File)
        WHERE f.file_id IS NULL
           OR f.file_path IS NULL
           OR f.file_name IS NULL
        RETURN count(f) AS incomplete_files
    """
}

for name, query in queries.items():
    print(f"\n{name.upper()}")
    result = run_cypher_query(query)
    display(pd.DataFrame(result))


FILE_COUNTS


,total_files,unique_ids,unique_paths
0,578,578,578



DIRECTORY_RELATIONSHIPS


,relationships,distinct_files
0,578,578



FILENAME_RELATIONSHIPS


,relationships,distinct_files
0,578,578



MISSING_PROPERTIES


,incomplete_files
0,0


In [127]:
# Step 13.6 — Connect canonical Files to FileTypes

query = """
MATCH (d:Directory)-[:CONTAINS]->(fn:FileName)
      -[:IS_TYPE_OF]->(ft:FileType)

MATCH (d)-[:CONTAINS_FILE]->(f:File)
      -[:HAS_NAME]->(fn)

WHERE ft.type = 'Javascript'

MERGE (f)-[:IS_TYPE_OF]->(ft)

RETURN count(DISTINCT f) AS classified_files
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,classified_files
0,578


In [128]:
# Step 13.6A — Validate Javascript classification
query = """
MATCH (f:File)-[:IS_TYPE_OF]->(ft:FileType)
RETURN ft.type AS file_type,
       count(f) AS file_count,
       count(DISTINCT f.file_id) AS unique_files
ORDER BY file_count DESC
"""

js_classification_df = pd.DataFrame(run_cypher_query(query))
display(js_classification_df)

,file_type,file_count,unique_files
0,Javascript,578,578


In [129]:
# Step 13.7 — Inspect available file types

query = """
MATCH (ft:FileType)
RETURN ft.type AS file_type
ORDER BY file_type
"""

file_types_df = pd.DataFrame(run_cypher_query(query))
display(file_types_df)

,file_type
0,.ini
1,.manifest
2,ATOM
3,C
4,C++
5,CSL
6,CSS
7,DTD_XML_def
8,Epub
9,FreeMarker_Java_template


In [130]:
# Step 13.8 — Profile existing inventory by extension and type

query = """
MATCH (d:Directory)-[:CONTAINS]->(fn:FileName)
      -[:IS_TYPE_OF]->(ft:FileType)

RETURN DISTINCT
       d.name AS directory,
       fn.name AS file_name,
       ft.type AS file_type
ORDER BY file_type, directory, file_name
"""

inventory_df = pd.DataFrame(run_cypher_query(query))

inventory_df["file_path"] = (
    inventory_df["directory"].fillna("").str.rstrip("/")
    + "/"
    + inventory_df["file_name"]
).str.lstrip("/")

inventory_df["extension"] = (
    inventory_df["file_name"]
    .str.extract(r"(\.[^.]+)$", expand=False)
    .fillna("[no extension]")
    .str.lower()
)

print("Inventory paths:", inventory_df["file_path"].nunique())

print("\nFILE TYPE COUNTS")
display(
    inventory_df.groupby("file_type")["file_path"]
    .nunique()
    .sort_values(ascending=False)
    .rename("file_count")
    .reset_index()
)

print("\nPHASE 1 EXTENSIONS")
phase1_extensions = [
    ".js", ".mjs", ".html", ".htm", ".xhtml",
    ".xml", ".xul", ".json", ".css", ".scss"
]

display(
    inventory_df[
        inventory_df["extension"].isin(phase1_extensions)
    ].groupby(["extension", "file_type"])["file_path"]
    .nunique()
    .reset_index(name="file_count")
    .sort_values(["extension", "file_type"])
)

Inventory paths: 3489

FILE TYPE COUNTS


,file_type,file_count
0,SVG,774
1,FreeMarker_Java_template,673
2,Javascript,578
3,DTD_XML_def,482
4,CSS,202
5,Java_prop,146
6,Other,108
7,Header,107
8,C++,77
9,JSON,63



PHASE 1 EXTENSIONS


,extension,file_type,file_count
0,.css,CSS,39
1,.html,HTML,23
2,.js,Javascript,489
3,.json,JSON,63
4,.mjs,Javascript,63
5,.scss,CSS,163
6,.xhtml,XHTML,51
7,.xml,XML,4
8,.xul,XUL,6


In [131]:
# Step 13.9 — Build Phase 1 inventory

phase1_extensions = [
    ".js", ".mjs",
    ".html", ".xhtml",
    ".xml", ".xul",
    ".json",
    ".css", ".scss"
]

phase1_files_df = (
    inventory_df[
        inventory_df["extension"].isin(phase1_extensions)
    ]
    .copy()
    .sort_values("file_path")
    .reset_index(drop=True)
)

# Verify path uniqueness
print("Phase 1 files:", len(phase1_files_df))
print("Unique paths:", phase1_files_df["file_path"].nunique())

print("\nCounts by extension:")
display(
    phase1_files_df.groupby("extension")
    .size()
    .rename("file_count")
    .reset_index()
)

print("\nCounts by file type:")
display(
    phase1_files_df.groupby("file_type")
    .size()
    .rename("file_count")
    .reset_index()
)

Phase 1 files: 901
Unique paths: 901

Counts by extension:


,extension,file_count
0,.css,39
1,.html,23
2,.js,489
3,.json,63
4,.mjs,63
5,.scss,163
6,.xhtml,51
7,.xml,4
8,.xul,6



Counts by file type:


,file_type,file_count
0,CSS,202
1,HTML,23
2,JSON,63
3,Javascript,552
4,XHTML,51
5,XML,4
6,XUL,6


In [132]:
# Step 13.10 — Assign canonical IDs to new Phase 1 files

phase1_files_df["file_id"] = phase1_files_df["file_path"].apply(
    lambda path: stable_id(REPOSITORY_ID, path, prefix="file_")
)

# Identify files not already represented by JavaScript File nodes
javascript_paths = set(js_file_mapping_df["file_path"])

additional_files_df = (
    phase1_files_df[
        ~phase1_files_df["file_path"].isin(javascript_paths)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Phase 1 files:", len(phase1_files_df))
print("Existing JavaScript files:", len(javascript_paths))
print("Additional files:", len(additional_files_df))
print("Unique additional IDs:", additional_files_df["file_id"].nunique())
print("Missing IDs:", additional_files_df["file_id"].isna().sum())

display(
    additional_files_df[
        ["file_id", "file_path", "extension", "file_type"]
    ].head(10)
)

Phase 1 files: 901
Existing JavaScript files: 578
Additional files: 349
Unique additional IDs: 349
Missing IDs: 0


,file_id,file_path,extension,file_type
0,file_713bec1f873ac982,ROOT/package-lock.json,.json,JSON
1,file_e98cdd9ff13769f2,ROOT/package.json,.json,JSON
2,file_081952353c2d936b,app/assets/mac/chrome/skin/preferences/prefere...,.css,CSS
3,file_1be062bdc3c248e6,app/assets/unix/skin/preferences/preferences.css,.css,CSS
4,file_bbab0a229eae0da0,app/assets/win/skin/preferences/preferences.css,.css,CSS
5,file_4d385c75fb1ca598,app/mac/entitlements.xml,.xml,XML
6,file_e757f8e2b525f988,chrome/content/scaffold/load.xhtml,.xhtml,XHTML
7,file_d8cd79fdfb21a77d,chrome/content/scaffold/monaco/monaco.html,.html,HTML
8,file_ea7f6ca9ccd1da53,chrome/content/scaffold/scaffold.xhtml,.xhtml,XHTML
9,file_7f41c43846b5f102,chrome/content/scaffold/select.xhtml,.xhtml,XHTML


In [134]:
# Find JavaScript files outside the nine selected extensions

javascript_inventory_df = inventory_df[
    inventory_df["file_type"] == "Javascript"
].copy()

uncovered_js_df = javascript_inventory_df[
    ~javascript_inventory_df["file_path"].isin(
        phase1_files_df["file_path"]
    )
]

print("Total JavaScript files:", len(javascript_inventory_df))
print("Already in Phase 1:", len(
    javascript_inventory_df[
        javascript_inventory_df["file_path"].isin(
            phase1_files_df["file_path"]
        )
    ]
))
print("Outside Phase 1 extensions:", len(uncovered_js_df))

display(
    uncovered_js_df[
        ["file_path", "extension", "file_type"]
    ].sort_values("file_path")
)

Total JavaScript files: 578
Already in Phase 1: 552
Outside Phase 1 extensions: 26


,file_path,extension,file_type
1839,chrome/content/zotero/collectionTree.jsx,.jsx,Javascript
1840,chrome/content/zotero/collectionViewItemTree.jsx,.jsx,Javascript
1910,chrome/content/zotero/components/annotation.jsx,.jsx,Javascript
1911,chrome/content/zotero/components/button.jsx,.jsx,Javascript
1912,chrome/content/zotero/components/draggable.jsx,.jsx,Javascript
1913,chrome/content/zotero/components/editable.jsx,.jsx,Javascript
1923,chrome/content/zotero/components/editable/cont...,.jsx,Javascript
1924,chrome/content/zotero/components/form/input.jsx,.jsx,Javascript
1925,chrome/content/zotero/components/form/select.jsx,.jsx,Javascript
1926,chrome/content/zotero/components/form/textArea...,.jsx,Javascript


Total JavaScript files: 578
Already in Phase 1: 552
Outside Phase 1 extensions: 26


,file_path,extension,file_type
1839,chrome/content/zotero/collectionTree.jsx,.jsx,Javascript
1840,chrome/content/zotero/collectionViewItemTree.jsx,.jsx,Javascript
1910,chrome/content/zotero/components/annotation.jsx,.jsx,Javascript
1911,chrome/content/zotero/components/button.jsx,.jsx,Javascript
1912,chrome/content/zotero/components/draggable.jsx,.jsx,Javascript
1913,chrome/content/zotero/components/editable.jsx,.jsx,Javascript
1923,chrome/content/zotero/components/editable/cont...,.jsx,Javascript
1924,chrome/content/zotero/components/form/input.jsx,.jsx,Javascript
1925,chrome/content/zotero/components/form/select.jsx,.jsx,Javascript
1926,chrome/content/zotero/components/form/textArea...,.jsx,Javascript


In [135]:
# Step 13.10B — Include JSX in Phase 1

phase1_extensions = [
    ".js", ".mjs", ".jsx",
    ".html", ".xhtml",
    ".xml", ".xul",
    ".json",
    ".css", ".scss"
]

phase1_files_df = (
    inventory_df[
        inventory_df["extension"].isin(phase1_extensions)
    ]
    .copy()
    .sort_values("file_path")
    .reset_index(drop=True)
)

phase1_files_df["file_id"] = phase1_files_df["file_path"].apply(
    lambda path: stable_id(REPOSITORY_ID, path, prefix="file_")
)

javascript_paths = set(js_file_mapping_df["file_path"])

additional_files_df = (
    phase1_files_df[
        ~phase1_files_df["file_path"].isin(javascript_paths)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Phase 1 files:", len(phase1_files_df))
print("Unique paths:", phase1_files_df["file_path"].nunique())
print("Existing JavaScript files:", len(javascript_paths))
print("Additional files:", len(additional_files_df))
print("Unique IDs:", phase1_files_df["file_id"].nunique())

print("\nCounts by extension:")
display(
    phase1_files_df.groupby("extension")
    .size()
    .rename("file_count")
    .reset_index()
)

Phase 1 files: 927
Unique paths: 927
Existing JavaScript files: 578
Additional files: 349
Unique IDs: 927

Counts by extension:


,extension,file_count
0,.css,39
1,.html,23
2,.js,489
3,.json,63
4,.jsx,26
5,.mjs,63
6,.scss,163
7,.xhtml,51
8,.xml,4
9,.xul,6


In [137]:
# Step 13.11A — Check for existing paths before ingestion

existing_file_paths_df = run_cypher_query("""
MATCH (f:File)
RETURN f.file_id AS file_id,
       f.file_path AS file_path
ORDER BY f.file_path
""")

existing_file_paths = {
    row["file_path"]
    for row in existing_file_paths_df
    if row.get("file_path") is not None
}

additional_existing_df = additional_files_df[
    additional_files_df["file_path"].isin(existing_file_paths)
].copy()

print("Existing paths found among additional files:", len(additional_existing_df))
display(additional_existing_df.head())

Existing paths found among additional files: 0


,directory,file_name,file_type,file_path,extension,file_id


In [138]:
# Step 13.11B — Create the 349 additional File nodes

rows = additional_files_df[
    ["file_id", "file_path", "file_name", "directory", "file_type"]
].to_dict("records")

query = """
UNWIND $rows AS row

MATCH (d:Directory {name: row.directory})
      -[:CONTAINS]->
      (fn:FileName {name: row.file_name})

MATCH (ft:FileType {type: row.file_type})

MERGE (f:File {file_id: row.file_id})
SET f.file_path = row.file_path,
    f.file_name = row.file_name,
    f.directory = row.directory

MERGE (d)-[:CONTAINS_FILE]->(f)
MERGE (f)-[:HAS_NAME]->(fn)
MERGE (f)-[:IS_TYPE_OF]->(ft)

RETURN count(DISTINCT f) AS files_created_or_matched
"""

result = run_cypher_query(query, {"rows": rows})

print(result)

[{'files_created_or_matched': 349}]


In [139]:
# Step 13.12 — Validate the complete inventory

result = run_cypher_query("""
MATCH (f:File)
OPTIONAL MATCH (f)-[:IS_TYPE_OF]->(ft:FileType)
RETURN
    count(DISTINCT f) AS total_files,
    count(DISTINCT f.file_id) AS unique_file_ids,
    count(DISTINCT f.file_path) AS unique_file_paths,
    count(DISTINCT CASE
        WHEN ft.type = 'Javascript' THEN f
    END) AS javascript_files,
    count(DISTINCT CASE
        WHEN ft.type <> 'Javascript' THEN f
    END) AS non_javascript_files
""")

display(pd.DataFrame(result))

,total_files,unique_file_ids,unique_file_paths,javascript_files,non_javascript_files
0,927,927,927,578,349


In [140]:
# Step 13.13 — Validate file relationships

result = run_cypher_query("""
MATCH (f:File)
OPTIONAL MATCH (d:Directory)-[:CONTAINS_FILE]->(f)
WITH f, count(DISTINCT d) AS directory_count

OPTIONAL MATCH (f)-[:HAS_NAME]->(fn:FileName)
WITH f, directory_count, count(DISTINCT fn) AS filename_count

OPTIONAL MATCH (f)-[:IS_TYPE_OF]->(ft:FileType)
RETURN
    count(f) AS total_files,
    sum(CASE WHEN directory_count = 0 THEN 1 ELSE 0 END)
        AS missing_directory,
    sum(CASE WHEN filename_count = 0 THEN 1 ELSE 0 END)
        AS missing_filename,
    sum(CASE WHEN ft IS NULL THEN 1 ELSE 0 END)
        AS missing_file_type
""")

display(pd.DataFrame(result))

,total_files,missing_directory,missing_filename,missing_file_type
0,927,0,0,0


In [141]:
# Step 13.14A - Inspect existing mapping

display(
    files_df[
        ["file_id", "file_path", "file_name"]
    ].head()
)

,file_id,file_path,file_name
0,file_b3236ac1011d7acd,ROOT/eslint.config.mjs,eslint.config.mjs
1,file_611738591214a13f,app/assets/commandLineHandler.js,commandLineHandler.js
2,file_da78ee625cb55359,app/assets/prefs.js,prefs.js
3,file_bc42db590d995f48,chrome/content/scaffold/content.js,content.js
4,file_02dccdc8a7fbf869,chrome/content/scaffold/load.js,load.js


In [142]:
# Step 13.14B — Build the file ID mapping

file_id_mapping = (
    files_df
    .set_index("file_path")["file_id"]
    .to_dict()
)

print("File mappings:", len(file_id_mapping))
print("Unique canonical IDs:", len(set(file_id_mapping.values())))

File mappings: 578
Unique canonical IDs: 578


In [143]:
# Step 13.15 — Identify DataFrames requiring file ID conversion

dataframes = {
    "symbols_df": symbols_df,
    "scopes_df": scopes_df,
    "imports_df": imports_df,
    "import_bindings_df": import_bindings_df,
    "exports_df": exports_df,
    "export_bindings_df": export_bindings_df,
    "references_df": references_df,
    "assignments_df": assignments_df,
    "calls_df": calls_df,
    "classes_df": classes_df,
    "inheritance_df": inheritance_df,
    "resolved_references_df": resolved_references_df,
    "resolved_calls_df": resolved_calls_df,
}

for name, df in dataframes.items():
    if "file_id" in df.columns:
        paths = set(df["file_id"].dropna())
        unmatched = paths - set(file_id_mapping.keys())

        print(
            f"{name}: rows={len(df):,}, "
            f"unique files={len(paths):,}, "
            f"unmatched={len(unmatched):,}"
        )

symbols_df: rows=63,228, unique files=499, unmatched=0
scopes_df: rows=74,469, unique files=550, unmatched=0
imports_df: rows=319, unique files=96, unmatched=0
import_bindings_df: rows=289, unique files=94, unmatched=0
exports_df: rows=225, unique files=107, unmatched=0
export_bindings_df: rows=269, unique files=107, unmatched=0
references_df: rows=254,402, unique files=540, unmatched=0
assignments_df: rows=72,972, unique files=496, unmatched=0
calls_df: rows=108,839, unique files=539, unmatched=0
classes_df: rows=384, unique files=126, unmatched=0
inheritance_df: rows=151, unique files=105, unmatched=0
resolved_references_df: rows=254,402, unique files=540, unmatched=0
resolved_calls_df: rows=108,839, unique files=539, unmatched=0


In [144]:
# Step 13.16 — Convert file IDs

canonical_dataframes = {}

for name, df in dataframes.items():
    converted_df = df.copy()

    if "file_id" in converted_df.columns:
        converted_df["file_id"] = converted_df["file_id"].map(
            file_id_mapping
        )

    canonical_dataframes[name] = converted_df

for name, df in canonical_dataframes.items():
    print(
        f"{name}: rows={len(df):,}, "
        f"missing file IDs={df['file_id'].isna().sum():,}"
    )

symbols_df: rows=63,228, missing file IDs=0
scopes_df: rows=74,469, missing file IDs=0
imports_df: rows=319, missing file IDs=0
import_bindings_df: rows=289, missing file IDs=0
exports_df: rows=225, missing file IDs=0
export_bindings_df: rows=269, missing file IDs=0
references_df: rows=254,402, missing file IDs=0
assignments_df: rows=72,972, missing file IDs=0
calls_df: rows=108,839, missing file IDs=0
classes_df: rows=384, missing file IDs=0
inheritance_df: rows=151, missing file IDs=0
resolved_references_df: rows=254,402, missing file IDs=0
resolved_calls_df: rows=108,839, missing file IDs=0


In [145]:
# Step 13.17 — Inspect entity identity columns

for name, df in canonical_dataframes.items():
    id_columns = [
        col for col in df.columns
        if col.endswith("_id")
    ]

    print(f"\n{name}")
    print("  Rows:", f"{len(df):,}")
    print("  ID columns:", id_columns)

    for col in id_columns:
        print(
            f"  {col}: "
            f"{df[col].nunique(dropna=True):,} unique, "
            f"{df[col].isna().sum():,} null"
        )


symbols_df
  Rows: 63,228
  ID columns: ['symbol_id', 'file_id', 'parent_symbol_id', 'scope_id']
  symbol_id: 63,228 unique, 0 null
  file_id: 499 unique, 0 null
  parent_symbol_id: 0 unique, 63,228 null
  scope_id: 38,173 unique, 0 null

scopes_df
  Rows: 74,469
  ID columns: ['scope_id', 'file_id', 'parent_scope_id', 'owner_symbol_id']
  scope_id: 74,469 unique, 0 null
  file_id: 550 unique, 0 null
  parent_scope_id: 43,029 unique, 550 null
  owner_symbol_id: 0 unique, 74,469 null

imports_df
  Rows: 319
  ID columns: ['import_id', 'file_id']
  import_id: 319 unique, 0 null
  file_id: 96 unique, 0 null

import_bindings_df
  Rows: 289
  ID columns: ['import_binding_id', 'import_id', 'file_id']
  import_binding_id: 289 unique, 0 null
  import_id: 259 unique, 0 null
  file_id: 94 unique, 0 null

exports_df
  Rows: 225
  ID columns: ['export_id', 'file_id']
  export_id: 225 unique, 0 null
  file_id: 107 unique, 0 null

export_bindings_df
  Rows: 269
  ID columns: ['export_binding_id', '

In [146]:
# Step 13.18 — Inspect null relationship fields

columns_to_check = {
    "symbols_df": ["parent_symbol_id", "scope_id"],
    "scopes_df": ["parent_scope_id", "owner_symbol_id"],
    "references_df": ["scope_id", "resolved_symbol_id"],
}

for name, columns in columns_to_check.items():
    df = canonical_dataframes[name]

    print(f"\n{name}")

    for col in columns:
        print(f"\n{col}")
        print("Non-null:", df[col].notna().sum())
        print("Null:", df[col].isna().sum())

        if df[col].notna().any():
            display(df.loc[df[col].notna(), [col]].head(5))


symbols_df

parent_symbol_id
Non-null: 0
Null: 63228

scope_id
Non-null: 63228
Null: 0


,scope_id
0,scope_54aafa45f9b4cdc9
1,scope_54aafa45f9b4cdc9
2,scope_4114a771dc9a0d0c
3,scope_96b025d4493a471e
4,scope_96b025d4493a471e



scopes_df

parent_scope_id
Non-null: 73919
Null: 550


,parent_scope_id
1,scope_54aafa45f9b4cdc9
2,scope_54aafa45f9b4cdc9
3,scope_54aafa45f9b4cdc9
4,scope_54aafa45f9b4cdc9
5,scope_54aafa45f9b4cdc9



owner_symbol_id
Non-null: 0
Null: 74469

references_df

scope_id
Non-null: 0
Null: 254402

resolved_symbol_id
Non-null: 0
Null: 254402


In [147]:
# Step 13.18A — Validate Symbol-to-Scope references

symbol_scope_ids = set(
    canonical_dataframes["symbols_df"]["scope_id"].dropna()
)

defined_scope_ids = set(
    canonical_dataframes["scopes_df"]["scope_id"].dropna()
)

missing_symbol_scopes = symbol_scope_ids - defined_scope_ids

print("Distinct scopes referenced by symbols:", len(symbol_scope_ids))
print("Distinct scopes defined:", len(defined_scope_ids))
print("Missing scope definitions:", len(missing_symbol_scopes))

Distinct scopes referenced by symbols: 38173
Distinct scopes defined: 74469
Missing scope definitions: 0


In [148]:
# Step 13.19 — Validate parent scope references

scopes = canonical_dataframes["scopes_df"]

parent_scope_ids = set(
    scopes["parent_scope_id"].dropna()
)

defined_scope_ids = set(scopes["scope_id"])

missing_parent_scopes = parent_scope_ids - defined_scope_ids

print("Parent scope references:", len(parent_scope_ids))
print("Defined scopes:", len(defined_scope_ids))
print("Missing parent scopes:", len(missing_parent_scopes))

Parent scope references: 43029
Defined scopes: 74469
Missing parent scopes: 0


In [149]:
# Step 13.20 — Create uniqueness constraints

constraints = [
    """
    CREATE CONSTRAINT scope_id_unique IF NOT EXISTS
    FOR (s:Scope)
    REQUIRE s.scope_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT symbol_id_unique IF NOT EXISTS
    FOR (s:Symbol)
    REQUIRE s.symbol_id IS UNIQUE
    """
]

for query in constraints:
    run_cypher_query(query)

print("Scope and Symbol uniqueness constraints created.")

Scope and Symbol uniqueness constraints created.


In [150]:
# Step 13.21 — Ingest Scope nodes

scopes_df_canonical = canonical_dataframes["scopes_df"]

scope_columns = [
    "scope_id",
    "file_id",
    "parent_scope_id",
    "scope_type",
    "owner_symbol_id",
    "start_line",
    "end_line",
]

scope_rows = (
    scopes_df_canonical[scope_columns]
    .where(pd.notna(scopes_df_canonical[scope_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (s:Scope {scope_id: row.scope_id})
SET s.file_id = row.file_id,
    s.parent_scope_id = row.parent_scope_id,
    s.scope_type = row.scope_type,
    s.owner_symbol_id = row.owner_symbol_id,
    s.start_line = row.start_line,
    s.end_line = row.end_line
RETURN count(s) AS scopes_created_or_matched
"""

result = run_cypher_query(query, {"rows": scope_rows})

display(pd.DataFrame(result))

,scopes_created_or_matched
0,74469


In [151]:
# Step 13.22 — Connect Scope nodes to Files

query = """
MATCH (f:File {file_id: $file_id})
MATCH (s:Scope {scope_id: $scope_id})
MERGE (f)-[:HAS_SCOPE]->(s)
RETURN count(s) AS scopes_connected
"""

scope_rows = (
    canonical_dataframes["scopes_df"]
    [["scope_id", "file_id"]]
    .to_dict("records")
)

connected = 0

for row in scope_rows:
    result = run_cypher_query(query, row)
    connected += result[0]["scopes_connected"]

print("Scope-to-File relationships:", connected)

Scope-to-File relationships: 74469


In [152]:
# Step 13.23 — Build the Scope hierarchy

scopes = canonical_dataframes["scopes_df"]

scope_relationship_rows = (
    scopes[["scope_id", "parent_scope_id"]]
    .dropna(subset=["parent_scope_id"])
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (parent:Scope {scope_id: row.parent_scope_id})
MATCH (child:Scope {scope_id: row.scope_id})
MERGE (parent)-[:CONTAINS_SCOPE]->(child)
RETURN count(*) AS relationships_created_or_matched
"""

result = run_cypher_query(
    query,
    {"rows": scope_relationship_rows}
)

display(pd.DataFrame(result))

,relationships_created_or_matched
0,73919


In [153]:
# Step 13.24 — Validate Scope graph integrity

result = run_cypher_query("""
MATCH (s:Scope)
OPTIONAL MATCH (f:File)-[:HAS_SCOPE]->(s)
WITH s, count(DISTINCT f) AS file_count

OPTIONAL MATCH (parent:Scope)-[:CONTAINS_SCOPE]->(s)
WITH s, file_count, count(DISTINCT parent) AS parent_count

RETURN
    count(s) AS total_scopes,
    sum(CASE WHEN file_count = 0 THEN 1 ELSE 0 END)
        AS scopes_without_file,
    sum(CASE WHEN file_count > 1 THEN 1 ELSE 0 END)
        AS scopes_with_multiple_files,
    sum(CASE WHEN parent_count > 1 THEN 1 ELSE 0 END)
        AS scopes_with_multiple_parents,
    sum(CASE WHEN parent_count = 0 THEN 1 ELSE 0 END)
        AS root_scopes
""")

display(pd.DataFrame(result))

,total_scopes,scopes_without_file,scopes_with_multiple_files,scopes_with_multiple_parents,root_scopes
0,74469,0,0,0,550


In [154]:
# Step 13.25 — Ingest Symbol nodes

symbols = canonical_dataframes["symbols_df"]

symbol_columns = [
    "symbol_id",
    "name",
    "qualified_name",
    "symbol_type",
    "declaration_type",
    "parent_symbol_id",
    "scope_id",
    "is_exported",
    "start_line",
    "end_line",
    "start_column",
    "end_column",
    "start_byte",
    "end_byte",
    "source_text",
]

symbol_rows = (
    symbols[symbol_columns]
    .where(pd.notna(symbols[symbol_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (s:Symbol {symbol_id: row.symbol_id})
SET s.name = row.name,
    s.qualified_name = row.qualified_name,
    s.symbol_type = row.symbol_type,
    s.declaration_type = row.declaration_type,
    s.parent_symbol_id = row.parent_symbol_id,
    s.scope_id = row.scope_id,
    s.is_exported = row.is_exported,
    s.start_line = row.start_line,
    s.end_line = row.end_line,
    s.start_column = row.start_column,
    s.end_column = row.end_column,
    s.start_byte = row.start_byte,
    s.end_byte = row.end_byte,
    s.source_text = row.source_text
RETURN count(s) AS symbols_created_or_matched
"""

result = run_cypher_query(query, {"rows": symbol_rows})

display(pd.DataFrame(result))

,symbols_created_or_matched
0,63228


In [155]:
# Step 13.26 — Connect Symbols to Files and Scopes

symbols = canonical_dataframes["symbols_df"]

symbol_relationship_rows = (
    symbols[["symbol_id", "file_id", "scope_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row

MATCH (f:File {file_id: row.file_id})
MATCH (s:Symbol {symbol_id: row.symbol_id})
MATCH (sc:Scope {scope_id: row.scope_id})

MERGE (f)-[:DECLARES]->(s)
MERGE (s)-[:IN_SCOPE]->(sc)

RETURN count(DISTINCT s) AS symbols_connected
"""

batch_size = 5000
connected = 0

for start in range(0, len(symbol_relationship_rows), batch_size):
    batch = symbol_relationship_rows[start:start + batch_size]

    result = run_cypher_query(query, {"rows": batch})
    connected += result[0]["symbols_connected"]

print("Symbols connected:", connected)
print("Expected:", len(symbol_relationship_rows))

Symbols connected: 63228
Expected: 63228


In [156]:
# Step 13.27 — Validate Symbol graph integrity

result = run_cypher_query("""
MATCH (s:Symbol)
OPTIONAL MATCH (f:File)-[:DECLARES]->(s)
WITH s, count(DISTINCT f) AS file_count

OPTIONAL MATCH (s)-[:IN_SCOPE]->(sc:Scope)
WITH s, file_count, count(DISTINCT sc) AS scope_count

RETURN
    count(s) AS total_symbols,
    sum(CASE WHEN file_count = 0 THEN 1 ELSE 0 END)
        AS symbols_without_file,
    sum(CASE WHEN file_count > 1 THEN 1 ELSE 0 END)
        AS symbols_with_multiple_files,
    sum(CASE WHEN scope_count = 0 THEN 1 ELSE 0 END)
        AS symbols_without_scope,
    sum(CASE WHEN scope_count > 1 THEN 1 ELSE 0 END)
        AS symbols_with_multiple_scopes
""")

display(pd.DataFrame(result))

,total_symbols,symbols_without_file,symbols_with_multiple_files,symbols_without_scope,symbols_with_multiple_scopes
0,63228,0,0,0,0


In [157]:
# Step 13.28 — Inspect Import and ImportBinding structures

for name in ["imports_df", "import_bindings_df"]:
    df = canonical_dataframes[name]

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head(3))


imports_df
Rows: 319
Columns: ['import_id', 'file_id', 'module_specifier', 'import_text', 'import_kind', 'import_system', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,import_id,file_id,module_specifier,import_text,import_kind,import_system,start_line,start_column,end_line,end_column,start_byte,end_byte
0,imp_d158a1fb0518df49,file_ebe5fee72d4d11cc,chrome://zotero/content/xpcom/translate/testTr...,import { AbstractWebTranslationEnvironment } f...,es_module,es_module,1,1,1,130,0,129
1,imp_e493445f19fe03e3,file_50524eedfc834457,chrome://zotero/content/BlockingObserver.mjs,"import { BlockingObserver } from ""chrome://zot...",es_module,es_module,27,1,27,81,939,1019
2,imp_cb4be66627adec70,file_50524eedfc834457,chrome://zotero/content/actors/ActorManager.mjs,"import ""chrome://zotero/content/actors/ActorMa...",side_effect,es_module,28,1,28,58,1020,1077



import_bindings_df
Rows: 289
Columns: ['import_binding_id', 'import_id', 'file_id', 'local_name', 'imported_name', 'binding_type', 'binding_text']


,import_binding_id,import_id,file_id,local_name,imported_name,binding_type,binding_text
0,ib_b003f0d03c4953cf,imp_d158a1fb0518df49,file_ebe5fee72d4d11cc,AbstractWebTranslationEnvironment,AbstractWebTranslationEnvironment,named,AbstractWebTranslationEnvironment
1,ib_8ecc706c97eb3119,imp_e493445f19fe03e3,file_50524eedfc834457,BlockingObserver,BlockingObserver,named,BlockingObserver
2,ib_5d368b4588c05bd5,imp_7069ee4ea240432e,file_56f736dc1fd42142,NetUtil,NetUtil,named,NetUtil


In [158]:
# Step 13.29 — Create Import uniqueness constraints

constraints = [
    """
    CREATE CONSTRAINT import_id_unique IF NOT EXISTS
    FOR (i:Import)
    REQUIRE i.import_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT import_binding_id_unique IF NOT EXISTS
    FOR (b:ImportBinding)
    REQUIRE b.import_binding_id IS UNIQUE
    """
]

for query in constraints:
    run_cypher_query(query)

print("Import and ImportBinding uniqueness constraints created.")

Import and ImportBinding uniqueness constraints created.


In [159]:
# Step 13.30 — Ingest Import nodes

imports = canonical_dataframes["imports_df"]

import_columns = [
    "import_id",
    "module_specifier",
    "import_text",
    "import_kind",
    "import_system",
    "start_line",
    "start_column",
    "end_line",
    "end_column",
    "start_byte",
    "end_byte",
]

import_rows = (
    imports[import_columns]
    .where(pd.notna(imports[import_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (i:Import {import_id: row.import_id})
SET i.module_specifier = row.module_specifier,
    i.import_text = row.import_text,
    i.import_kind = row.import_kind,
    i.import_system = row.import_system,
    i.start_line = row.start_line,
    i.start_column = row.start_column,
    i.end_line = row.end_line,
    i.end_column = row.end_column,
    i.start_byte = row.start_byte,
    i.end_byte = row.end_byte
RETURN count(i) AS imports_created_or_matched
"""

result = run_cypher_query(query, {"rows": import_rows})

display(pd.DataFrame(result))

,imports_created_or_matched
0,319


In [160]:
# Step 13.31 — Connect Imports to File

imports = canonical_dataframes["imports_df"]

import_file_rows = (
    imports[["import_id", "file_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (f:File {file_id: row.file_id})
MATCH (i:Import {import_id: row.import_id})
MERGE (f)-[:HAS_IMPORT]->(i)
RETURN count(i) AS imports_connected
"""

result = run_cypher_query(
    query,
    {"rows": import_file_rows}
)

display(pd.DataFrame(result))

,imports_connected
0,319


In [161]:
# Step 13.32 — Ingest ImportBinding nodes

bindings = canonical_dataframes["import_bindings_df"]

binding_columns = [
    "import_binding_id",
    "local_name",
    "imported_name",
    "binding_type",
    "binding_text",
]

binding_rows = (
    bindings[binding_columns]
    .where(pd.notna(bindings[binding_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (b:ImportBinding {
    import_binding_id: row.import_binding_id
})
SET b.local_name = row.local_name,
    b.imported_name = row.imported_name,
    b.binding_type = row.binding_type,
    b.binding_text = row.binding_text
RETURN count(b) AS bindings_created_or_matched
"""

result = run_cypher_query(query, {"rows": binding_rows})

display(pd.DataFrame(result))

,bindings_created_or_matched
0,289


In [162]:
# Step 13.33 — Connect ImportBindings to Imports

bindings = canonical_dataframes["import_bindings_df"]

binding_relationship_rows = (
    bindings[["import_binding_id", "import_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (i:Import {import_id: row.import_id})
MATCH (b:ImportBinding {
    import_binding_id: row.import_binding_id
})
MERGE (i)-[:HAS_BINDING]->(b)
RETURN count(b) AS bindings_connected
"""

result = run_cypher_query(
    query,
    {"rows": binding_relationship_rows}
)

display(pd.DataFrame(result))

,bindings_connected
0,289


In [163]:
# Step 13.34 — Validate Import graph integrity

result = run_cypher_query("""
MATCH (i:Import)
OPTIONAL MATCH (f:File)-[:HAS_IMPORT]->(i)
WITH i, count(DISTINCT f) AS file_count

OPTIONAL MATCH (i)-[:HAS_BINDING]->(b:ImportBinding)
WITH i, file_count, count(DISTINCT b) AS binding_count

RETURN
    count(i) AS total_imports,
    sum(CASE WHEN file_count = 0 THEN 1 ELSE 0 END)
        AS imports_without_file,
    sum(CASE WHEN file_count > 1 THEN 1 ELSE 0 END)
        AS imports_with_multiple_files,
    sum(CASE WHEN binding_count = 0 THEN 1 ELSE 0 END)
        AS imports_without_bindings,
    sum(binding_count) AS total_bindings
""")

display(pd.DataFrame(result))

,total_imports,imports_without_file,imports_with_multiple_files,imports_without_bindings,total_bindings
0,319,0,0,60,289


In [164]:
# Step 13.35 — Inspect Export structures

for name in ["exports_df", "export_bindings_df"]:
    df = canonical_dataframes[name]

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head(3))


exports_df
Rows: 225
Columns: ['export_id', 'file_id', 'export_text', 'export_kind', 'export_system', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,export_id,file_id,export_text,export_kind,export_system,start_line,start_column,end_line,end_column,start_byte,end_byte
0,exp_109af1fe96eb42b7,file_ebe5fee72d4d11cc,export class ZoteroWebTranslationEnvironment e...,es_module,es_module,3,1,77,2,131,2267
1,exp_c7b424b7751b6abe,file_0281e7f33a39c986,export class BlockingObserver {\n\tshouldBlock...,es_module,es_module,30,1,88,2,1038,2457
2,exp_be267970a819465d,file_fa9609b79c312f0f,export class EPUB {\n\t_zipReader;\n\n\t_conte...,es_module,es_module,39,1,190,2,1244,5679



export_bindings_df
Rows: 269
Columns: ['export_binding_id', 'export_id', 'file_id', 'local_name', 'exported_name', 'binding_type', 'binding_text']


,export_binding_id,export_id,file_id,local_name,exported_name,binding_type,binding_text
0,eb_9a30e35842a1f24e,exp_109af1fe96eb42b7,file_ebe5fee72d4d11cc,ZoteroWebTranslationEnvironment,ZoteroWebTranslationEnvironment,declaration,class ZoteroWebTranslationEnvironment extends ...
1,eb_bed3f8b4450f4b76,exp_c7b424b7751b6abe,file_0281e7f33a39c986,BlockingObserver,BlockingObserver,declaration,class BlockingObserver {\n\tshouldBlock;\n\t\n...
2,eb_67963ec6e6105121,exp_be267970a819465d,file_fa9609b79c312f0f,EPUB,EPUB,declaration,class EPUB {\n\t_zipReader;\n\n\t_contentOPF =...


In [165]:
# Step 13.36 — Create Export uniqueness constraints

constraints = [
    """
    CREATE CONSTRAINT export_id_unique IF NOT EXISTS
    FOR (e:Export)
    REQUIRE e.export_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT export_binding_id_unique IF NOT EXISTS
    FOR (b:ExportBinding)
    REQUIRE b.export_binding_id IS UNIQUE
    """
]

for query in constraints:
    run_cypher_query(query)

print("Export and ExportBinding uniqueness constraints created.")

Export and ExportBinding uniqueness constraints created.


In [166]:
# Step 13.37 — Ingest Export nodes

exports = canonical_dataframes["exports_df"]

export_columns = [
    "export_id",
    "export_text",
    "export_kind",
    "export_system",
    "start_line",
    "start_column",
    "end_line",
    "end_column",
    "start_byte",
    "end_byte",
]

export_rows = (
    exports[export_columns]
    .where(pd.notna(exports[export_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (e:Export {export_id: row.export_id})
SET e.export_text = row.export_text,
    e.export_kind = row.export_kind,
    e.export_system = row.export_system,
    e.start_line = row.start_line,
    e.start_column = row.start_column,
    e.end_line = row.end_line,
    e.end_column = row.end_column,
    e.start_byte = row.start_byte,
    e.end_byte = row.end_byte
RETURN count(e) AS exports_created_or_matched
"""

result = run_cypher_query(query, {"rows": export_rows})
display(pd.DataFrame(result))

,exports_created_or_matched
0,225


In [167]:
# Step 13.38 — Connect Export nodes to Files

exports = canonical_dataframes["exports_df"]

export_file_rows = (
    exports[["export_id", "file_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (f:File {file_id: row.file_id})
MATCH (e:Export {export_id: row.export_id})
MERGE (f)-[:HAS_EXPORT]->(e)
RETURN count(e) AS exports_connected
"""

result = run_cypher_query(query, {"rows": export_file_rows})
display(pd.DataFrame(result))

,exports_connected
0,225


In [168]:
# Step 13.39 — Ingest ExportBinding nodes

export_bindings = canonical_dataframes["export_bindings_df"]

binding_columns = [
    "export_binding_id",
    "local_name",
    "exported_name",
    "binding_type",
    "binding_text"
]

binding_rows = (
    export_bindings[binding_columns]
    .where(pd.notna(export_bindings[binding_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (b:ExportBinding {
    export_binding_id: row.export_binding_id
})
SET b.local_name = row.local_name,
    b.exported_name = row.exported_name,
    b.binding_type = row.binding_type,
    b.binding_text = row.binding_text
RETURN count(b) AS bindings_created_or_matched
"""

result = run_cypher_query(query, {"rows": binding_rows})
display(pd.DataFrame(result))

,bindings_created_or_matched
0,269


In [169]:
# Step 13.40 — Connect ExportBinding nodes to Exports

export_bindings = canonical_dataframes["export_bindings_df"]

binding_export_rows = (
    export_bindings[["export_id", "export_binding_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (e:Export {export_id: row.export_id})
MATCH (b:ExportBinding {
    export_binding_id: row.export_binding_id
})
MERGE (e)-[:HAS_BINDING]->(b)
RETURN count(b) AS bindings_connected
"""

result = run_cypher_query(query, {"rows": binding_export_rows})
display(pd.DataFrame(result))

,bindings_connected
0,269


In [171]:
# Step 13.41 — Validate Export graph integrity

query = """
CALL {
    MATCH (e:Export)
    OPTIONAL MATCH (f:File)-[:HAS_EXPORT]->(e)
    RETURN
        count(DISTINCT e) AS total_exports,
        count(DISTINCT CASE WHEN f IS NULL THEN e END) AS exports_without_file,
        count(DISTINCT CASE WHEN f IS NOT NULL THEN e END) AS exports_with_file
}
CALL {
    MATCH (e:Export)-[:HAS_BINDING]->(b:ExportBinding)
    RETURN count(DISTINCT b) AS connected_bindings
}
CALL {
    MATCH (b:ExportBinding)
    OPTIONAL MATCH (e:Export)-[:HAS_BINDING]->(b)
    RETURN
        count(DISTINCT b) AS total_bindings,
        count(DISTINCT CASE WHEN e IS NULL THEN b END) AS bindings_without_export
}
RETURN *
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL {\n    MATCH (e:Export)\n    OPTIONAL MATCH (f:File)-[:HAS_EXPORT]->(e)\n    RETURN\n        count(DISTINCT e) AS total_exports,\n        count(DISTINCT CASE WHEN f IS NULL THEN e END) AS exports_without_file,\n        count(DISTINCT CASE WHEN f IS NOT NULL THEN e END) AS exports_with_file\n}\nCALL {\n    MATCH (e:Export)-[:HAS_BINDING]->

,bindings_without_export,connected_bindings,exports_with_file,exports_without_file,total_bindings,total_exports
0,0,269,225,0,269,225


In [172]:
# tep 13.42A — Reference ingestion.

references = canonical_dataframes["references_df"]

print(f"Rows: {len(references):,}")
print(f"Columns: {references.columns.tolist()}")

display(references.head(5))

Rows: 254,402
Columns: ['reference_id', 'file_id', 'name', 'reference_role', 'scope_id', 'resolved_symbol_id', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,reference_id,file_id,name,reference_role,scope_id,resolved_symbol_id,start_line,start_column,end_line,end_column,start_byte,end_byte
0,ref_6a0ce5fabc0d788e,file_611738591214a13f,ChromeUtils,call_callee,None,None,1,43,1,54,42,53
1,ref_f274b1f8b0893350,file_611738591214a13f,cmdLine,reference,None,None,5,5,5,12,262,269
2,ref_b1d050d716f004e8,file_611738591214a13f,Ci,reference,None,None,5,22,5,24,279,281
3,ref_9d9636f34aed2aa1,file_611738591214a13f,cmdLine,assignment_target,None,None,6,2,6,9,322,329
4,ref_80f5d9a64f1c4b0b,file_611738591214a13f,cmdLine,call_callee,None,None,10,5,10,12,392,399


In [173]:
# Step 13.42B — Create Reference uniqueness constraint

query = """
CREATE CONSTRAINT reference_id_unique IF NOT EXISTS
FOR (r:Reference)
REQUIRE r.reference_id IS UNIQUE
"""

run_cypher_query(query)

print("Reference uniqueness constraint created.")

Reference uniqueness constraint created.


In [174]:
# Step 13.43 — Ingest Reference nodes

references = canonical_dataframes["references_df"]

reference_columns = [
    "reference_id",
    "name",
    "reference_role",
    "start_line",
    "start_column",
    "end_line",
    "end_column",
    "start_byte",
    "end_byte"
]

reference_rows = (
    references[reference_columns]
    .where(pd.notna(references[reference_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (r:Reference {reference_id: row.reference_id})
SET r.name = row.name,
    r.reference_role = row.reference_role,
    r.start_line = row.start_line,
    r.start_column = row.start_column,
    r.end_line = row.end_line,
    r.end_column = row.end_column,
    r.start_byte = row.start_byte,
    r.end_byte = row.end_byte
RETURN count(r) AS references_created_or_matched
"""

batch_size = 10_000
total = 0

for start in range(0, len(reference_rows), batch_size):
    batch = reference_rows[start:start + batch_size]
    result = run_cypher_query(query, {"rows": batch})
    total += result[0]["references_created_or_matched"]

print(f"References created or matched: {total:,}")

References created or matched: 254,402


In [175]:
# Step 13.44 — Connect References to Files

references = canonical_dataframes["references_df"]

reference_file_rows = (
    references[["reference_id", "file_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (f:File {file_id: row.file_id})
MATCH (r:Reference {reference_id: row.reference_id})
MERGE (f)-[:HAS_REFERENCE]->(r)
RETURN count(r) AS references_connected
"""

batch_size = 10_000
total = 0

for start in range(0, len(reference_file_rows), batch_size):
    batch = reference_file_rows[start:start + batch_size]
    result = run_cypher_query(query, {"rows": batch})
    total += result[0]["references_connected"]

print(f"References connected: {total:,}")

References connected: 254,402


In [176]:
# Step 13.45 — Validate Reference graph integrity

query = """
MATCH (r:Reference)
OPTIONAL MATCH (f:File)-[:HAS_REFERENCE]->(r)
WITH r, count(DISTINCT f) AS file_count
RETURN
    count(r) AS total_references,
    sum(CASE WHEN file_count = 0 THEN 1 ELSE 0 END) AS references_without_file,
    sum(CASE WHEN file_count > 1 THEN 1 ELSE 0 END) AS references_with_multiple_files
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,total_references,references_without_file,references_with_multiple_files
0,254402,0,0


In [177]:
# Step 13.46 — Inspect Assignments

assignments = canonical_dataframes["assignments_df"]

print(f"Rows: {len(assignments):,}")
print(f"Columns: {assignments.columns.tolist()}")

display(assignments.head(5))

Rows: 72,972
Columns: ['assignment_id', 'file_id', 'assignment_kind', 'target_expression', 'value_expression', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,assignment_id,file_id,assignment_kind,target_expression,value_expression,start_line,start_column,end_line,end_column,start_byte,end_byte
0,asg_688a35739fce99cc,file_611738591214a13f,variable_declarator,"{ CommandLineOptions, TestOptions }","ChromeUtils.importESModule(""chrome://zotero/co...",1,5,1,127,4,126
1,asg_9d9636f34aed2aa1,file_611738591214a13f,assignment_expression,cmdLine.preventDefault,true,6,2,6,31,322,351
2,asg_878bff57d642f3f6,file_611738591214a13f,assignment_expression,CommandLineOptions.forceDebugLog,2,11,2,11,38,437,473
3,asg_348ffe0ecd591af6,file_611738591214a13f,assignment_expression,CommandLineOptions.forceDebugLog,1,15,2,15,38,573,609
4,asg_559ddbdd7eb6dbd6,file_611738591214a13f,assignment_expression,CommandLineOptions.forceDataDir,"cmdLine.handleFlagWithParam(""datadir"", false)",24,1,24,80,1016,1095


In [178]:
# Step 13.47 — Create Assignment uniqueness constraint

query = """
CREATE CONSTRAINT assignment_id_unique IF NOT EXISTS
FOR (a:Assignment)
REQUIRE a.assignment_id IS UNIQUE
"""

run_cypher_query(query)

print("Assignment uniqueness constraint created.")

Assignment uniqueness constraint created.


In [179]:
# Step 13.48 — Ingest Assignment nodes

assignments = canonical_dataframes["assignments_df"]

assignment_columns = [
    "assignment_id",
    "assignment_kind",
    "target_expression",
    "value_expression",
    "start_line",
    "start_column",
    "end_line",
    "end_column",
    "start_byte",
    "end_byte"
]

assignment_rows = (
    assignments[assignment_columns]
    .where(pd.notna(assignments[assignment_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (a:Assignment {assignment_id: row.assignment_id})
SET a.assignment_kind = row.assignment_kind,
    a.target_expression = row.target_expression,
    a.value_expression = row.value_expression,
    a.start_line = row.start_line,
    a.start_column = row.start_column,
    a.end_line = row.end_line,
    a.end_column = row.end_column,
    a.start_byte = row.start_byte,
    a.end_byte = row.end_byte
RETURN count(a) AS assignments_created_or_matched
"""

batch_size = 10_000
total = 0

for start in range(0, len(assignment_rows), batch_size):
    batch = assignment_rows[start:start + batch_size]
    result = run_cypher_query(query, {"rows": batch})
    total += result[0]["assignments_created_or_matched"]

print(f"Assignments created or matched: {total:,}")

Assignments created or matched: 72,972


In [180]:
# Step 13.49 — Connect Assignments to Files

assignments = canonical_dataframes["assignments_df"]

assignment_file_rows = (
    assignments[["assignment_id", "file_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (f:File {file_id: row.file_id})
MATCH (a:Assignment {assignment_id: row.assignment_id})
MERGE (f)-[:HAS_ASSIGNMENT]->(a)
RETURN count(a) AS assignments_connected
"""

batch_size = 10_000
total = 0

for start in range(0, len(assignment_file_rows), batch_size):
    batch = assignment_file_rows[start:start + batch_size]
    result = run_cypher_query(query, {"rows": batch})
    total += result[0]["assignments_connected"]

print(f"Assignments connected: {total:,}")

Assignments connected: 72,972


In [181]:
# Step 13.50 — Validate Assignment graph integrity

query = """
MATCH (a:Assignment)
OPTIONAL MATCH (f:File)-[:HAS_ASSIGNMENT]->(a)
WITH a, count(DISTINCT f) AS file_count
RETURN
    count(a) AS total_assignments,
    sum(CASE WHEN file_count = 0 THEN 1 ELSE 0 END) AS assignments_without_file,
    sum(CASE WHEN file_count > 1 THEN 1 ELSE 0 END) AS assignments_with_multiple_files
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,total_assignments,assignments_without_file,assignments_with_multiple_files
0,72972,0,0


In [182]:
# Step 13.51 — Inspect Function Calls

calls = canonical_dataframes["calls_df"]

print(f"Rows: {len(calls):,}")
print(f"Columns: {calls.columns.tolist()}")

display(calls.head(5))

Rows: 108,839
Columns: ['call_id', 'file_id', 'call_kind', 'call_node_type', 'callee_expression', 'call_text', 'arguments_text', 'argument_count', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,call_id,file_id,call_kind,call_node_type,callee_expression,call_text,arguments_text,argument_count,start_line,start_column,end_line,end_column,start_byte,end_byte
0,call_7ad8d0433055c936,file_611738591214a13f,function_call,call_expression,ChromeUtils.importESModule,"ChromeUtils.importESModule(""chrome://zotero/co...","(""chrome://zotero/content/modules/commandLineO...",1,1,43,1,127,42,126
1,call_6c88a4a466caa540,file_611738591214a13f,function_call,call_expression,cmdLine.handleFlag,"cmdLine.handleFlag(""ZoteroDebug"", false)","(""ZoteroDebug"", false)",2,10,5,10,45,392,432
2,call_85b7a36ee0731bc3,file_611738591214a13f,function_call,call_expression,cmdLine.handleFlag,"cmdLine.handleFlag(""ZoteroDebugText"", false)","(""ZoteroDebugText"", false)",2,14,10,14,54,524,568
3,call_1923a9daaf3b6815,file_611738591214a13f,function_call,call_expression,"Services.prefs.getBranch(""toolkit.startup."").c...","Services.prefs.getBranch(""toolkit.startup."").c...","(""recent_crashes"")",1,21,2,21,78,935,1011
4,call_fe610d8b78929d54,file_611738591214a13f,function_call,call_expression,Services.prefs.getBranch,"Services.prefs.getBranch(""toolkit.startup."")","(""toolkit.startup."")",1,21,2,21,46,935,979


In [184]:
# Step 13.52 — Create CallSite uniqueness constraint

query = """
CREATE CONSTRAINT call_id_unique IF NOT EXISTS
FOR (c:CallSite)
REQUIRE c.call_id IS UNIQUE
"""

run_cypher_query(query)

print("CallSite uniqueness constraint created.")

CallSite uniqueness constraint created.


In [185]:
# Step 13.53 — Ingest CallSite nodes

calls = canonical_dataframes["calls_df"]

call_columns = [
    "call_id",
    "call_kind",
    "call_node_type",
    "callee_expression",
    "call_text",
    "arguments_text",
    "argument_count",
    "start_line",
    "start_column",
    "end_line",
    "end_column",
    "start_byte",
    "end_byte"
]

call_rows = (
    calls[call_columns]
    .where(pd.notna(calls[call_columns]), None)
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MERGE (c:CallSite {call_id: row.call_id})
SET c.call_kind = row.call_kind,
    c.call_node_type = row.call_node_type,
    c.callee_expression = row.callee_expression,
    c.call_text = row.call_text,
    c.arguments_text = row.arguments_text,
    c.argument_count = row.argument_count,
    c.start_line = row.start_line,
    c.start_column = row.start_column,
    c.end_line = row.end_line,
    c.end_column = row.end_column,
    c.start_byte = row.start_byte,
    c.end_byte = row.end_byte
RETURN count(c) AS calls_created_or_matched
"""

batch_size = 10_000
total = 0

for start in range(0, len(call_rows), batch_size):
    batch = call_rows[start:start + batch_size]
    result = run_cypher_query(query, {"rows": batch})
    total += result[0]["calls_created_or_matched"]

print(f"CallSites created or matched: {total:,}")

CallSites created or matched: 108,839


In [186]:
# Step 13.54 — Connect CallSites to Files

calls = canonical_dataframes["calls_df"]

call_file_rows = (
    calls[["call_id", "file_id"]]
    .to_dict("records")
)

query = """
UNWIND $rows AS row
MATCH (f:File {file_id: row.file_id})
MATCH (c:CallSite {call_id: row.call_id})
MERGE (f)-[:HAS_CALL]->(c)
RETURN count(c) AS calls_connected
"""

batch_size = 10_000
total = 0

for start in range(0, len(call_file_rows), batch_size):
    batch = call_file_rows[start:start + batch_size]
    result = run_cypher_query(query, {"rows": batch})
    total += result[0]["calls_connected"]

print(f"CallSites connected: {total:,}")

CallSites connected: 108,839


In [187]:
# Step 13.55 — Validate CallSite graph integrity

query = """
MATCH (c:CallSite)
OPTIONAL MATCH (f:File)-[:HAS_CALL]->(c)
WITH c, count(DISTINCT f) AS file_count
RETURN
    count(c) AS total_calls,
    sum(CASE WHEN file_count = 0 THEN 1 ELSE 0 END) AS calls_without_file,
    sum(CASE WHEN file_count > 1 THEN 1 ELSE 0 END) AS calls_with_multiple_files
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,total_calls,calls_without_file,calls_with_multiple_files
0,108839,0,0


In [188]:
# Step 13.56 — Inspect Classes

classes = canonical_dataframes["classes_df"]

print(f"Rows: {len(classes):,}")
print(f"Columns: {classes.columns.tolist()}")

display(classes.head(5))

Rows: 384
Columns: ['class_id', 'file_id', 'class_name', 'class_node_type', 'heritage_text', 'base_expression', 'source_text', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,class_id,file_id,class_name,class_node_type,heritage_text,base_expression,source_text,start_line,start_column,end_line,end_column,start_byte,end_byte
0,class_1cf2cf1de88c1454,file_18307e89f1ddb102,ScaffoldItemPreview,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreview extends XULElementBa...,31,2,216,3,980,6363
1,class_3aa029c8f7074984,file_18307e89f1ddb102,NaN,class,NaN,NaN,class,31,2,31,7,980,985
2,class_dbe41342a3a7eae8,file_a6b19f7562edaee3,ScaffoldItemPreviews,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreviews extends XULElementB...,29,2,143,3,866,3844
3,class_c533bfdbbf0ce5f1,file_a6b19f7562edaee3,NaN,class,NaN,NaN,class,29,2,29,7,866,871
4,class_de7620c0b29c9e32,file_ebe5fee72d4d11cc,ZoteroWebTranslationEnvironment,class_declaration,extends AbstractWebTranslationEnvironment,AbstractWebTranslationEnvironment,class ZoteroWebTranslationEnvironment extends ...,3,8,77,2,138,2267


In [189]:
# Step 13.57 — Reconcile Classes with Symbols

classes = canonical_dataframes["classes_df"]
symbols = canonical_dataframes["symbols_df"]

print("CLASS NODE TYPES")
display(classes["class_node_type"].value_counts(dropna=False).to_frame("count"))

print("CLASS NAME AVAILABILITY")
display(
    classes["class_name"]
    .isna()
    .value_counts()
    .rename(index={True: "Missing", False: "Named"})
    .to_frame("count")
)

print("SYMBOL TYPES")
display(symbols["symbol_type"].value_counts(dropna=False).to_frame("count"))

print("CLASS RECORDS SHARING SOURCE LOCATIONS")
display(
    classes.groupby(
        ["file_id", "start_byte", "end_byte"]
    ).size().value_counts().sort_index().to_frame("location_groups")
)

CLASS NODE TYPES


,count
class_node_type,
class,237
class_declaration,147


CLASS NAME AVAILABILITY


,count
class_name,
Missing,231
Named,153


SYMBOL TYPES


,count
symbol_type,
variable,40539
function_expression,13940
arrow_function,4527
method,2692
function,1377
class,153


CLASS RECORDS SHARING SOURCE LOCATIONS


,location_groups
1,384


In [190]:
# Step 13.58 — Match named Classes to Symbols

named_classes = classes[classes["class_name"].notna()].copy()

class_symbols = symbols[
    symbols["symbol_type"] == "class"
].copy()

matches = named_classes.merge(
    class_symbols,
    left_on=["file_id", "class_name"],
    right_on=["file_id", "name"],
    how="left",
    suffixes=("_class", "_symbol")
)

print(f"Named class records: {len(named_classes)}")
print(f"Class symbols: {len(class_symbols)}")
print(f"Matched by file and name: {matches['symbol_id'].notna().sum()}")
print(f"Unmatched class records: {matches['symbol_id'].isna().sum()}")

matches["location_match"] = (
    (matches["start_byte_class"] == matches["start_byte_symbol"]) &
    (matches["end_byte_class"] == matches["end_byte_symbol"])
)

print(f"Exact source location matches: {matches['location_match'].sum()}")

display(
    matches[
        ["class_name", "file_id", "symbol_id",
         "start_byte_class", "end_byte_class",
         "start_byte_symbol", "end_byte_symbol",
         "location_match"]
    ].head(10)
)

Named class records: 153
Class symbols: 153
Matched by file and name: 153
Unmatched class records: 0
Exact source location matches: 153


,class_name,file_id,symbol_id,start_byte_class,end_byte_class,start_byte_symbol,end_byte_symbol,location_match
0,ScaffoldItemPreview,file_18307e89f1ddb102,sym_525951bac32ed528,980,6363,980,6363,True
1,ScaffoldItemPreviews,file_a6b19f7562edaee3,sym_fba95d24baa694f1,866,3844,866,3844,True
2,ZoteroWebTranslationEnvironment,file_ebe5fee72d4d11cc,sym_32f6629ed96850fa,138,2267,138,2267,True
3,BlockingObserver,file_0281e7f33a39c986,sym_6cd93fc5c9415a19,1045,2457,1045,2457,True
4,EPUB,file_fa9609b79c312f0f,sym_71512dfe6fec13e3,1251,5679,1251,5679,True
5,HiddenBrowser,file_50524eedfc834457,sym_872e7ab7b2474f3c,1590,11122,1590,11122,True
6,RemoteTranslate,file_02896191d5d2433e,sym_b7985edabd18320d,1236,8640,1236,8640,True
7,DocumentIsReadyChild,file_a4c81444f3cabedc,sym_c88d2a74805ecd91,89,353,89,353,True
8,ExternalLinkHandlerChild,file_d1d72f82a6caef82,sym_558875849eb36265,7,1698,7,1698,True
9,ExternalLinkHandlerParent,file_af9e6a91db409e8e,sym_7463a357c9050ca7,106,302,106,302,True


In [191]:
# Step 13.59 — Inspect anonymous Class records

anonymous_classes = classes[classes["class_name"].isna()].copy()

print(f"Anonymous class records: {len(anonymous_classes)}")

display(
    anonymous_classes[
        [
            "class_id",
            "file_id",
            "class_node_type",
            "heritage_text",
            "base_expression",
            "source_text",
            "start_line",
            "end_line",
            "start_byte",
            "end_byte"
        ]
    ].head(10)
)

Anonymous class records: 231


,class_id,file_id,class_node_type,heritage_text,base_expression,source_text,start_line,end_line,start_byte,end_byte
1,class_3aa029c8f7074984,file_18307e89f1ddb102,class,NaN,NaN,class,31,31,980,985
3,class_c533bfdbbf0ce5f1,file_a6b19f7562edaee3,class,NaN,NaN,class,29,29,866,871
5,class_851f1a6470c52603,file_ebe5fee72d4d11cc,class,NaN,NaN,class,3,3,138,143
7,class_3c4b690c23b2a117,file_0281e7f33a39c986,class,None,None,class,30,30,1045,1050
9,class_c1920bd61ee6c839,file_fa9609b79c312f0f,class,None,None,class,39,39,1251,1256
11,class_6ca8d7b9e2b935c7,file_50524eedfc834457,class,None,None,class,47,47,1590,1595
13,class_77ba4b098d8a4245,file_02896191d5d2433e,class,None,None,class,33,33,1236,1241
15,class_6f72b6e6b05a06b7,file_a4c81444f3cabedc,class,NaN,NaN,class,3,3,89,94
17,class_8b26a8ed12ec5e99,file_d1d72f82a6caef82,class,NaN,NaN,class,1,1,7,12
19,class_a1dfbc8acf25e493,file_af9e6a91db409e8e,class,NaN,NaN,class,5,5,106,111


In [193]:
# Step 13.60 — Check overlapping class records

named_classes = classes[classes["class_name"].notna()].copy()
anonymous_classes = classes[classes["class_name"].isna()].copy()

overlap = anonymous_classes.merge(
    named_classes,
    on=["file_id", "start_byte"],
    how="inner",
    suffixes=("_anonymous", "_named")
)

print(f"Anonymous records: {len(anonymous_classes)}")
print(f"Overlapping named class records: {len(overlap)}")
print(f"Non-overlapping anonymous records: {len(anonymous_classes) - len(overlap)}")

display(
    overlap[
        [
            "file_id",
            "start_byte",
            "end_byte_anonymous",
            "end_byte_named",
            "source_text_anonymous",
            "class_name",
            "source_text_named"
        ]
    ].head(10)
)

Anonymous records: 231
Overlapping named class records: 153
Non-overlapping anonymous records: 78


KeyError: "['class_name'] not in index"

In [194]:
print("Overlap columns:")
print(overlap.columns.tolist())

display(overlap.head(5))

Overlap columns:
['class_id_anonymous', 'file_id', 'class_name_anonymous', 'class_node_type_anonymous', 'heritage_text_anonymous', 'base_expression_anonymous', 'source_text_anonymous', 'start_line_anonymous', 'start_column_anonymous', 'end_line_anonymous', 'end_column_anonymous', 'start_byte', 'end_byte_anonymous', 'class_id_named', 'class_name_named', 'class_node_type_named', 'heritage_text_named', 'base_expression_named', 'source_text_named', 'start_line_named', 'start_column_named', 'end_line_named', 'end_column_named', 'end_byte_named']


,class_id_anonymous,file_id,class_name_anonymous,class_node_type_anonymous,heritage_text_anonymous,base_expression_anonymous,source_text_anonymous,start_line_anonymous,start_column_anonymous,end_line_anonymous,...,class_name_named,class_node_type_named,heritage_text_named,base_expression_named,source_text_named,start_line_named,start_column_named,end_line_named,end_column_named,end_byte_named
0,class_3aa029c8f7074984,file_18307e89f1ddb102,NaN,class,NaN,NaN,class,31,2,31,...,ScaffoldItemPreview,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreview extends XULElementBa...,31,2,216,3,6363
1,class_c533bfdbbf0ce5f1,file_a6b19f7562edaee3,NaN,class,NaN,NaN,class,29,2,29,...,ScaffoldItemPreviews,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreviews extends XULElementB...,29,2,143,3,3844
2,class_851f1a6470c52603,file_ebe5fee72d4d11cc,NaN,class,NaN,NaN,class,3,8,3,...,ZoteroWebTranslationEnvironment,class_declaration,extends AbstractWebTranslationEnvironment,AbstractWebTranslationEnvironment,class ZoteroWebTranslationEnvironment extends ...,3,8,77,2,2267
3,class_3c4b690c23b2a117,file_0281e7f33a39c986,NaN,class,None,None,class,30,8,30,...,BlockingObserver,class_declaration,None,None,class BlockingObserver {\n\tshouldBlock;\n\t\n...,30,8,88,2,2457
4,class_c1920bd61ee6c839,file_fa9609b79c312f0f,NaN,class,None,None,class,39,8,39,...,EPUB,class_declaration,None,None,class EPUB {\n\t_zipReader;\n\n\t_contentOPF =...,39,8,190,2,5679


In [195]:
# Step 13.60 — Inspect overlapping records

print(f"Anonymous records: {len(anonymous_classes)}")
print(f"Overlapping named class records: {len(overlap)}")
print(f"Non-overlapping anonymous records: {len(anonymous_classes) - len(overlap)}")

display(
    overlap[
        [
            "file_id",
            "start_byte",
            "end_byte_anonymous",
            "end_byte_named",
            "source_text_anonymous",
            "class_name_named",
            "source_text_named"
        ]
    ].head(10)
)

Anonymous records: 231
Overlapping named class records: 153
Non-overlapping anonymous records: 78


,file_id,start_byte,end_byte_anonymous,end_byte_named,source_text_anonymous,class_name_named,source_text_named
0,file_18307e89f1ddb102,980,985,6363,class,ScaffoldItemPreview,class ScaffoldItemPreview extends XULElementBa...
1,file_a6b19f7562edaee3,866,871,3844,class,ScaffoldItemPreviews,class ScaffoldItemPreviews extends XULElementB...
2,file_ebe5fee72d4d11cc,138,143,2267,class,ZoteroWebTranslationEnvironment,class ZoteroWebTranslationEnvironment extends ...
3,file_0281e7f33a39c986,1045,1050,2457,class,BlockingObserver,class BlockingObserver {\n\tshouldBlock;\n\t\n...
4,file_fa9609b79c312f0f,1251,1256,5679,class,EPUB,class EPUB {\n\t_zipReader;\n\n\t_contentOPF =...
5,file_50524eedfc834457,1590,1595,11122,class,HiddenBrowser,class HiddenBrowser {\n\t/**\n\t * @param {Obj...
6,file_02896191d5d2433e,1236,1241,8640,class,RemoteTranslate,class RemoteTranslate {\n\t_browser = null;\n\...
7,file_a4c81444f3cabedc,89,94,353,class,DocumentIsReadyChild,class DocumentIsReadyChild extends JSWindowAct...
8,file_d1d72f82a6caef82,7,12,1698,class,ExternalLinkHandlerChild,class ExternalLinkHandlerChild extends JSWindo...
9,file_af9e6a91db409e8e,106,111,302,class,ExternalLinkHandlerParent,class ExternalLinkHandlerParent extends JSWind...


In [196]:
# Step 13.61 — Inspect the remaining 78 records

overlapping_ids = set(overlap["class_id_anonymous"])

remaining_anonymous = anonymous_classes[
    ~anonymous_classes["class_id"].isin(overlapping_ids)
].copy()

print(f"Remaining records: {len(remaining_anonymous)}")

display(
    remaining_anonymous[
        [
            "class_id",
            "file_id",
            "class_node_type",
            "class_name",
            "heritage_text",
            "base_expression",
            "source_text",
            "start_line",
            "end_line",
            "start_byte",
            "end_byte"
        ]
    ].head(15)
)

Remaining records: 78


,class_id,file_id,class_node_type,class_name,heritage_text,base_expression,source_text,start_line,end_line,start_byte,end_byte
42,class_b37c99c1d6597118,file_4557095a55e65e15,class,NaN,NaN,NaN,class {\n\t_registeredRemoteTranslates = new M...,3,70,101,1871
43,class_00dbbb189da583f6,file_4557095a55e65e15,class,NaN,NaN,NaN,class,3,3,101,106
84,class_024af776c0636f67,file_555966c236571b4d,class,NaN,NaN,NaN,class {\n\tconstructor(virtualizedTable) {\n\t...,1906,2184,62092,71894
85,class_f42c885212e837a9,file_555966c236571b4d,class,NaN,NaN,NaN,class,1906,1906,62092,62097
86,class_cc0f9b4b901f3ed8,file_72d2cdb95acab69d,class,None,None,None,class {\n\t/**\n\t * @param options (required)...,42,337,1631,11305
87,class_69df2f911b926f98,file_72d2cdb95acab69d,class,None,None,None,class,42,42,1631,1636
116,class_60148e2c16792c63,file_b9a538ee02f4f3ab,class,None,extends Class,Class,class extends Class {\n\t\tinitialized = false...,31,82,1082,2161
117,class_9ca1774eb97c96aa,file_b9a538ee02f4f3ab,class,None,NaN,NaN,class,31,31,1082,1087
282,class_910d535dd4da10fc,file_096847be2a24c649,class,NaN,None,None,"class {\n\t\t\tconstructor(citationField, data...",1432,1477,43340,44475
283,class_645d04eba5fdd236,file_096847be2a24c649,class,NaN,None,None,class,1432,1432,43340,43345


In [197]:
# Step 13.62 — Verify the paired records

remaining_anonymous["source_length"] = (
    remaining_anonymous["end_byte"]
    - remaining_anonymous["start_byte"]
)

print(
    remaining_anonymous["source_length"]
    .value_counts()
    .sort_index()
)

print(
    "\nKeyword-only records:",
    (remaining_anonymous["source_text"] == "class").sum()
)

print(
    "Full-expression records:",
    (remaining_anonymous["source_text"] != "class").sum()
)

source_length
5       39
119      1
131      1
143      1
196      1
209      1
298      1
314      1
320      1
324      1
560      1
568      1
571      2
575      1
591      1
602      1
780      1
814      1
842      1
1002     1
1017     1
1064     1
1079     1
1135     1
1157     1
1202     1
1311     1
1566     1
1690     1
1770     1
1923     1
3250     1
4349     1
4796     1
6472     1
7592     1
9674     1
9797     1
9802     1
Name: count, dtype: int64

Keyword-only records: 39
Full-expression records: 39


In [198]:
# Step 13.63 — Correct the extraction DataFrame

classes_clean_df = classes_df[
    classes_df["source_text"].str.strip().ne("class")
].copy()

print(f"Original class records: {len(classes_df)}")
print(f"Clean class records: {len(classes_clean_df)}")
print(
    "Named classes:",
    classes_clean_df["class_name"].notna().sum()
)
print(
    "Anonymous classes:",
    classes_clean_df["class_name"].isna().sum()
)

display(classes_clean_df.head())

Original class records: 384
Clean class records: 192
Named classes: 153
Anonymous classes: 39


,class_id,file_id,class_name,class_node_type,heritage_text,base_expression,source_text,start_line,start_column,end_line,end_column,start_byte,end_byte
0,class_1cf2cf1de88c1454,chrome/content/scaffold/scaffoldItemPreview.js,ScaffoldItemPreview,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreview extends XULElementBa...,31,2,216,3,980,6363
2,class_dbe41342a3a7eae8,chrome/content/scaffold/scaffoldItemPreviews.js,ScaffoldItemPreviews,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreviews extends XULElementB...,29,2,143,3,866,3844
4,class_de7620c0b29c9e32,chrome/content/scaffold/zoteroWebTranslationEn...,ZoteroWebTranslationEnvironment,class_declaration,extends AbstractWebTranslationEnvironment,AbstractWebTranslationEnvironment,class ZoteroWebTranslationEnvironment extends ...,3,8,77,2,138,2267
6,class_05f8eb1b48c356db,chrome/content/zotero/BlockingObserver.mjs,BlockingObserver,class_declaration,None,None,class BlockingObserver {\n\tshouldBlock;\n\t\n...,30,8,88,2,1045,2457
8,class_8627cb3e80d87537,chrome/content/zotero/EPUB.mjs,EPUB,class_declaration,None,None,class EPUB {\n\t_zipReader;\n\n\t_contentOPF =...,39,8,190,2,1251,5679


In [199]:
# Step 13.64 — Prepare canonical class records

classes_clean_canonical_df = classes_clean_df.copy()

classes_clean_canonical_df["file_id"] = (
    classes_clean_canonical_df["file_id"]
    .map(file_id_mapping)
)

print(f"Class records: {len(classes_clean_canonical_df)}")
print(
    "Missing canonical file IDs:",
    classes_clean_canonical_df["file_id"].isna().sum()
)
print(
    "Unique files:",
    classes_clean_canonical_df["file_id"].nunique()
)

display(classes_clean_canonical_df.head())

Class records: 192
Missing canonical file IDs: 0
Unique files: 126


,class_id,file_id,class_name,class_node_type,heritage_text,base_expression,source_text,start_line,start_column,end_line,end_column,start_byte,end_byte
0,class_1cf2cf1de88c1454,file_18307e89f1ddb102,ScaffoldItemPreview,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreview extends XULElementBa...,31,2,216,3,980,6363
2,class_dbe41342a3a7eae8,file_a6b19f7562edaee3,ScaffoldItemPreviews,class_declaration,extends XULElementBase,XULElementBase,class ScaffoldItemPreviews extends XULElementB...,29,2,143,3,866,3844
4,class_de7620c0b29c9e32,file_ebe5fee72d4d11cc,ZoteroWebTranslationEnvironment,class_declaration,extends AbstractWebTranslationEnvironment,AbstractWebTranslationEnvironment,class ZoteroWebTranslationEnvironment extends ...,3,8,77,2,138,2267
6,class_05f8eb1b48c356db,file_0281e7f33a39c986,BlockingObserver,class_declaration,None,None,class BlockingObserver {\n\tshouldBlock;\n\t\n...,30,8,88,2,1045,2457
8,class_8627cb3e80d87537,file_fa9609b79c312f0f,EPUB,class_declaration,None,None,class EPUB {\n\t_zipReader;\n\n\t_contentOPF =...,39,8,190,2,1251,5679


In [200]:
# Step 13.65 — Reconcile named classes with existing Symbols

named_classes_clean = classes_clean_canonical_df[
    classes_clean_canonical_df["class_name"].notna()
].copy()

class_symbol_matches = named_classes_clean.merge(
    canonical_dataframes["symbols_df"][
        canonical_dataframes["symbols_df"]["symbol_type"] == "class"
    ],
    left_on=["file_id", "class_name"],
    right_on=["file_id", "name"],
    how="left",
    suffixes=("_class", "_symbol"),
    indicator=True
)

print(f"Named class records: {len(named_classes_clean)}")
print(
    "Matched to existing Symbols:",
    (class_symbol_matches["_merge"] == "both").sum()
)
print(
    "Unmatched:",
    (class_symbol_matches["_merge"] == "left_only").sum()
)

display(
    class_symbol_matches.loc[
        class_symbol_matches["_merge"] == "left_only",
        ["file_id", "class_name", "class_id"]
    ].head(10)
)

Named class records: 153
Matched to existing Symbols: 153
Unmatched: 0


,file_id,class_name,class_id


In [201]:
# Step 13.66 — Inspect anonymous class records

anonymous_classes_clean = classes_clean_canonical_df[
    classes_clean_canonical_df["class_name"].isna()
].copy()

print(f"Anonymous class expressions: {len(anonymous_classes_clean)}")
print(
    "Distinct files:",
    anonymous_classes_clean["file_id"].nunique()
)

display(
    anonymous_classes_clean[
        [
            "class_id",
            "file_id",
            "class_node_type",
            "heritage_text",
            "base_expression",
            "start_line",
            "end_line",
            "source_text"
        ]
    ].head(15)
)

Anonymous class expressions: 39
Distinct files: 11


,class_id,file_id,class_node_type,heritage_text,base_expression,start_line,end_line,source_text
42,class_b37c99c1d6597118,file_4557095a55e65e15,class,NaN,NaN,3,70,class {\n\t_registeredRemoteTranslates = new M...
84,class_024af776c0636f67,file_555966c236571b4d,class,NaN,NaN,1906,2184,class {\n\tconstructor(virtualizedTable) {\n\t...
86,class_cc0f9b4b901f3ed8,file_72d2cdb95acab69d,class,None,None,42,337,class {\n\t/**\n\t * @param options (required)...
116,class_60148e2c16792c63,file_b9a538ee02f4f3ab,class,extends Class,Class,31,82,class extends Class {\n\t\tinitialized = false...
282,class_910d535dd4da10fc,file_096847be2a24c649,class,None,None,1432,1477,"class {\n\t\t\tconstructor(citationField, data..."
286,class_33d269e1170c91b4,file_622b5ecb6a364c92,class,extends Headers,Headers,1726,1776,class extends Headers {\n\t\t_originalNames = ...
288,class_c4b38c1ad9b410cb,file_ccff9253207adc8c,class,NaN,NaN,2874,2937,"class {\n\tconstructor(field, rawCode) {\n\t\t..."
290,class_c9099f3a89e4bf88,file_ccff9253207adc8c,class,extends Zotero.Integration.Field,Zotero.Integration.Field,2982,3172,class extends Zotero.Integration.Field {\n\tco...
292,class_a233f3ffb20ce853,file_ccff9253207adc8c,class,extends Zotero.Integration.Field,Zotero.Integration.Field,3175,3201,class extends Zotero.Integration.Field {\n\tco...
294,class_0f13d34682b84b2c,file_ccff9253207adc8c,class,NaN,NaN,3203,3471,class {\n\tstatic refreshEmbeddedData(itemData...


In [202]:
# Step 13.67 — Inspect anonymous class metadata

display(
    anonymous_classes_clean[
        [
            "class_id",
            "class_node_type",
            "heritage_text",
            "base_expression",
            "start_line",
            "end_line",
            "start_byte",
            "end_byte"
        ]
    ].to_string(index=False)
)


'              class_id class_node_type                    heritage_text          base_expression  start_line  end_line  start_byte  end_byte\nclass_b37c99c1d6597118           class                              NaN                      NaN           3        70         101      1871\nclass_024af776c0636f67           class                              NaN                      NaN        1906      2184       62092     71894\nclass_cc0f9b4b901f3ed8           class                             None                     None          42       337        1631     11305\nclass_60148e2c16792c63           class                    extends Class                    Class          31        82        1082      2161\nclass_910d535dd4da10fc           class                             None                     None        1432      1477       43340     44475\nclass_33d269e1170c91b4           class                  extends Headers                  Headers        1726      1776       53106     54108\nclass

In [203]:
# Step 13.68 — Inspect the existing Symbol schema

query = """
MATCH (s:Symbol)
RETURN
    count(s) AS symbol_count,
    collect(DISTINCT s.symbol_type) AS symbol_types,
    collect(DISTINCT keys(s)) AS property_sets
"""

display(pd.DataFrame(run_cypher_query(query)))

,symbol_count,symbol_types,property_sets
0,63228,"[variable, function_expression, method, functi...","[[name, scope_id, symbol_id, start_line, end_l..."


In [204]:
# Step 13.69 — Inspect inheritance extraction

print(f"Inheritance records: {len(inheritance_df)}")

display(
    inheritance_df[
        [
            "inheritance_id",
            "class_id",
            "file_id",
            "base_expression",
            "resolved_base_symbol_id"
        ]
    ].head(15)
)

Inheritance records: 151


,inheritance_id,class_id,file_id,base_expression,resolved_base_symbol_id
0,inh_1cf2cf1de88c1454,class_1cf2cf1de88c1454,chrome/content/scaffold/scaffoldItemPreview.js,XULElementBase,None
1,inh_dbe41342a3a7eae8,class_dbe41342a3a7eae8,chrome/content/scaffold/scaffoldItemPreviews.js,XULElementBase,None
2,inh_de7620c0b29c9e32,class_de7620c0b29c9e32,chrome/content/scaffold/zoteroWebTranslationEn...,AbstractWebTranslationEnvironment,None
3,inh_58ddf19bd0ff4d05,class_58ddf19bd0ff4d05,chrome/content/zotero/actors/DocumentIsReadyCh...,JSWindowActorChild,None
4,inh_bd00d61eb0c223d9,class_bd00d61eb0c223d9,chrome/content/zotero/actors/ExternalLinkHandl...,JSWindowActorChild,None
5,inh_f016bb58b47ee370,class_f016bb58b47ee370,chrome/content/zotero/actors/ExternalLinkHandl...,JSWindowActorParent,None
6,inh_1d9c3a53afc35d65,class_1d9c3a53afc35d65,chrome/content/zotero/actors/FeedAbstractChild...,JSWindowActorChild,None
7,inh_60872808a921878b,class_60872808a921878b,chrome/content/zotero/actors/FeedAbstractParen...,JSWindowActorParent,None
8,inh_ea0f90737e63d54b,class_ea0f90737e63d54b,chrome/content/zotero/actors/MendeleyAuthChild...,JSWindowActorChild,None
9,inh_e216872f6b7e6faf,class_e216872f6b7e6faf,chrome/content/zotero/actors/MendeleyAuthParen...,JSWindowActorParent,None


In [207]:
# Step 13.70A — Check inheritance coverage

inheritance_class_check = inheritance_df.merge(
    classes_clean_canonical_df[
        ["class_id", "class_name"]
    ],
    on="class_id",
    how="left",
    indicator=True
)

print(
    inheritance_class_check["class_name"]
    .notna()
    .sum(),
    "inheritance records from named classes"
)

print(
    inheritance_class_check["class_name"]
    .isna()
    .sum(),
    "inheritance records from anonymous classes"
)

print(
    "Unmatched class IDs:",
    (inheritance_class_check["_merge"] == "left_only").sum()
)

KeyError: 'class_name'

In [208]:
# Step 13.70B - REconcile column names

print("inheritance_df:", inheritance_df.columns.tolist())
print(
    "classes_clean_canonical_df:",
    classes_clean_canonical_df.columns.tolist()
)

inheritance_df: ['inheritance_id', 'class_id', 'file_id', 'class_name', 'base_expression', 'heritage_text', 'resolved_base_symbol_id', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']
classes_clean_canonical_df: ['class_id', 'file_id', 'class_name', 'class_node_type', 'heritage_text', 'base_expression', 'source_text', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


In [210]:
# Step 13.70C — Reconcile inheritance coverage

named_class_ids = set(
    classes_clean_canonical_df.loc[
        classes_clean_canonical_df["class_name"].notna(),
        "class_id"
    ]
)

anonymous_class_ids = set(
    classes_clean_canonical_df.loc[
        classes_clean_canonical_df["class_name"].isna(),
        "class_id"
    ]
)

inheritance_class_ids = set(inheritance_df["class_id"])

print(
    "Named class inheritance records:",
    inheritance_df["class_id"].isin(named_class_ids).sum()
)

print(
    "Anonymous class inheritance records:",
    inheritance_df["class_id"].isin(anonymous_class_ids).sum()
)

print(
    "Unmatched inheritance records:",
    (~inheritance_df["class_id"].isin(
        named_class_ids | anonymous_class_ids
    )).sum()
)

Named class inheritance records: 124
Anonymous class inheritance records: 27
Unmatched inheritance records: 0


In [211]:
# Step 13.71 — Check inheritance resolution status

print(
    inheritance_df["resolved_base_symbol_id"]
    .notna()
    .value_counts()
    .rename({True: "Resolved", False: "Unresolved"})
)

print("\nDistinct base expressions:")
print(
    inheritance_df["base_expression"]
    .nunique()
)

display(
    inheritance_df[
        [
            "class_name",
            "base_expression",
            "resolved_base_symbol_id"
        ]
    ].head(10)
)

resolved_base_symbol_id
Unresolved    151
Name: count, dtype: int64

Distinct base expressions:
36


,class_name,base_expression,resolved_base_symbol_id
0,ScaffoldItemPreview,XULElementBase,None
1,ScaffoldItemPreviews,XULElementBase,None
2,ZoteroWebTranslationEnvironment,AbstractWebTranslationEnvironment,None
3,DocumentIsReadyChild,JSWindowActorChild,None
4,ExternalLinkHandlerChild,JSWindowActorChild,None
5,ExternalLinkHandlerParent,JSWindowActorParent,None
6,FeedAbstractChild,JSWindowActorChild,None
7,FeedAbstractParent,JSWindowActorParent,None
8,MendeleyAuthChild,JSWindowActorChild,None
9,MendeleyAuthParent,JSWindowActorParent,None


In [ ]:
# Step 13.72 — Proposed graph model

# (File)-[:DECLARES]->(Symbol)
# (Symbol)-[:HAS_INHERITANCE_EVIDENCE]->(Inheritance)

# (File)-[:HAS_ANONYMOUS_CLASS]->(AnonymousClass)
# (AnonymousClass)-[:HAS_INHERITANCE_EVIDENCE]->(Inheritance)

In [212]:
# Step 13.73 — Inspect inheritance DataFrame integrity

print("Inheritance records:", len(inheritance_df))
print(
    "Unique inheritance IDs:",
    inheritance_df["inheritance_id"].nunique()
)
print(
    "Unique class IDs:",
    inheritance_df["class_id"].nunique()
)
print(
    "Missing class IDs:",
    inheritance_df["class_id"].isna().sum()
)
print(
    "Missing base expressions:",
    inheritance_df["base_expression"].isna().sum()
)
print(
    "Missing file IDs:",
    inheritance_df["file_id"].isna().sum()
)

Inheritance records: 151
Unique inheritance IDs: 151
Unique class IDs: 151
Missing class IDs: 0
Missing base expressions: 0
Missing file IDs: 0


In [213]:
# Step 13.74 — Prepare anonymous class records

anonymous_classes_ingest_df = anonymous_classes_clean[
    [
        "class_id",
        "file_id",
        "class_node_type",
        "source_text",
        "heritage_text",
        "base_expression",
        "start_line",
        "start_column",
        "end_line",
        "end_column",
        "start_byte",
        "end_byte"
    ]
].copy()

print("Anonymous classes:", len(anonymous_classes_ingest_df))
print("Unique class IDs:", anonymous_classes_ingest_df["class_id"].nunique())
print("Unique files:", anonymous_classes_ingest_df["file_id"].nunique())
print("Missing file IDs:", anonymous_classes_ingest_df["file_id"].isna().sum())

display(anonymous_classes_ingest_df.head())

Anonymous classes: 39
Unique class IDs: 39
Unique files: 11
Missing file IDs: 0


,class_id,file_id,class_node_type,source_text,heritage_text,base_expression,start_line,start_column,end_line,end_column,start_byte,end_byte
42,class_b37c99c1d6597118,file_4557095a55e65e15,class,class {\n\t_registeredRemoteTranslates = new M...,NaN,NaN,3,40,70,2,101,1871
84,class_024af776c0636f67,file_555966c236571b4d,class,class {\n\tconstructor(virtualizedTable) {\n\t...,NaN,NaN,1906,15,2184,2,62092,71894
86,class_cc0f9b4b901f3ed8,file_72d2cdb95acab69d,class,class {\n\t/**\n\t * @param options (required)...,None,None,42,18,337,2,1631,11305
116,class_60148e2c16792c63,file_b9a538ee02f4f3ab,class,class extends Class {\n\t\tinitialized = false...,extends Class,Class,31,9,82,3,1082,2161
282,class_910d535dd4da10fc,file_096847be2a24c649,class,"class {\n\t\t\tconstructor(citationField, data...",None,None,1432,18,1477,4,43340,44475


In [214]:
# Step 13.75 — Create the anonymous class constraint

query = """
CREATE CONSTRAINT anonymous_class_id_unique IF NOT EXISTS
FOR (c:AnonymousClass)
REQUIRE c.class_id IS UNIQUE
"""

run_cypher_query(query)

[]

In [215]:
# Step 13.76 — Create AnonymousClass nodes

query = """
UNWIND $rows AS row
MATCH (f:File {file_id: row.file_id})
MERGE (c:AnonymousClass {class_id: row.class_id})
SET c.class_node_type = row.class_node_type,
    c.source_text = row.source_text,
    c.heritage_text = row.heritage_text,
    c.base_expression = row.base_expression,
    c.start_line = row.start_line,
    c.start_column = row.start_column,
    c.end_line = row.end_line,
    c.end_column = row.end_column,
    c.start_byte = row.start_byte,
    c.end_byte = row.end_byte
MERGE (f)-[:HAS_ANONYMOUS_CLASS]->(c)
RETURN count(c) AS classes_created_or_matched
"""

rows = anonymous_classes_ingest_df.where(
    pd.notna(anonymous_classes_ingest_df), None
).to_dict("records")

result = run_cypher_query(query, {"rows": rows})
display(pd.DataFrame(result))

,classes_created_or_matched
0,39


In [216]:
# Step 13.77 — Validate the anonymous class graph

query = """
MATCH (c:AnonymousClass)
OPTIONAL MATCH (f:File)-[:HAS_ANONYMOUS_CLASS]->(c)
RETURN
    count(DISTINCT c) AS anonymous_classes,
    count(DISTINCT f) AS connected_files,
    count(DISTINCT CASE WHEN f IS NULL THEN c END) AS classes_without_file
"""

display(pd.DataFrame(run_cypher_query(query)))

,anonymous_classes,connected_files,classes_without_file
0,39,11,0


In [217]:
# Step 13.78 — Create the inheritance constraint

query = """
CREATE CONSTRAINT inheritance_id_unique IF NOT EXISTS
FOR (i:Inheritance)
REQUIRE i.inheritance_id IS UNIQUE
"""

run_cypher_query(query)

[]

In [218]:
# Step 13.79 — Prepare inheritance records

inheritance_ingest_df = inheritance_df.copy()

inheritance_ingest_df["file_id"] = (
    inheritance_ingest_df["file_id"].map(file_id_mapping)
)

print("Inheritance records:", len(inheritance_ingest_df))
print("Unique inheritance IDs:", inheritance_ingest_df["inheritance_id"].nunique())
print("Missing canonical file IDs:", inheritance_ingest_df["file_id"].isna().sum())

display(inheritance_ingest_df.head())

Inheritance records: 151
Unique inheritance IDs: 151
Missing canonical file IDs: 0


,inheritance_id,class_id,file_id,class_name,base_expression,heritage_text,resolved_base_symbol_id,start_line,start_column,end_line,end_column,start_byte,end_byte
0,inh_1cf2cf1de88c1454,class_1cf2cf1de88c1454,file_18307e89f1ddb102,ScaffoldItemPreview,XULElementBase,extends XULElementBase,None,31,2,216,3,980,6363
1,inh_dbe41342a3a7eae8,class_dbe41342a3a7eae8,file_a6b19f7562edaee3,ScaffoldItemPreviews,XULElementBase,extends XULElementBase,None,29,2,143,3,866,3844
2,inh_de7620c0b29c9e32,class_de7620c0b29c9e32,file_ebe5fee72d4d11cc,ZoteroWebTranslationEnvironment,AbstractWebTranslationEnvironment,extends AbstractWebTranslationEnvironment,None,3,8,77,2,138,2267
3,inh_58ddf19bd0ff4d05,class_58ddf19bd0ff4d05,file_a4c81444f3cabedc,DocumentIsReadyChild,JSWindowActorChild,extends JSWindowActorChild,None,3,8,12,2,89,353
4,inh_bd00d61eb0c223d9,class_bd00d61eb0c223d9,file_d1d72f82a6caef82,ExternalLinkHandlerChild,JSWindowActorChild,extends JSWindowActorChild,None,1,8,67,2,7,1698


In [219]:
# Step 13.80 — Build the class identity mapping

named_class_symbol_map = (
    named_classes_clean[
        ["class_id", "file_id", "class_name"]
    ]
    .merge(
        canonical_dataframes["symbols_df"][
            ["symbol_id", "file_id", "name", "symbol_type"]
        ].query("symbol_type == 'class'"),
        left_on=["file_id", "class_name"],
        right_on=["file_id", "name"],
        how="left"
    )
    [["class_id", "symbol_id"]]
)

inheritance_ingest_df["symbol_id"] = (
    inheritance_ingest_df["class_id"]
    .map(named_class_symbol_map.set_index("class_id")["symbol_id"])
)

print("Named class mappings:", inheritance_ingest_df["symbol_id"].notna().sum())
print("Anonymous class records:", inheritance_ingest_df["symbol_id"].isna().sum())
print(
    "Unmatched class IDs:",
    (~inheritance_ingest_df["class_id"].isin(
        set(named_class_ids) | set(anonymous_class_ids)
    )).sum()
)

Named class mappings: 124
Anonymous class records: 27
Unmatched class IDs: 0


In [220]:
# Step 13.81 — Ingest inheritance evidence

query = """
UNWIND $rows AS row

OPTIONAL MATCH (s:Symbol {symbol_id: row.symbol_id})
OPTIONAL MATCH (a:AnonymousClass {class_id: row.class_id})

WITH row, coalesce(s, a) AS class_construct
WHERE class_construct IS NOT NULL

MERGE (i:Inheritance {inheritance_id: row.inheritance_id})
SET i.base_expression = row.base_expression,
    i.heritage_text = row.heritage_text,
    i.resolution_status = 'UNRESOLVED',
    i.resolved_base_symbol_id = null,
    i.start_line = row.start_line,
    i.start_column = row.start_column,
    i.end_line = row.end_line,
    i.end_column = row.end_column,
    i.start_byte = row.start_byte,
    i.end_byte = row.end_byte

MERGE (class_construct)-[:HAS_INHERITANCE_EVIDENCE]->(i)

RETURN count(i) AS inheritance_records_created_or_matched
"""

rows = inheritance_ingest_df.where(
    pd.notna(inheritance_ingest_df), None
).to_dict("records")

result = run_cypher_query(query, {"rows": rows})
display(pd.DataFrame(result))

,inheritance_records_created_or_matched
0,151


In [221]:
# Step 13.82 — Validate inheritance relationships

query = """
MATCH (i:Inheritance)
OPTIONAL MATCH (s:Symbol)-[:HAS_INHERITANCE_EVIDENCE]->(i)
OPTIONAL MATCH (a:AnonymousClass)-[:HAS_INHERITANCE_EVIDENCE]->(i)
RETURN
    count(DISTINCT i) AS inheritance_records,
    count(DISTINCT CASE WHEN s IS NOT NULL THEN i END) AS named_class_inheritance,
    count(DISTINCT CASE WHEN a IS NOT NULL THEN i END) AS anonymous_class_inheritance,
    count(DISTINCT CASE WHEN s IS NULL AND a IS NULL THEN i END) AS orphaned_inheritance,
    count(DISTINCT CASE WHEN i.resolution_status = 'UNRESOLVED' THEN i END) AS unresolved_inheritance
"""

display(pd.DataFrame(run_cypher_query(query)))

,inheritance_records,named_class_inheritance,anonymous_class_inheritance,orphaned_inheritance,unresolved_inheritance
0,151,124,27,0,151


In [222]:
# Step 13.83 — Resolved references

print(f"Resolved reference records: {len(resolved_references_df)}")
print("Columns:", resolved_references_df.columns.tolist())

display(resolved_references_df.head())

Resolved reference records: 254402
Columns: ['reference_id', 'file_id', 'name', 'reference_role', 'scope_id', 'resolved_symbol_id', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte', 'candidate_symbol_ids', 'candidate_count', 'resolution_status', 'resolution_method']


,reference_id,file_id,name,reference_role,scope_id,resolved_symbol_id,start_line,start_column,end_line,end_column,start_byte,end_byte,candidate_symbol_ids,candidate_count,resolution_status,resolution_method
0,ref_6a0ce5fabc0d788e,app/assets/commandLineHandler.js,ChromeUtils,call_callee,None,None,1,43,1,54,42,53,[],0,UNRESOLVED,None
1,ref_f274b1f8b0893350,app/assets/commandLineHandler.js,cmdLine,reference,None,None,5,5,5,12,262,269,[],0,UNRESOLVED,None
2,ref_b1d050d716f004e8,app/assets/commandLineHandler.js,Ci,reference,None,None,5,22,5,24,279,281,[],0,UNRESOLVED,None
3,ref_9d9636f34aed2aa1,app/assets/commandLineHandler.js,cmdLine,assignment_target,None,None,6,2,6,9,322,329,[],0,UNRESOLVED,None
4,ref_80f5d9a64f1c4b0b,app/assets/commandLineHandler.js,cmdLine,call_callee,None,None,10,5,10,12,392,399,[],0,UNRESOLVED,None


In [223]:
# Step 13.84 - Check the status distribution and candidate data

print("Resolution status:")
display(
    resolved_references_df["resolution_status"]
    .value_counts(dropna=False)
    .rename_axis("resolution_status")
    .reset_index(name="records")
)

print("Candidate count distribution:")
display(
    resolved_references_df["candidate_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("candidate_count")
    .reset_index(name="records")
)

print("Resolution methods:")
display(
    resolved_references_df["resolution_method"]
    .value_counts(dropna=False)
    .rename_axis("resolution_method")
    .reset_index(name="records")
)

Resolution status:


,resolution_status,records
0,AMBIGUOUS,109082
1,UNRESOLVED,80274
2,RESOLVED,65046


Candidate count distribution:


,candidate_count,records
0,0,80274
1,1,65046
2,2,17109
3,3,10727
4,4,7816
...,...,...
75,301,1320
76,397,1856
77,503,2372
78,513,4211


Resolution methods:


,resolution_method,records
0,same_file_name,174128
1,NaN,80274


In [225]:
# Step 13.85 - Check python representation of data

print("Column data types:")
display(resolved_references_df[
    ["resolved_symbol_id", "candidate_symbol_ids",
     "candidate_count", "resolution_status"]
].dtypes)

print("\nActual Python value types:")
for column in ["resolved_symbol_id", "candidate_symbol_ids"]:
    print(f"\n{column}:")
    display(
        resolved_references_df[column]
        .map(lambda x: type(x).__name__)
        .value_counts()
        .rename_axis("python_type")
        .reset_index(name="records")
    )

print("\nRepresentative records:")
display(
    resolved_references_df.loc[
        resolved_references_df["resolution_status"].isin(
            ["RESOLVED", "AMBIGUOUS"]
        ),
        ["reference_id", "resolved_symbol_id",
         "candidate_symbol_ids", "candidate_count",
         "resolution_status"]
    ].head(10)
)

Column data types:


resolved_symbol_id      object
candidate_symbol_ids    object
candidate_count          int64
resolution_status          str
dtype: object


Actual Python value types:

resolved_symbol_id:


,python_type,records
0,NoneType,189356
1,str,65046



candidate_symbol_ids:


,python_type,records
0,list,254402



Representative records:


,reference_id,resolved_symbol_id,candidate_symbol_ids,candidate_count,resolution_status
22,ref_831d2368599ba3a2,sym_3b0d08eff5808038,[sym_3b0d08eff5808038],1,RESOLVED
27,ref_2aa24d88663ece1c,sym_3b0d08eff5808038,[sym_3b0d08eff5808038],1,RESOLVED
61,ref_a5f3fe4174cc257d,sym_281dac345cf207b2,[sym_281dac345cf207b2],1,RESOLVED
62,ref_c58dae64e0bc6a1f,sym_281dac345cf207b2,[sym_281dac345cf207b2],1,RESOLVED
68,ref_62c4c83486b93d8f,sym_21cf878906be0255,[sym_21cf878906be0255],1,RESOLVED
71,ref_2c3ffe6be3200348,sym_340803e89f92fd2d,[sym_340803e89f92fd2d],1,RESOLVED
72,ref_027863bd1a4e6be0,sym_c5727e816fee1abb,[sym_c5727e816fee1abb],1,RESOLVED
76,ref_4c69c121844a2d84,sym_21cf878906be0255,[sym_21cf878906be0255],1,RESOLVED
158,ref_629cd0fa34dfcd9d,sym_d7f11767ca70d14d,[sym_d7f11767ca70d14d],1,RESOLVED
163,ref_8a650b995c83775f,sym_1ac07a30fa06bd8e,[sym_1ac07a30fa06bd8e],1,RESOLVED


In [226]:
# Step 13.86 - QA on candidate-list consistency, resolved-target consistency, and whether candidate IDs exist in the extracted Symbol inventory

symbol_ids = set(canonical_dataframes["symbols_df"]["symbol_id"])

candidate_lengths = resolved_references_df["candidate_symbol_ids"].map(len)

candidate_ids = {
    symbol_id
    for candidates in resolved_references_df["candidate_symbol_ids"]
    for symbol_id in candidates
}

resolved_rows = resolved_references_df[
    resolved_references_df["resolution_status"] == "RESOLVED"
]

qa_results = pd.DataFrame([
    ("Candidate count matches list length",
     (candidate_lengths == resolved_references_df["candidate_count"]).all()),
    ("Resolved target is in candidate list",
     resolved_rows.apply(
         lambda row: row["resolved_symbol_id"] in row["candidate_symbol_ids"],
         axis=1
     ).all()),
    ("All candidate IDs exist as Symbols",
     candidate_ids.issubset(symbol_ids)),
    ("All reference IDs are unique",
     resolved_references_df["reference_id"].is_unique),
])

qa_results.columns = ["check", "passed"]
display(qa_results)

print("Distinct candidate symbols:", len(candidate_ids))
print("Missing candidate symbols:", len(candidate_ids - symbol_ids))

,check,passed
0,Candidate count matches list length,True
1,Resolved target is in candidate list,True
2,All candidate IDs exist as Symbols,True
3,All reference IDs are unique,True


Distinct candidate symbols: 41115
Missing candidate symbols: 0


In [227]:
# Step 13.87 — Prepare resolved-reference ingestion data

resolved_references_ingest_df = resolved_references_df.copy()

print("Records:", len(resolved_references_ingest_df))
print("Unique reference IDs:",
      resolved_references_ingest_df["reference_id"].nunique())
print("Missing reference IDs:",
      resolved_references_ingest_df["reference_id"].isna().sum())
print("Candidate IDs retained:",
      resolved_references_ingest_df["candidate_symbol_ids"].map(len).sum())

display(
    resolved_references_ingest_df[
        ["reference_id", "resolution_status",
         "resolution_method", "resolved_symbol_id",
         "candidate_count", "candidate_symbol_ids"]
    ].head()
)

Records: 254402
Unique reference IDs: 254402
Missing reference IDs: 0
Candidate IDs retained: 10449701


,reference_id,resolution_status,resolution_method,resolved_symbol_id,candidate_count,candidate_symbol_ids
0,ref_6a0ce5fabc0d788e,UNRESOLVED,None,None,0,[]
1,ref_f274b1f8b0893350,UNRESOLVED,None,None,0,[]
2,ref_b1d050d716f004e8,UNRESOLVED,None,None,0,[]
3,ref_9d9636f34aed2aa1,UNRESOLVED,None,None,0,[]
4,ref_80f5d9a64f1c4b0b,UNRESOLVED,None,None,0,[]


In [228]:
# Step 13.88 — Create the ReferenceResolution uniqueness constraint

query = """
CREATE CONSTRAINT reference_resolution_id_unique IF NOT EXISTS
FOR (r:ReferenceResolution)
REQUIRE r.reference_id IS UNIQUE
"""

result = run_cypher_query(query)

print("ReferenceResolution uniqueness constraint:")
print(result)

ReferenceResolution uniqueness constraint:
[]


In [229]:
# Step 13.89 — Ingest ReferenceResolution nodes

BATCH_SIZE = 5000

query = """
UNWIND $rows AS row

MERGE (rr:ReferenceResolution {
    reference_id: row.reference_id
})

SET rr.resolution_status = row.resolution_status,
    rr.resolution_method = row.resolution_method,
    rr.resolved_symbol_id = row.resolved_symbol_id,
    rr.candidate_count = row.candidate_count,
    rr.candidate_symbol_ids = row.candidate_symbol_ids

WITH row, rr
MATCH (r:Reference {reference_id: row.reference_id})
MERGE (r)-[:HAS_RESOLUTION]->(rr)

RETURN count(rr) AS records_ingested
"""

total_ingested = 0

for start in range(0, len(resolved_references_ingest_df), BATCH_SIZE):
    batch = resolved_references_ingest_df.iloc[start:start + BATCH_SIZE]

    rows = batch[
        [
            "reference_id",
            "resolution_status",
            "resolution_method",
            "resolved_symbol_id",
            "candidate_count",
            "candidate_symbol_ids"
        ]
    ].to_dict("records")

    result = run_cypher_query(query, {"rows": rows})

    batch_count = result[0]["records_ingested"] if result else 0
    total_ingested += batch_count

    print(
        f"Batch {start // BATCH_SIZE + 1}: "
        f"{batch_count:,} records"
    )

print(f"\nTotal records ingested: {total_ingested:,}")
print(f"Expected records: {len(resolved_references_ingest_df):,}")

Batch 1: 5,000 records
Batch 2: 5,000 records
Batch 3: 5,000 records
Batch 4: 5,000 records
Batch 5: 5,000 records
Batch 6: 5,000 records
Batch 7: 5,000 records
Batch 8: 5,000 records
Batch 9: 5,000 records
Batch 10: 5,000 records
Batch 11: 5,000 records
Batch 12: 5,000 records
Batch 13: 5,000 records
Batch 14: 5,000 records
Batch 15: 5,000 records
Batch 16: 5,000 records
Batch 17: 5,000 records
Batch 18: 5,000 records
Batch 19: 5,000 records
Batch 20: 5,000 records
Batch 21: 5,000 records
Batch 22: 5,000 records
Batch 23: 5,000 records
Batch 24: 5,000 records
Batch 25: 5,000 records
Batch 26: 5,000 records
Batch 27: 5,000 records
Batch 28: 5,000 records
Batch 29: 5,000 records
Batch 30: 5,000 records
Batch 31: 5,000 records
Batch 32: 5,000 records
Batch 33: 5,000 records
Batch 34: 5,000 records
Batch 35: 5,000 records
Batch 36: 5,000 records
Batch 37: 5,000 records
Batch 38: 5,000 records
Batch 39: 5,000 records
Batch 40: 5,000 records
Batch 41: 5,000 records
Batch 42: 5,000 records
B

In [230]:
# Step 13.90 — Validate ReferenceResolution nodes and relationships

query = """
MATCH (rr:ReferenceResolution)
OPTIONAL MATCH (r:Reference)-[:HAS_RESOLUTION]->(rr)
WITH rr, count(r) AS reference_count

RETURN
    count(rr) AS total_resolutions,
    sum(CASE WHEN reference_count = 0 THEN 1 ELSE 0 END)
        AS without_reference,
    sum(CASE WHEN reference_count = 1 THEN 1 ELSE 0 END)
        AS with_one_reference,
    sum(CASE WHEN reference_count > 1 THEN 1 ELSE 0 END)
        AS with_multiple_references
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

query = """
MATCH (rr:ReferenceResolution)
RETURN
    rr.resolution_status AS resolution_status,
    count(*) AS records
ORDER BY resolution_status
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,total_resolutions,without_reference,with_one_reference,with_multiple_references
0,254402,0,254402,0


,resolution_status,records
0,AMBIGUOUS,109082
1,RESOLVED,65046
2,UNRESOLVED,80274


In [231]:
# Step 13.91 — Validate ReferenceResolution connectivity

query = """
MATCH (rr:ReferenceResolution)
OPTIONAL MATCH (r:Reference)-[:HAS_RESOLUTION]->(rr)
WITH rr, count(r) AS reference_count

RETURN
    count(rr) AS total_resolutions,
    sum(CASE WHEN reference_count = 0 THEN 1 ELSE 0 END)
        AS without_reference,
    sum(CASE WHEN reference_count = 1 THEN 1 ELSE 0 END)
        AS with_one_reference,
    sum(CASE WHEN reference_count > 1 THEN 1 ELSE 0 END)
        AS with_multiple_references
"""

connectivity_result = run_cypher_query(query)

display(pd.DataFrame(connectivity_result))

query = """
MATCH (rr:ReferenceResolution)
RETURN
    count(*) AS total_resolutions,
    sum(rr.candidate_count) AS total_candidate_entries,
    sum(CASE WHEN rr.resolution_status = 'RESOLVED'
             AND rr.resolved_symbol_id IS NOT NULL
             THEN 1 ELSE 0 END) AS resolved_with_target,
    sum(CASE WHEN rr.resolution_status = 'AMBIGUOUS'
             THEN 1 ELSE 0 END) AS ambiguous,
    sum(CASE WHEN rr.resolution_status = 'UNRESOLVED'
             THEN 1 ELSE 0 END) AS unresolved
"""

candidate_result = run_cypher_query(query)

display(pd.DataFrame(candidate_result))

,total_resolutions,without_reference,with_one_reference,with_multiple_references
0,254402,0,254402,0


,total_resolutions,total_candidate_entries,resolved_with_target,ambiguous,unresolved
0,254402,10449701,65046,109082,80274


In [232]:
# Step 13.92 — Validate candidate evidence and resolved targets

query = """
MATCH (rr:ReferenceResolution)
RETURN
    count(*) AS total_resolutions,
    sum(rr.candidate_count) AS total_candidate_entries,
    sum(CASE
        WHEN rr.resolution_status = 'RESOLVED'
         AND rr.resolved_symbol_id IS NOT NULL
        THEN 1 ELSE 0
    END) AS resolved_with_target,
    sum(CASE
        WHEN rr.resolution_status = 'AMBIGUOUS'
        THEN 1 ELSE 0
    END) AS ambiguous,
    sum(CASE
        WHEN rr.resolution_status = 'UNRESOLVED'
        THEN 1 ELSE 0
    END) AS unresolved
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,total_resolutions,total_candidate_entries,resolved_with_target,ambiguous,unresolved
0,254402,10449701,65046,109082,80274


In [233]:
# Step 13.93 — Create confirmed RESOLVES_TO relationships

query = """
MATCH (rr:ReferenceResolution)
WHERE rr.resolution_status = 'RESOLVED'
  AND rr.resolved_symbol_id IS NOT NULL

MATCH (s:Symbol {symbol_id: rr.resolved_symbol_id})

MERGE (rr)-[:RESOLVES_TO]->(s)

RETURN count(*) AS relationships_created_or_matched
"""

result = run_cypher_query(query)

display(pd.DataFrame(result))

,relationships_created_or_matched
0,65046


In [234]:
# Step 13.94 — Validate confirmed reference resolution relationships

query = """
MATCH (rr:ReferenceResolution)
OPTIONAL MATCH (rr)-[:RESOLVES_TO]->(s:Symbol)

WITH rr, count(s) AS target_count

RETURN
    count(rr) AS total_resolutions,

    sum(CASE
        WHEN rr.resolution_status = 'RESOLVED'
         AND target_count = 1
        THEN 1 ELSE 0
    END) AS resolved_with_one_target,

    sum(CASE
        WHEN rr.resolution_status = 'RESOLVED'
         AND target_count = 0
        THEN 1 ELSE 0
    END) AS resolved_without_target,

    sum(CASE
        WHEN rr.resolution_status = 'RESOLVED'
         AND target_count > 1
        THEN 1 ELSE 0
    END) AS resolved_with_multiple_targets,

    sum(CASE
        WHEN rr.resolution_status IN ['AMBIGUOUS', 'UNRESOLVED']
         AND target_count > 0
        THEN 1 ELSE 0
    END) AS non_resolved_with_confirmed_target
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,total_resolutions,resolved_with_one_target,resolved_without_target,resolved_with_multiple_targets,non_resolved_with_confirmed_target
0,254402,65046,0,0,0


In [237]:
# Step 13.95 — Create candidate relationships for AMBIGUOUS references

BATCH_SIZE = 1000

ambiguous_df = resolved_references_ingest_df[
    resolved_references_ingest_df["resolution_status"] == "AMBIGUOUS"
].copy()

query = """
UNWIND $rows AS row

MATCH (rr:ReferenceResolution {
    reference_id: row.reference_id
})

UNWIND row.candidate_symbol_ids AS candidate_id

MATCH (s:Symbol {symbol_id: candidate_id})

MERGE (rr)-[:HAS_CANDIDATE]->(s)

RETURN count(*) AS relationships_created_or_matched
"""

total_candidates = 0

for start in range(0, len(ambiguous_df), BATCH_SIZE):

    batch = ambiguous_df.iloc[start:start + BATCH_SIZE]

    rows = batch[
        ["reference_id", "candidate_symbol_ids"]
    ].to_dict("records")

    result = run_cypher_query(query, {"rows": rows})

    batch_count = result[0]["relationships_created_or_matched"]
    total_candidates += batch_count

    print(
        f"Batch {start // BATCH_SIZE + 1}: "
        f"{batch_count:,} candidate relationships"
    )

print(f"\nTotal candidate relationships: {total_candidates:,}")
print(
    "Expected candidate relationships:",
    int(ambiguous_df["candidate_count"].sum())
)

Batch 1: 4,502 candidate relationships
Batch 2: 7,162 candidate relationships
Batch 3: 5,280 candidate relationships
Batch 4: 3,036 candidate relationships
Batch 5: 4,669 candidate relationships
Batch 6: 3,944 candidate relationships
Batch 7: 3,497 candidate relationships
Batch 8: 3,872 candidate relationships
Batch 9: 5,947 candidate relationships
Batch 10: 3,181 candidate relationships
Batch 11: 4,110 candidate relationships
Batch 12: 7,920 candidate relationships
Batch 13: 73,386 candidate relationships
Batch 14: 60,319 candidate relationships
Batch 15: 59,857 candidate relationships
Batch 16: 62,317 candidate relationships
Batch 17: 58,949 candidate relationships
Batch 18: 64,891 candidate relationships
Batch 19: 39,551 candidate relationships
Batch 20: 63,445 candidate relationships
Batch 21: 55,104 candidate relationships
Batch 22: 64,661 candidate relationships
Batch 23: 75,402 candidate relationships
Batch 24: 8,839 candidate relationships
Batch 25: 6,098 candidate relationship

In [238]:
# Step 13.96 — Validate ambiguous candidate relationships

query = """
MATCH (rr:ReferenceResolution)
WHERE rr.resolution_status = 'AMBIGUOUS'

OPTIONAL MATCH (rr)-[:HAS_CANDIDATE]->(s:Symbol)

WITH rr, count(s) AS actual_candidates

RETURN
    count(rr) AS ambiguous_references,
    sum(rr.candidate_count) AS expected_candidates,
    sum(actual_candidates) AS actual_candidates,
    sum(CASE
        WHEN actual_candidates = rr.candidate_count
        THEN 1 ELSE 0
    END) AS references_with_matching_candidate_counts,
    sum(CASE
        WHEN actual_candidates <> rr.candidate_count
        THEN 1 ELSE 0
    END) AS references_with_mismatched_candidate_counts
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,ambiguous_references,expected_candidates,actual_candidates,references_with_matching_candidate_counts,references_with_mismatched_candidate_counts
0,109082,10384655,10384655,109082,0


In [239]:
# Step 13.97 — Inspect resolved call structure

print(f"Resolved call records: {len(resolved_calls_df)}")
print("Columns:", resolved_calls_df.columns.tolist())

display(resolved_calls_df.head())

print("\nResolution status:")
display(
    resolved_calls_df["resolution_status"]
    .value_counts(dropna=False)
    .rename_axis("resolution_status")
    .reset_index(name="records")
)

Resolved call records: 108839
Columns: ['call_id', 'file_id', 'call_kind', 'call_node_type', 'callee_expression', 'call_text', 'arguments_text', 'argument_count', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte', 'candidate_symbol_ids', 'candidate_count', 'resolved_callee_symbol_id', 'resolution_status', 'resolution_method']


,call_id,file_id,call_kind,call_node_type,callee_expression,call_text,arguments_text,argument_count,start_line,start_column,end_line,end_column,start_byte,end_byte,candidate_symbol_ids,candidate_count,resolved_callee_symbol_id,resolution_status,resolution_method
0,call_7ad8d0433055c936,app/assets/commandLineHandler.js,function_call,call_expression,ChromeUtils.importESModule,"ChromeUtils.importESModule(""chrome://zotero/co...","(""chrome://zotero/content/modules/commandLineO...",1,1,43,1,127,42,126,[],0,None,UNRESOLVED,None
1,call_6c88a4a466caa540,app/assets/commandLineHandler.js,function_call,call_expression,cmdLine.handleFlag,"cmdLine.handleFlag(""ZoteroDebug"", false)","(""ZoteroDebug"", false)",2,10,5,10,45,392,432,[],0,None,UNRESOLVED,None
2,call_85b7a36ee0731bc3,app/assets/commandLineHandler.js,function_call,call_expression,cmdLine.handleFlag,"cmdLine.handleFlag(""ZoteroDebugText"", false)","(""ZoteroDebugText"", false)",2,14,10,14,54,524,568,[],0,None,UNRESOLVED,None
3,call_1923a9daaf3b6815,app/assets/commandLineHandler.js,function_call,call_expression,"Services.prefs.getBranch(""toolkit.startup."").c...","Services.prefs.getBranch(""toolkit.startup."").c...","(""recent_crashes"")",1,21,2,21,78,935,1011,[],0,None,UNRESOLVED,None
4,call_fe610d8b78929d54,app/assets/commandLineHandler.js,function_call,call_expression,Services.prefs.getBranch,"Services.prefs.getBranch(""toolkit.startup."")","(""toolkit.startup."")",1,21,2,21,46,935,979,[],0,None,UNRESOLVED,None



Resolution status:


,resolution_status,records
0,UNRESOLVED,99027
1,RESOLVED,7734
2,AMBIGUOUS,2078


In [240]:
# Step 13.98 — Validate resolved call status and candidate data

print("Resolution status:")
display(
    resolved_calls_df["resolution_status"]
    .value_counts(dropna=False)
    .rename_axis("resolution_status")
    .reset_index(name="records")
)

print("\nCandidate count distribution:")
display(
    resolved_calls_df["candidate_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("candidate_count")
    .reset_index(name="records")
)

print("\nPython value types:")
for column in [
    "resolved_callee_symbol_id",
    "candidate_symbol_ids"
]:
    print(f"\n{column}:")
    display(
        resolved_calls_df[column]
        .map(lambda x: type(x).__name__)
        .value_counts()
        .rename_axis("python_type")
        .reset_index(name="records")
    )

Resolution status:


,resolution_status,records
0,UNRESOLVED,99027
1,RESOLVED,7734
2,AMBIGUOUS,2078



Candidate count distribution:


,candidate_count,records
0,0,99027
1,1,7734
2,2,312
3,3,169
4,4,122
5,5,43
6,6,16
7,7,27
8,8,25
9,9,25



Python value types:

resolved_callee_symbol_id:


,python_type,records
0,NoneType,101105
1,str,7734



candidate_symbol_ids:


,python_type,records
0,list,108839


In [241]:
# Step 13.99 — Validate call candidate data types

print("Python value types:")

for column in [
    "resolved_callee_symbol_id",
    "candidate_symbol_ids"
]:
    print(f"\n{column}:")
    display(
        resolved_calls_df[column]
        .map(lambda x: type(x).__name__)
        .value_counts()
        .rename_axis("python_type")
        .reset_index(name="records")
    )

print("\nCandidate count consistency:")

display(pd.DataFrame([{
    "total_calls": len(resolved_calls_df),
    "candidate_entries": int(
        resolved_calls_df["candidate_count"].sum()
    ),
    "candidate_list_entries": int(
        resolved_calls_df["candidate_symbol_ids"].map(len).sum()
    ),
    "counts_match": (
        resolved_calls_df["candidate_count"]
        == resolved_calls_df["candidate_symbol_ids"].map(len)
    ).all()
}]))

Python value types:

resolved_callee_symbol_id:


,python_type,records
0,NoneType,101105
1,str,7734



candidate_symbol_ids:


,python_type,records
0,list,108839



Candidate count consistency:


,total_calls,candidate_entries,candidate_list_entries,counts_match
0,108839,237115,237115,True


In [242]:
# Step 13.100 — Validate call candidate Symbol IDs

symbol_ids = set(
    canonical_dataframes["symbols_df"]["symbol_id"]
)

candidate_ids = {
    symbol_id
    for candidates in resolved_calls_df["candidate_symbol_ids"]
    for symbol_id in candidates
}

resolved_targets = set(
    resolved_calls_df["resolved_callee_symbol_id"].dropna()
)

qa_results = pd.DataFrame([
    {
        "check": "All candidate IDs exist as Symbols",
        "passed": candidate_ids.issubset(symbol_ids),
        "count": len(candidate_ids - symbol_ids)
    },
    {
        "check": "All resolved targets exist as Symbols",
        "passed": resolved_targets.issubset(symbol_ids),
        "count": len(resolved_targets - symbol_ids)
    },
    {
        "check": "All call IDs are unique",
        "passed": resolved_calls_df["call_id"].is_unique,
        "count": resolved_calls_df["call_id"].duplicated().sum()
    }
])

display(qa_results)

print("Distinct candidate symbols:", len(candidate_ids))
print("Distinct resolved target symbols:", len(resolved_targets))

,check,passed,count
0,All candidate IDs exist as Symbols,True,0
1,All resolved targets exist as Symbols,True,0
2,All call IDs are unique,True,0


Distinct candidate symbols: 8476
Distinct resolved target symbols: 2633


In [243]:
# Step 13.101 — Prepare resolved-call ingestion data

resolved_calls_ingest_df = resolved_calls_df.copy()

print("Records:", len(resolved_calls_ingest_df))
print(
    "Unique call IDs:",
    resolved_calls_ingest_df["call_id"].nunique()
)
print(
    "Missing call IDs:",
    resolved_calls_ingest_df["call_id"].isna().sum()
)
print(
    "Candidate entries:",
    resolved_calls_ingest_df["candidate_symbol_ids"].map(len).sum()
)

display(
    resolved_calls_ingest_df[
        [
            "call_id",
            "resolution_status",
            "resolution_method",
            "resolved_callee_symbol_id",
            "candidate_count",
            "candidate_symbol_ids"
        ]
    ].head()
)

Records: 108839
Unique call IDs: 108839
Missing call IDs: 0
Candidate entries: 237115


,call_id,resolution_status,resolution_method,resolved_callee_symbol_id,candidate_count,candidate_symbol_ids
0,call_7ad8d0433055c936,UNRESOLVED,None,None,0,[]
1,call_6c88a4a466caa540,UNRESOLVED,None,None,0,[]
2,call_85b7a36ee0731bc3,UNRESOLVED,None,None,0,[]
3,call_1923a9daaf3b6815,UNRESOLVED,None,None,0,[]
4,call_fe610d8b78929d54,UNRESOLVED,None,None,0,[]


In [244]:
# Step 13.102 — Create the CallResolution uniqueness constraint

query = """
CREATE CONSTRAINT call_resolution_id_unique IF NOT EXISTS
FOR (cr:CallResolution)
REQUIRE cr.call_id IS UNIQUE
"""

result = run_cypher_query(query)

print("CallResolution uniqueness constraint:")
print(result)

CallResolution uniqueness constraint:
[]


In [245]:
# Step 13.103 — Ingest CallResolution nodes

BATCH_SIZE = 5000

query = """
UNWIND $rows AS row

MERGE (cr:CallResolution {
    call_id: row.call_id
})

SET cr.resolution_status = row.resolution_status,
    cr.resolution_method = row.resolution_method,
    cr.resolved_callee_symbol_id = row.resolved_callee_symbol_id,
    cr.candidate_count = row.candidate_count,
    cr.candidate_symbol_ids = row.candidate_symbol_ids

WITH row, cr
MATCH (c:CallSite {call_id: row.call_id})
MERGE (c)-[:HAS_RESOLUTION]->(cr)

RETURN count(cr) AS records_ingested
"""

total_ingested = 0

for start in range(0, len(resolved_calls_ingest_df), BATCH_SIZE):

    batch = resolved_calls_ingest_df.iloc[
        start:start + BATCH_SIZE
    ]

    rows = batch[
        [
            "call_id",
            "resolution_status",
            "resolution_method",
            "resolved_callee_symbol_id",
            "candidate_count",
            "candidate_symbol_ids"
        ]
    ].to_dict("records")

    result = run_cypher_query(query, {"rows": rows})

    batch_count = result[0]["records_ingested"] if result else 0
    total_ingested += batch_count

    print(
        f"Batch {start // BATCH_SIZE + 1}: "
        f"{batch_count:,} records"
    )

print(f"\nTotal records ingested: {total_ingested:,}")
print(f"Expected records: {len(resolved_calls_ingest_df):,}")

Batch 1: 5,000 records
Batch 2: 5,000 records
Batch 3: 5,000 records
Batch 4: 5,000 records
Batch 5: 5,000 records
Batch 6: 5,000 records
Batch 7: 5,000 records
Batch 8: 5,000 records
Batch 9: 5,000 records
Batch 10: 5,000 records
Batch 11: 5,000 records
Batch 12: 5,000 records
Batch 13: 5,000 records
Batch 14: 5,000 records
Batch 15: 5,000 records
Batch 16: 5,000 records
Batch 17: 5,000 records
Batch 18: 5,000 records
Batch 19: 5,000 records
Batch 20: 5,000 records
Batch 21: 5,000 records
Batch 22: 3,839 records

Total records ingested: 108,839
Expected records: 108,839


In [246]:
# Step 13.104 — Validate CallResolution nodes and relationships

query = """
MATCH (cr:CallResolution)
OPTIONAL MATCH (c:CallSite)-[:HAS_RESOLUTION]->(cr)

WITH cr, count(c) AS call_count

RETURN
    count(cr) AS total_resolutions,
    sum(CASE WHEN call_count = 0 THEN 1 ELSE 0 END)
        AS without_call,
    sum(CASE WHEN call_count = 1 THEN 1 ELSE 0 END)
        AS with_one_call,
    sum(CASE WHEN call_count > 1 THEN 1 ELSE 0 END)
        AS with_multiple_calls
"""

connectivity_result = run_cypher_query(query)
display(pd.DataFrame(connectivity_result))

query = """
MATCH (cr:CallResolution)
RETURN
    cr.resolution_status AS resolution_status,
    count(*) AS records
ORDER BY resolution_status
"""

status_result = run_cypher_query(query)
display(pd.DataFrame(status_result))

,total_resolutions,without_call,with_one_call,with_multiple_calls
0,108839,0,108839,0


,resolution_status,records
0,AMBIGUOUS,2078
1,RESOLVED,7734
2,UNRESOLVED,99027


In [247]:
# Step 13.105 — Create confirmed RESOLVES_TO relationships for calls

query = """
MATCH (cr:CallResolution)
WHERE cr.resolution_status = 'RESOLVED'
  AND cr.resolved_callee_symbol_id IS NOT NULL

MATCH (s:Symbol {
    symbol_id: cr.resolved_callee_symbol_id
})

MERGE (cr)-[:RESOLVES_TO]->(s)

RETURN count(*) AS relationships_created_or_matched
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,relationships_created_or_matched
0,7734


In [248]:
# Step 13.106 — Validate confirmed call resolution relationships

query = """
MATCH (cr:CallResolution)
OPTIONAL MATCH (cr)-[:RESOLVES_TO]->(s:Symbol)

WITH cr, count(s) AS target_count

RETURN
    count(cr) AS total_resolutions,

    sum(CASE
        WHEN cr.resolution_status = 'RESOLVED'
         AND target_count = 1
        THEN 1 ELSE 0
    END) AS resolved_with_one_target,

    sum(CASE
        WHEN cr.resolution_status = 'RESOLVED'
         AND target_count = 0
        THEN 1 ELSE 0
    END) AS resolved_without_target,

    sum(CASE
        WHEN cr.resolution_status = 'RESOLVED'
         AND target_count > 1
        THEN 1 ELSE 0
    END) AS resolved_with_multiple_targets,

    sum(CASE
        WHEN cr.resolution_status IN ['AMBIGUOUS', 'UNRESOLVED']
         AND target_count > 0
        THEN 1 ELSE 0
    END) AS non_resolved_with_confirmed_target
"""

result = run_cypher_query(query)
display(pd.DataFrame(result))

,total_resolutions,resolved_with_one_target,resolved_without_target,resolved_with_multiple_targets,non_resolved_with_confirmed_target
0,108839,7734,0,0,0


In [249]:
# Step 13.107 — Ingest ambiguous call candidate relationships

BATCH_SIZE = 500

ambiguous_calls_df = resolved_calls_ingest_df[
    resolved_calls_ingest_df["resolution_status"] == "AMBIGUOUS"
].copy()

query = """
UNWIND $rows AS row

MATCH (cr:CallResolution {
    call_id: row.call_id
})

UNWIND row.candidate_symbol_ids AS candidate_id

MATCH (s:Symbol {symbol_id: candidate_id})

MERGE (cr)-[:HAS_CANDIDATE]->(s)

RETURN count(*) AS candidate_relationships_created_or_matched
"""

total_candidates = 0

for start in range(0, len(ambiguous_calls_df), BATCH_SIZE):

    batch = ambiguous_calls_df.iloc[start:start + BATCH_SIZE]

    rows = batch[
        ["call_id", "candidate_symbol_ids"]
    ].to_dict("records")

    result = run_cypher_query(query, {"rows": rows})

    batch_count = (
        result[0]["candidate_relationships_created_or_matched"]
        if result else 0
    )

    total_candidates += batch_count

    print(
        f"Batch {start // BATCH_SIZE + 1}: "
        f"{batch_count:,} candidate relationships"
    )

expected_candidates = int(
    ambiguous_calls_df["candidate_count"].sum()
)

print(f"\nTotal candidate relationships: {total_candidates:,}")
print(f"Expected candidate relationships: {expected_candidates:,}")

Batch 1: 14,139 candidate relationships
Batch 2: 83,831 candidate relationships
Batch 3: 72,389 candidate relationships
Batch 4: 58,646 candidate relationships
Batch 5: 376 candidate relationships

Total candidate relationships: 229,381
Expected candidate relationships: 229,381


In [250]:
# Step 13.108 — Validate ambiguous call candidates

query = """
MATCH (cr:CallResolution {resolution_status: "AMBIGUOUS"})
OPTIONAL MATCH (cr)-[:HAS_CANDIDATE]->(s:Symbol)

WITH
    cr,
    count(s) AS actual_candidate_count

RETURN
    count(cr) AS ambiguous_calls,
    sum(cr.candidate_count) AS expected_candidates,
    sum(actual_candidate_count) AS actual_candidates,
    sum(
        CASE
            WHEN actual_candidate_count <> cr.candidate_count
            THEN 1
            ELSE 0
        END
    ) AS calls_with_candidate_count_mismatch
"""

result = run_cypher_query(query)
pd.DataFrame(result)

,ambiguous_calls,expected_candidates,actual_candidates,calls_with_candidate_count_mismatch
0,2078,229381,229381,0


In [251]:
# Step 13.109 — Overall CallResolution integrity

query = """
MATCH (cr:CallResolution)
OPTIONAL MATCH (cs:CallSite)-[:HAS_RESOLUTION]->(cr)
WITH cr, count(cs) AS call_site_count

OPTIONAL MATCH (cr)-[r:RESOLVES_TO]->(target:Symbol)
WITH cr, call_site_count, count(r) AS resolved_target_count

OPTIONAL MATCH (cr)-[c:HAS_CANDIDATE]->(candidate:Symbol)
RETURN
    count(cr) AS total_call_resolutions,
    sum(CASE WHEN call_site_count = 0 THEN 1 ELSE 0 END)
        AS without_call_site,
    sum(CASE WHEN call_site_count > 1 THEN 1 ELSE 0 END)
        AS multiple_call_sites,
    sum(CASE WHEN cr.resolution_status = 'RESOLVED'
              AND resolved_target_count = 1 THEN 1 ELSE 0 END)
        AS resolved_with_one_target,
    sum(CASE WHEN cr.resolution_status = 'RESOLVED'
              AND resolved_target_count <> 1 THEN 1 ELSE 0 END)
        AS resolved_target_mismatch,
    sum(CASE WHEN cr.resolution_status <> 'RESOLVED'
              AND resolved_target_count > 0 THEN 1 ELSE 0 END)
        AS non_resolved_with_confirmed_target
"""

result = run_cypher_query(query)
pd.DataFrame(result)

,total_call_resolutions,without_call_site,multiple_call_sites,resolved_with_one_target,resolved_target_mismatch,non_resolved_with_confirmed_target
0,336142,0,0,7734,0,0


In [252]:
# Step 13.110 — Investigate CallResolution inventory

query = """
MATCH (cr:CallResolution)
RETURN
    count(cr) AS total_nodes,
    count(DISTINCT cr.call_id) AS distinct_call_ids,
    count(CASE WHEN cr.call_id IS NULL THEN 1 END)
        AS missing_call_ids
"""

result = run_cypher_query(query)
pd.DataFrame(result)

,total_nodes,distinct_call_ids,missing_call_ids
0,108839,108839,0


In [253]:
# Step 13.111 — Corrected CallResolution integrity validation

query = """
MATCH (cr:CallResolution)

CALL (cr) {
    OPTIONAL MATCH (cs:CallSite)-[:HAS_RESOLUTION]->(cr)
    RETURN count(DISTINCT cs) AS call_site_count
}

CALL (cr) {
    OPTIONAL MATCH (cr)-[:RESOLVES_TO]->(s:Symbol)
    RETURN count(DISTINCT s) AS resolved_target_count
}

CALL (cr) {
    OPTIONAL MATCH (cr)-[:HAS_CANDIDATE]->(s:Symbol)
    RETURN count(DISTINCT s) AS candidate_count
}

RETURN
    count(cr) AS total_call_resolutions,

    sum(CASE WHEN cr.resolution_status = 'RESOLVED'
        THEN 1 ELSE 0 END) AS resolved_calls,

    sum(CASE WHEN cr.resolution_status = 'AMBIGUOUS'
        THEN 1 ELSE 0 END) AS ambiguous_calls,

    sum(CASE WHEN cr.resolution_status = 'UNRESOLVED'
        THEN 1 ELSE 0 END) AS unresolved_calls,

    sum(CASE WHEN call_site_count <> 1
        THEN 1 ELSE 0 END) AS call_site_mismatches,

    sum(CASE WHEN cr.resolution_status = 'RESOLVED'
              AND resolved_target_count <> 1
        THEN 1 ELSE 0 END) AS resolved_target_mismatches,

    sum(CASE WHEN cr.resolution_status <> 'RESOLVED'
              AND resolved_target_count <> 0
        THEN 1 ELSE 0 END) AS unexpected_confirmed_targets,

    sum(candidate_count) AS actual_candidate_relationships,

    sum(CASE WHEN candidate_count <> cr.candidate_count
        THEN 1 ELSE 0 END) AS candidate_count_mismatches
"""

result = run_cypher_query(query)
pd.DataFrame(result)

,total_call_resolutions,resolved_calls,ambiguous_calls,unresolved_calls,call_site_mismatches,resolved_target_mismatches,unexpected_confirmed_targets,actual_candidate_relationships,candidate_count_mismatches
0,108839,7734,2078,99027,0,0,0,229381,7734


In [254]:
# Step 13.112 — Check candidate counts by resolution status

query = """
MATCH (cr:CallResolution)

CALL (cr) {
    OPTIONAL MATCH (cr)-[:HAS_CANDIDATE]->(s:Symbol)
    RETURN count(s) AS actual_candidate_count
}

RETURN
    cr.resolution_status AS resolution_status,
    count(*) AS call_resolutions,
    sum(cr.candidate_count) AS expected_candidates,
    sum(actual_candidate_count) AS actual_candidates,
    sum(
        CASE
            WHEN cr.candidate_count = actual_candidate_count
            THEN 0 ELSE 1
        END
    ) AS candidate_count_mismatches
ORDER BY resolution_status
"""

result = run_cypher_query(query)
pd.DataFrame(result)

,resolution_status,call_resolutions,expected_candidates,actual_candidates,candidate_count_mismatches
0,AMBIGUOUS,2078,229381,229381,0
1,RESOLVED,7734,7734,0,7734
2,UNRESOLVED,99027,0,0,0


In [255]:
# Step 13.113 — Final Step 13 graph inventory

query = """
CALL () {
    MATCH (n)
    WHERE n:File OR n:Scope OR n:Symbol
       OR n:Import OR n:ImportBinding
       OR n:Export OR n:ExportBinding
       OR n:Reference OR n:ReferenceResolution
       OR n:Assignment OR n:CallSite OR n:CallResolution
       OR n:AnonymousClass OR n:Inheritance
    RETURN labels(n) AS node_labels, count(*) AS node_count
}

RETURN node_labels, node_count
ORDER BY node_labels
"""

result = run_cypher_query(query)
pd.DataFrame(result)

,node_labels,node_count
0,[AnonymousClass],39
1,[Assignment],72972
2,[CallResolution],108839
3,[CallSite],108839
4,[Export],225
5,[ExportBinding],269
6,[File],927
7,[Import],319
8,[ImportBinding],289
9,[Inheritance],151


In [256]:
# Step 13.114 — Save critical DataFrames

from pathlib import Path
import pandas as pd
import pickle
from datetime import datetime

# Create backup directory in the current project
PICKLE_DIR = Path.cwd() / "artifacts" / "pickles"
PICKLE_DIR.mkdir(parents=True, exist_ok=True)

# Original extraction DataFrames
dataframes_to_save = {
    "files_df": files_df,
    "errors_df": errors_df,
    "symbols_df": symbols_df,
    "scopes_df": scopes_df,
    "imports_df": imports_df,
    "import_bindings_df": import_bindings_df,
    "exports_df": exports_df,
    "export_bindings_df": export_bindings_df,
    "references_df": references_df,
    "assignments_df": assignments_df,
    "calls_df": calls_df,
    "classes_df": classes_df,
    "inheritance_df": inheritance_df,
    "resolved_references_df": resolved_references_df,
    "resolved_calls_df": resolved_calls_df,
}

# Additional working DataFrames
optional_dataframes = [
    "classes_clean_df",
    "classes_clean_canonical_df",
    "inheritance_ingest_df",
    "resolved_calls_ingest_df",
]

for name in optional_dataframes:
    if name in globals():
        dataframes_to_save[name] = globals()[name]

# Include canonical DataFrames
if "canonical_dataframes" in globals():
    for name, df in canonical_dataframes.items():
        dataframes_to_save[f"canonical_{name}"] = df

# Save individual DataFrames
manifest = []

for name, df in dataframes_to_save.items():
    file_path = PICKLE_DIR / f"{name}.pkl"
    df.to_pickle(file_path)

    manifest.append({
        "dataframe": name,
        "rows": len(df),
        "columns": len(df.columns),
        "file": file_path.name,
    })

# Save supporting mappings
supporting_objects = {}

for name in [
    "file_id_mapping",
    "named_class_symbol_map",
]:
    if name in globals():
        supporting_objects[name] = globals()[name]

# Save complete consolidated backup
backup = {
    "created_at": datetime.now().isoformat(),
    "dataframes": dataframes_to_save,
    "supporting_objects": supporting_objects,
}

bundle_path = PICKLE_DIR / "step13_complete_backup.pkl"

with open(bundle_path, "wb") as f:
    pickle.dump(backup, f, protocol=pickle.HIGHEST_PROTOCOL)

# Display backup manifest
manifest_df = pd.DataFrame(manifest)

print(f"Backup directory: {PICKLE_DIR}")
print(f"DataFrames saved: {len(dataframes_to_save)}")
print(f"Supporting objects saved: {len(supporting_objects)}")
print(f"Consolidated backup: {bundle_path}")
print(f"Total DataFrame rows: {manifest_df['rows'].sum():,}")

display(manifest_df)

Backup directory: E:\Projects\knowledge_graph_tools\notebooks\artifacts\pickles
DataFrames saved: 32
Supporting objects saved: 2
Consolidated backup: E:\Projects\knowledge_graph_tools\notebooks\artifacts\pickles\step13_complete_backup.pkl
Total DataFrame rows: 1,987,556


,dataframe,rows,columns,file
0,files_df,578,12,files_df.pkl
1,errors_df,28,3,errors_df.pkl
2,symbols_df,63228,16,symbols_df.pkl
3,scopes_df,74469,9,scopes_df.pkl
4,imports_df,319,12,imports_df.pkl
5,import_bindings_df,289,7,import_bindings_df.pkl
6,exports_df,225,11,exports_df.pkl
7,export_bindings_df,269,7,export_bindings_df.pkl
8,references_df,254402,12,references_df.pkl
9,assignments_df,72972,11,assignments_df.pkl


In [257]:
# Step 13.115 — Verify the backup

with open(bundle_path, "rb") as f:
    restored_backup = pickle.load(f)

verification = []

for name, original_df in dataframes_to_save.items():
    restored_df = restored_backup["dataframes"][name]

    verification.append({
        "dataframe": name,
        "original_rows": len(original_df),
        "restored_rows": len(restored_df),
        "columns_match": list(original_df.columns) == list(restored_df.columns),
        "verified": original_df.equals(restored_df),
    })

verification_df = pd.DataFrame(verification)

display(verification_df)

print(
    "All DataFrames verified:",
    verification_df["verified"].all()
)

,dataframe,original_rows,restored_rows,columns_match,verified
0,files_df,578,578,True,True
1,errors_df,28,28,True,True
2,symbols_df,63228,63228,True,True
3,scopes_df,74469,74469,True,True
4,imports_df,319,319,True,True
5,import_bindings_df,289,289,True,True
6,exports_df,225,225,True,True
7,export_bindings_df,269,269,True,True
8,references_df,254402,254402,True,True
9,assignments_df,72972,72972,True,True


All DataFrames verified: True
